In [1]:
# Add teams notification for if this doesn't run properly
# Calculate success rate of all deltas and send notification if it doesn't fall within a certain range?
# Send a notification if the start_range is more than a month back

In [2]:
import pandas as pd
import numpy as np
import psycopg2, datetime
import statistics
import itertools
from IPython.display import clear_output
import time
import sys
import datetime

In [3]:
POSTGRESQL_PARAMS = {
  'username': "HPzg5vzqsmye9PvIygPtXf2SVZrk16oi@bbdb-master",
  'pass': "8GCacTSXObYR6nUllbx9SdF1KyT3psJX",
  'host': "bbdb-master.postgres.database.azure.com",
  'DB': "bbc"
}

In [4]:
import requests, json

# See https://adaptivecards.io/explorer/Fact.html for info on how to customise notifications

channels_dict = {
    'production_notifications': 'https://blackboxcapital.webhook.office.com/webhookb2/12832e5c-9eb6-4b76-a26a-ff293a2c9663@4de4e7e1-bc7f-498b-8438-ec890555edb6/IncomingWebhook/bbbb9136ce65431791647c4e88f96d33/42c482d2-d77f-431f-b445-20ca3690f2a4'
}


def notifyTeams(message):
    channel = channels_dict['production_notifications']
    
    print(message)
    print("Notifying teams")
    payload = {
        "text": message
    }
    headers = {
        'Content-Type': 'application/json'
    }
    try:
        url = channel
        # os.environ['BBC_TEAMS_WEBHOOK']
        response = requests.post(url, headers=headers, data=json.dumps(payload))
        print("Request sent")
    except KeyError:
        print("Request failed as channel="+str(channel)+" is not recognised")

def notifyTeamsAdvanced(
    message:str,
    summary:str, 
    error_code:str,
    additional_info_dict:dict, 
    channel:str, 
    # importance:str
    ):
    print("Notifying teams")
    # if importance in icons_dict.keys():
    #     image_url = icons_dict[importance]
    # else:
    #     image_url = None
    payload = {
        "@type": "MessageCard",
        "@context": "http://schema.org/extensions",
        "themeColor": "0076D7",
        "summary": summary,
        "sections": [{
            "activityTitle": summary,
            "activitySubtitle": "Error code: "+str(error_code),
            # "activityImage": image_url,
            "text": message,
            "facts": [
                {"name":k,
                "value":v}
                for k,v in additional_info_dict.items()
            ],
            "markdown": True
        }],
    }
    headers = {
        'Content-Type': 'application/json'
    }
    try:
        url = channels_dict[channel]
        # os.environ['BBC_TEAMS_WEBHOOK']
        response = requests.post(url, headers=headers, data=json.dumps(payload))
    except KeyError:
        print("Request failed as channel="+str(channel)+" is not recognised")


In [5]:
def get_table_info(table_name):

    sql_statement = "SELECT column_name, data_type, udt_name FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_NAME = '" + str(table_name) + "' ORDER BY ORDINAL_POSITION;"
    table_info = postgres_Retreive_Insert(sql_statement, POSTGRESQL_PARAMS, retrieve_data = True)
    
    return table_info

In [6]:
# Create sql statement For inserts

def insert_sql(table_name, formatted_data):
    
    last_save = 0
    global_error = False
    
    sql_complete = ''
    sql_statement_part1 = 'insert into ' + table_name + ' ('

    for col in formatted_data.columns:
        sql_statement_part1 = sql_statement_part1 + '"' + col + '", '

    sql_statement_part1 = sql_statement_part1[:-2] + ') values '


    for new_row in range (0, len(formatted_data)):
        #print(new_row, len(formatted_data))

        sql_statement_part2 = ''

        for col in formatted_data.columns:
            
            sql_statement_part2 = sql_statement_part2 + str(formatted_data.iloc[new_row][col]) + ', '

        sql_statement_part2 = '(' + sql_statement_part2[:-2] + '), '

        sql_complete = sql_complete + sql_statement_part2
        
        
        
        if (new_row - last_save) >= 10000:
        
            sql_complete = sql_statement_part1 + sql_complete[:-2] + ';'
            no_data = postgres_Retreive_Insert(sql_complete, POSTGRESQL_PARAMS, retrieve_data = False)
            
            sql_complete = ''
            last_save = new_row
            
            
    # It hasn't just saved
    if (last_save != new_row) | (new_row == 0):
        
        sql_complete = sql_statement_part1 + sql_complete[:-2] + ';'
        no_data = postgres_Retreive_Insert(sql_complete, POSTGRESQL_PARAMS, retrieve_data = False)

    print(str(sys._getframe().f_code.co_name) + ' - Inserts Complete - ' + str(len(formatted_data)))
    
    return


In [7]:
# Create sql statement For inserts

def update_sql(table_name, formatted_data, table_id_column):
    # Create sql statement For inserts
    
    global_error = False
    
    sql_queries = []
    last_save = 0
    new_row = 0

    sql_all = ''

    sql_statement_part1 = 'update ' + table_name + ' set '

    for new_row in range (0, len(formatted_data)):

        sql_complete = ''

        sql_statement_part2 = ''

        for col in formatted_data.columns:
            
            if col != table_id_column:
                sql_statement_part2 = sql_statement_part2 + '"' + str(col) + '" = ' + str(formatted_data.iloc[new_row][col]) + ', '

        #sql_statement_part2 = '(' + sql_statement_part2[:-2] + '), '
        sql_statement_part2 = sql_statement_part2[:-2]
        sql_statement_part2 = sql_statement_part2 + ' where ' + str(table_id_column) + ' = ' + str(formatted_data.iloc[new_row][table_id_column])  + "'"

        #sql_complete = sql_complete + sql_statement_part2


        sql_complete = sql_statement_part1 + sql_statement_part2[:-1] + ';'


        sql_all = sql_all + ' ' + sql_complete
        
        if ( (new_row - last_save) >= 10000) | (new_row == len(formatted_data)):
            
            no_data = postgres_Retreive_Insert(sql_all, POSTGRESQL_PARAMS, retrieve_data = False)

            
            sql_all = ''
            last_save = new_row
            
            
    if (last_save != new_row) | ( (new_row == 0) & (sql_all != '')):
        no_data = postgres_Retreive_Insert(sql_all, POSTGRESQL_PARAMS, retrieve_data = False)    

    print(str(sys._getframe().f_code.co_name) + ' - Update Complete - ' + str(len(formatted_data)))
    
    return


In [8]:
# Format data to add to the database

def format_data_for_postgres(formatted_data, powerbi_table_info):

    formatted_data = formatted_data.replace("'", "''")
    formatted_data = formatted_data.replace(to_replace = '', value = 'NULL')
    formatted_data = formatted_data.replace(to_replace = 'null', value = 'NULL')


    # Replace nulls with NULL
    formatted_data.fillna('NULL', inplace = True)

    for col in formatted_data.columns:

        #print(col)
        column_type = powerbi_table_info[ powerbi_table_info['column_name'] == col]['data_type'].iloc[0]

        if (column_type == 'numeric') | (column_type == 'float') | (column_type == 'double precision'):

            formatted_data[col] = formatted_data[[col]].apply(lambda x: 'NULL' if x[0] == 'NULL' else float(x[0]), axis = 1)

        elif column_type == 'boolean':

            formatted_data[col] = formatted_data[col].apply(lambda x: False if x == 0 else True)
            formatted_data[col] = formatted_data[col].apply(lambda x: str(x).upper() )
            #formatted_data[col] = formatted_data[col].apply(lambda x: "'" + str(x).upper() + "'")


        elif column_type == 'timestamp without time zone':

            #formatted_data[col] = pd.to_datetime(formatted_data[col])
            formatted_data[col] = formatted_data[[col]].apply(lambda x: convert_datetimes(x[0]), axis = 1)
            #formatted_data[col].apply(lambda x: 'NULL' if ( pd.isna(x) | (x == 'NULL') ) else pd.to_datetime(datetime.datetime.strptime(str(x), '%d-%m-%Y %H:%M:%S')) if str(x)[2]=='-' else pd.to_datetime(datetime.datetime.strptime(str(x), '%d/%m/%Y %H:%M:%S')) if str(x)[2]=='/' else pd.to_datetime(datetime.datetime.strptime(str(x), '%Y/%m/%d %H:%M:%S')) if str(x)[4]=='/' else pd.to_datetime(datetime.datetime.strptime(str(x), '%Y-%m-%d %H:%M:%S')) if str(x)[4]=='-' else 'NULL')
            formatted_data[col] = formatted_data[col].apply(lambda x: 'NULL' if x == 'NULL' else "'" + str(x) + "'")


            
        elif column_type == 'timestamp with time zone':

            formatted_data[col] = formatted_data[col].apply(lambda x: "'" + str(x) + "'")

            
 
        elif column_type == 'time without time zone':

            formatted_data[col] = formatted_data[col].apply(lambda x: "'" + str(x) + "'")

            
            
        elif column_type == 'date':
            formatted_data[col] = formatted_data[col].apply(lambda x: convert_datetimes(x))
            formatted_data[col] = formatted_data[col].apply(lambda x: 'NULL' if x == 'NULL' else "'" + str(x) + "'")

            
            
        elif column_type == 'interval':

            formatted_data[col] = formatted_data[col].apply(lambda x: "'" + str(x) + "'")


        elif column_type == 'uuid':

            formatted_data[col] = formatted_data[col].apply(lambda x: "'" + str(x) + "'")


        elif column_type == 'ARRAY':

            array_type = powerbi_table_info[ powerbi_table_info['column_name'] == col]['udt_name'].iloc[0]

            #formatted_data[col] = formatted_data[col].apply(lambda x: str(x).replace("False", 'false'))
            #formatted_data[col] = formatted_data[col].apply(lambda x: str(x).replace("True", 'true'))
            #formatted_data[col] = formatted_data[col].apply(lambda x: str(x).replace("'", '"'))
            #formatted_data[col] = formatted_data[col].apply(lambda x: str(x).replace("\\", '\\\\'))

            formatted_data[col] = formatted_data[col].apply(lambda x: str(x).replace("False", 'false').replace("True", 'true').replace("'", '"').replace("\\", '\\\\'))

            if array_type == '_json':
                array_type_string = '::json[]'

                #formatted_data[col] = formatted_data[col].apply(lambda x: x.replace("{", "'{"))
                #formatted_data[col] = formatted_data[col].apply(lambda x: x.replace("}", "}'"))

                formatted_data[col] = formatted_data[col].apply(lambda x: x.replace("{", "'{").replace("}", "}'"))
                formatted_data[col] = formatted_data[col].apply(lambda x: "'" + str(x) + "'" )
                formatted_data[col] = formatted_data[col].apply(lambda x: "['{" + str(x)[1:-1] + "}']" if str(x)[1:2].isnumeric() else x)

            elif array_type == '_int4':
                array_type_string = '::integer[]'

            formatted_data[col] = formatted_data[col].apply(lambda x: 'NULL' if (x == 'NULL') | (x == '[]') else "ARRAY[" + str(x)[1:-1] + "]" + str(array_type_string))



        elif column_type == 'json':

            formatted_data[col] = formatted_data[col].replace("'", '"')
            formatted_data[col] = formatted_data[col].apply(lambda x: "'" + str(x).replace("'", '"') + "'")

            
        elif column_type == 'jsonb':

            formatted_data[col] = formatted_data[col].replace("'", '"')
            formatted_data[col] = formatted_data[col].apply(lambda x: "'" + str(x).replace("'", '"') + "'")

            

        elif column_type == 'character varying':
            formatted_data[col] = formatted_data[col].apply(lambda x: str(x).replace("'", '"'))
            formatted_data[col] = formatted_data[col].apply(lambda x: "'" + str(x) + "'")


        elif column_type == 'text':

            formatted_data[col] = formatted_data[col].apply(lambda x: str(x).replace("'", '"'))
            formatted_data[col] = formatted_data[col].apply(lambda x: "'" + str(x) + "'")
            

        elif column_type == 'integer':

            formatted_data[col] = formatted_data[col].apply(lambda x: x if x == str("NULL") else "NULL" if x == '' else int(float(x)))

        else:
            print('DATA TYPE IS NOT COVERED - FIX - %s'%(column_type))


    formatted_data = formatted_data.replace("'NULL'", 'NULL')
    
    return formatted_data

In [9]:
def convert_datetimes(datetime_obj):
        
    #if datetime_obj != 'NULL':
    #    datetime_obj = str(pd.to_datetime(datetime_obj).replace(tzinfo=None))
    
    datetime_obj = str(datetime_obj)
    
    
    if datetime_obj == 'NULL':

        datetime_obj =  'NULL'
        
    if datetime_obj.find(',')>0:
        datetime_obj = datetime_obj.replace(',', '')
        
    if datetime_obj.find('T')>0:
        datetime_obj = datetime_obj.replace('T', ' ')
    
    elif len(datetime_obj) == 10:
        
        if str(datetime_obj)[2] == '-':
            datetime_obj = (datetime.datetime.strptime(str(datetime_obj), '%d-%m-%Y'))
            datetime_obj =  datetime.datetime.strftime(datetime_obj, '%Y-%m-%d')
        elif str(datetime_obj)[2] == '/':
            
            if int(str(datetime_obj)[3:5]) > 12:
                datetime_obj = (datetime.datetime.strptime(str(datetime_obj), '%m/%d/%Y')) 
            else:
                datetime_obj = (datetime.datetime.strptime(str(datetime_obj), '%d/%m/%Y')) 
            datetime_obj =  datetime.datetime.strftime(datetime_obj, '%Y-%m-%d')
        elif str(datetime_obj)[4]=='-':
            datetime_obj = (datetime.datetime.strptime(str(datetime_obj), '%Y-%m-%d'))
            datetime_obj =  datetime.datetime.strftime(datetime_obj, '%Y-%m-%d')
        elif str(datetime_obj)[4]=='/':
            datetime_obj = (datetime.datetime.strptime(str(datetime_obj), '%Y/%m/%d')) 
            datetime_obj =  datetime.datetime.strftime(datetime_obj, '%Y-%m-%d')
        else:
            datetime_obj =  'NULL'
        
    elif len(datetime_obj) > 10:

        if str(datetime_obj)[2] == '-':
            datetime_obj = (datetime.datetime.strptime(str(datetime_obj), '%d-%m-%Y %H:%M:%S'))
            datetime_obj =  datetime.datetime.strftime(datetime_obj, '%Y-%m-%d %H:%M:%S')
        elif str(datetime_obj)[2] == '/':
            if int(str(datetime_obj)[3:5]) > 12:
                datetime_obj = (datetime.datetime.strptime(str(datetime_obj), '%m/%d/%Y %H:%M:%S')) 
            else:
                datetime_obj = (datetime.datetime.strptime(str(datetime_obj), '%d/%m/%Y %H:%M:%S')) 
            datetime_obj =  datetime.datetime.strftime(datetime_obj, '%Y-%m-%d %H:%M:%S')
        elif str(datetime_obj)[4]=='-':
            datetime_obj = (datetime.datetime.strptime(str(datetime_obj), '%Y-%m-%d %H:%M:%S'))
            datetime_obj =  datetime.datetime.strftime(datetime_obj, '%Y-%m-%d %H:%M:%S')
        elif str(datetime_obj)[4]=='/':
            datetime_obj = (datetime.datetime.strptime(str(datetime_obj), '%Y/%m/%d %H:%M:%S')) 
            datetime_obj =  datetime.datetime.strftime(datetime_obj, '%Y-%m-%d %H:%M:%S')
        else:
            datetime_obj =  'NULL'
        
    else:
        datetime_obj =  'NULL'
        

    
    return datetime_obj
    

In [10]:
def postgres_Retreive_Insert(sql_statement, POSTGRESQL_PARAMS, retrieve_data):
    
#     start_time = datetime.datetime.now()
#     print('-postgres_Retreive_Insert')
    
#     if retrieve_data:
#         print('-retrieving data')
#     else:
#         print('-inserting data')


    return_list = []
    try:
        #if date_delta is None:
        #  date_delta = datetime.timedelta(days=100000)
        conn = psycopg2.connect(
          host = POSTGRESQL_PARAMS['host'],
          database = POSTGRESQL_PARAMS['DB'],
          user = POSTGRESQL_PARAMS['username'],
          password = POSTGRESQL_PARAMS['pass'],
        )
        conn.set_client_encoding('UTF-8')
        cur = conn.cursor()
        cur.execute(sql_statement)


        if retrieve_data:
            temp = pd.DataFrame(cur.fetchall(), columns = [desc[0] for desc in cur.description])
            for col in temp.columns:
                if (col == 'password') | (col == 'token') | (col =='session'):
                    temp.drop(col, axis = 1, inplace = True)  
                    
#             end_time = datetime.datetime.now()
#             print('--Complete-' + str(end_time - start_time))
            return temp, False
        
        else:
            conn.commit()
            
#             end_time = datetime.datetime.now()
#             print('--Complete-' + str(end_time - start_time))
            return None, False

        cur.close()
        conn.close()
        
        
    except Exception as ex:
        print('Error Message - ' + str(ex))

#         end_time = datetime.datetime.now()
#         print('--Complete-' + str(end_time - start_time))
        return None, True


In [11]:
def get_all_events():
    
    function_start_time = datetime.datetime.now()
    print('-get_all_events')

    sql_statement = "select * from view_event order by start_time asc;"
    all_events, error_occured = postgres_Retreive_Insert(sql_statement, POSTGRESQL_PARAMS, True)
    
    # Drop Mayanks calculated columns and use my own
    for col in all_events.columns:

        if col in ['home_team_national_team_id', 'away_team_national_team_id',
           'home_team_type', 'away_team_type',
           'venue_national_team', 'home_previous_n_games_with_venue',
           'num_fix_last_20_games', 'venue_perc', 'home_venue', 'home_travelled',
           'away_travelled', 'travel_advantage']:
            all_events.drop(col, axis = 1, inplace = True)

    
    function_end_time = datetime.datetime.now()
    print('--Complete-' + str(function_end_time - function_start_time))
    print('')

    return all_events

In [12]:
def get_all_previous_deltas(float_columns):
    
    function_start_time = datetime.datetime.now()
    print('-get_all_previous_deltas')

    sql_statement = "select * from event_deltas order by start_time asc;"
    all_deltas, error_occured = postgres_Retreive_Insert(sql_statement, POSTGRESQL_PARAMS, True)
    
    #all_deltas = pd.read_csv('event_deltas.csv')
    for col in float_columns:
        
        all_deltas[col] = all_deltas[col].apply(lambda x: float(x) if pd.notna(x) else None)
        
    #all_deltas['away_pre_delta'] = all_deltas['away_pre_delta'].apply(lambda x: float(x))
    #all_deltas['home_post_delta'] = all_deltas['home_post_delta'].apply(lambda x: float(x))
    #all_deltas['away_post_delta'] = all_deltas['away_post_delta'].apply(lambda x: float(x))

    function_end_time = datetime.datetime.now()
    print('--Complete-' + str(function_end_time - function_start_time))
    print('')

    return all_deltas

In [13]:
def fixtures_to_remove(vec):
    
    to_remove = False
    
    home_score = vec[0]
    away_score = vec[1]
    fix_date = vec[2].replace(tzinfo=None)
    
    if (home_score == 0) & (away_score == 0):
        
        to_remove = True
        
    elif pd.isna(home_score) | pd.isna(away_score):
        
        to_remove = True
        
    elif (pd.to_datetime(fix_date) > pd.to_datetime('2019-01-03')) & (pd.to_datetime(fix_date) < pd.to_datetime('2022-09-01')):
        if (home_score == 28) & (away_score == 0):
            to_remove = True
        elif (home_score == 0) & (away_score == 28):
            to_remove = True  
        
    return to_remove


In [14]:
def remove_faulty_fixtures(all_events):
    
    function_start_time = datetime.datetime.now()
    print('-remove_faulty_fixtures')
    
    all_events['to_remove'] = all_events[['home_score', 'away_score', 'start_time']].apply(lambda x: fixtures_to_remove(x), axis = 1)
    
    all_events = all_events[(all_events['to_remove']==False) | (all_events['start_time']>(str(pd.to_datetime(datetime.datetime.now() - datetime.timedelta(days = 1)).replace(tzinfo=None).date())))]

    all_events = all_events[ all_events['start_time'] > '2003-01-01']
    all_events.drop('to_remove', axis = 1, inplace = True)
    
    
    
    # Make sure future events are set to None and not 0 for their scores
    future_events = all_events[ all_events['start_time'] >= str(datetime.datetime.now())].index
    all_events.loc[future_events, 'home_score'] = None
    all_events.loc[future_events, 'away_score'] = None
    
    
    ######### Sort out games that have no IDs #########
    games_with_no_ids = list(all_events[ pd.isna(all_events['home_team_internal_id']) | pd.isna(all_events['away_team_internal_id']) ]['event_id'])

    if len(games_with_no_ids) > 0:

        # Alert teams with the events that don't have an id
        message = 'There are events where there are no ids for the homeor away team (' + str(games_with_no_ids) + ')'
        notifyTeams(message)
        print(message)

        all_events = all_events[ ~all_events['event_id'].isin(games_with_no_ids) ]
    ###################################################



    function_end_time = datetime.datetime.now()
    print('--Complete-' + str(function_end_time - function_start_time))
    print('')

    return all_events

In [15]:
def check_fixtures_that_have_changed(all_previous_deltas, all_events):

    function_start_time = datetime.datetime.now()
    print('-check_fixtures_that_have_changed')
    
    cols_to_keep = all_previous_deltas.columns
    
    all_previous_deltas = all_previous_deltas.merge(all_events[['event_id', 'home_team_internal_id', 'away_team_internal_id', 'competition_internal_id', 'home_score', 'away_score', 'venue_internal_id']].rename(columns = {'event_id':'event_id_AE', 'home_team_internal_id':'home_team_internal_id_AE', 'away_team_internal_id':'away_team_internal_id_AE', 'competition_internal_id':'competition_internal_id_AE', 'home_score':'home_score_AE', 'away_score':'away_score_AE', 'venue_internal_id':'venue_internal_id_AE'}), how = 'left', left_on = 'event_id', right_on = 'event_id_AE')

    
    # Check for fixtures that have changed since ELO's have last been updated
    all_previous_deltas['competition_internal_id_different'] = all_previous_deltas['competition_internal_id'] == all_previous_deltas['competition_internal_id_AE']
    all_previous_deltas['home_team_internal_id_different'] = all_previous_deltas['home_team_internal_id'] == all_previous_deltas['home_team_internal_id_AE']
    all_previous_deltas['away_team_internal_id_different'] = all_previous_deltas['away_team_internal_id'] == all_previous_deltas['away_team_internal_id_AE']
    all_previous_deltas['home_score_different'] = all_previous_deltas['home_score'] == all_previous_deltas['home_score_AE']
    all_previous_deltas['away_score_different'] = all_previous_deltas['away_score'] == all_previous_deltas['away_score_AE']
    
    for record in all_previous_deltas.index:
        if all_previous_deltas.loc[record, 'venue_internal_id'] == all_previous_deltas.loc[record, 'venue_internal_id_AE']:
            all_previous_deltas['venue_different'] = True
        else:
            all_previous_deltas['venue_different'] = False
    #all_previous_deltas['venue_different'] = all_previous_deltas['venue_internal_id'] == all_previous_deltas['venue_internal_id_AE']

    earliest_fixture_change = all_previous_deltas[ (all_previous_deltas['competition_internal_id_different'] == False) | (all_previous_deltas['home_team_internal_id_different'] == False) | (all_previous_deltas['away_team_internal_id_different'] == False) | (all_previous_deltas['home_score_different'] == False) | (all_previous_deltas['away_score_different'] == False) | (all_previous_deltas['venue_different'] == False) ]['start_time'].min()
    
    if pd.notna(earliest_fixture_change):
        previous_deltas_to_keep = all_previous_deltas[ all_previous_deltas['start_time'] < earliest_fixture_change]
    else:
        previous_deltas_to_keep = all_previous_deltas
        
    # Check for fixtures that are no longer in the database
    earliest_fixture_change = all_previous_deltas[ pd.isna(all_previous_deltas['competition_internal_id_different']) ]['start_time'].min()
    if pd.notna(earliest_fixture_change):
        previous_deltas_to_keep = previous_deltas_to_keep[ previous_deltas_to_keep['start_time'] < earliest_fixture_change]

    
    previous_deltas_to_keep = previous_deltas_to_keep[cols_to_keep]

    function_end_time = datetime.datetime.now()
    print('--Complete-' + str(function_end_time - function_start_time))
    print('')

    return previous_deltas_to_keep

In [16]:
def check_for_any_new_events(all_events, all_previous_deltas, float_columns):
    
    function_start_time = datetime.datetime.now()
    print('-check_for_any_new_events')
    
    temp_columns = float_columns.copy()
    temp_columns.append('event_id')
    all_previous_deltas = all_previous_deltas[temp_columns]
    
    all_events = all_events.merge(all_previous_deltas.rename(columns = {'event_id':'event_id_PD'}), how = 'left', left_on = 'event_id', right_on = 'event_id_PD').reset_index()    
    earliest_fixture_change = all_events[ pd.isna(all_events['home_pre_delta']) |  pd.isna(all_events['home_post_delta']) |  pd.isna(all_events['away_pre_delta']) |  pd.isna(all_events['away_post_delta']) ]['start_time'].min()

    for col in float_columns:
        all_events.loc[ all_events['start_time'] >= earliest_fixture_change, col] = None
    #all_events.loc[ all_events['start_time'] >= earliest_fixture_change, 'away_pre_delta'] = None
    #all_events.loc[ all_events['start_time'] >= earliest_fixture_change, 'home_post_delta'] = None
    #all_events.loc[ all_events['start_time'] >= earliest_fixture_change, 'away_post_delta'] = None

    function_end_time = datetime.datetime.now()
    print('--Complete-' + str(function_end_time - function_start_time))
    print('')
    
    return all_events

In [17]:
def get_team_fixture_numbers(all_events):

    function_start_time = datetime.datetime.now()
    print('-get_team_fixture_numbers')

    all_events['home_team_total_fixture_number'] = None
    all_events['home_team_home_fixture_number'] = None
    all_events['away_team_total_fixture_number'] = None
    all_events['away_team_away_fixture_number'] = None

    home_teams = all_events[['home_team_internal_id', 'start_time']].rename(columns = {'home_team_internal_id' : 'team_id'})
    home_teams['h_a'] = 'Home'
    away_teams = all_events[['away_team_internal_id', 'start_time']].rename(columns = {'away_team_internal_id' : 'team_id'})
    away_teams['h_a'] = 'Away'

    fixtures_by_team = pd.concat([home_teams, away_teams])
    #fixtures_by_team = home_teams.append(away_teams)
    fixtures_by_team.sort_values('start_time', inplace = True)
    
    fixtures_by_team['team_total_fixture_numbers'] = fixtures_by_team.groupby(['team_id'])['start_time'].rank()
    fixtures_by_team['team_ha_fixture_numbers'] = fixtures_by_team.groupby(['team_id', 'h_a'])['start_time'].rank()
    
    home_team_fix = fixtures_by_team[ fixtures_by_team['h_a'] == 'Home']
    all_events['home_team_total_fixture_number'].update(home_team_fix['team_total_fixture_numbers'])
    all_events['home_team_home_fixture_number'].update(home_team_fix['team_ha_fixture_numbers'])

    away_team_fix = fixtures_by_team[ fixtures_by_team['h_a'] == 'Away']
    all_events['away_team_total_fixture_number'].update(away_team_fix['team_total_fixture_numbers'])
    all_events['away_team_away_fixture_number'].update(away_team_fix['team_ha_fixture_numbers'])

    return all_events


In [18]:
def get_competition_fixture_numbers():

    function_start_time = datetime.datetime.now()
    print('-get_competition_fixture_numbers')

    all_events['competition_fixture_number'] = None
    all_events.sort_values('start_time', inplace = True)

    all_events['competition_fixture_number'] = all_events.groupby(['competition_internal_id'])['start_time'].rank()
    all_events['home_competition_fixture_number'] = all_events.groupby(['home_competition_group'])['start_time'].rank()

    function_end_time = datetime.datetime.now()
    print('--Complete-' + str(function_end_time - function_start_time))
    print('')

    return all_events

In [19]:
def update_home_venue_on_venue_nation(vec):
    
    
    home_team_type = vec[0]
    home_team_internal_id = vec[1]
    venue_national_team = vec[2]
    home_venue = vec[3]
    home_team_national_team_id = vec[4]
    venue_id = vec[5]
    
    teams_to_leave = ['564ae346-5235-40b8-ab48-fc4f435601be']

    # Change International mens venue to True if they are playing in the same country
    if home_team_type == 'international':

        if home_team_internal_id == venue_national_team:
            home_venue = True
        elif (home_team_internal_id != venue_national_team) & pd.notna(venue_national_team):
            home_venue = False

    # Change other international teams if its in the same country
    elif home_team_type.find('international') >= 0:

        if home_team_national_team_id == venue_national_team:
            home_venue = True
        elif (home_team_internal_id != venue_national_team) & pd.notna(venue_national_team):
            home_venue = False

    elif home_team_internal_id not in teams_to_leave:

        if (home_team_national_team_id != venue_national_team) & pd.notna(home_team_national_team_id) & pd.notna(venue_national_team) & pd.notna(venue_id):
            home_venue = False


    return home_venue

In [20]:
def calculate_travel_advantage(vec):

    team_home_countries = {'564ae346-5235-40b8-ab48-fc4f435601be':'cd0d53f5-4a11-4ded-8594-641ef025842d'}
    
    home_team_type = vec[0]
    team_internal_id = vec[1]
    venue_national_team = vec[2]
    team_national_team_id = vec[3]
    
    if team_internal_id in team_home_countries.keys():
    
        if (team_home_countries['564ae346-5235-40b8-ab48-fc4f435601be'] != venue_national_team) & pd.notna(venue_national_team):
            travel_advantage = 1
        else:
            travel_advantage = 0
    
    else:
        
        if home_team_type == 'international':

            if (team_internal_id != venue_national_team) & pd.notna(venue_national_team):
                travel_advantage = 1
            else:
                travel_advantage = 0


        else:
            if (team_national_team_id != venue_national_team) & pd.notna(team_national_team_id) & pd.notna(venue_national_team):
                travel_advantage = 1
            else:
                travel_advantage = 0


    return travel_advantage

In [21]:
def set_home_venues(all_events, all_teams, all_venues):
    
    function_start_time = datetime.datetime.now()
    print('-set_home_venues')
    
    fixture_length = 20

    all_events.loc[ : , 'home_venue'] = True
    all_events.loc[ : , 'venue_perc'] = None


    for team_id in all_events['home_team_internal_id'].drop_duplicates():

        team_df = all_events[ all_events['home_team_internal_id'] == team_id]


        for fix in team_df.index:

            venue_id = team_df.loc[fix, 'venue_internal_id']

            if pd.notna(venue_id):

                current_fix_num = team_df.loc[fix, 'home_team_home_fixture_number']
                number_of_previous_games_at_venue = len(team_df[ (team_df['venue_internal_id'] == venue_id) & (team_df['home_team_home_fixture_number'] <= current_fix_num) & (team_df['home_team_home_fixture_number'] > (current_fix_num - fixture_length)) ])
                number_of_previous_games_with_venue = len(team_df[ pd.notna(team_df['venue_internal_id']) & (team_df['home_team_home_fixture_number'] <= current_fix_num) & (team_df['home_team_home_fixture_number'] > (current_fix_num - fixture_length)) ])

                venue_perc = number_of_previous_games_at_venue / number_of_previous_games_with_venue
                team_df.loc[fix,'venue_perc'] = venue_perc
                all_events['venue_perc'].update(team_df['venue_perc'])

                if venue_perc < 0.2:
                    team_df.loc[fix,'home_venue'] = False
                    all_events['home_venue'].update(team_df['home_venue'])
                    

    # Add national team for home and away teams 
    all_events = all_events.merge(all_teams[['id', 'type', 'national_team_id']].rename(columns = {'id':'home_team_internal_id', 'type':'home_team_type', 'national_team_id':'home_team_national_team_id'}), how = 'left', left_on = 'home_team_internal_id', right_on = 'home_team_internal_id')
    all_events = all_events.merge(all_teams[['id', 'type', 'national_team_id']].rename(columns = {'id':'away_team_internal_id', 'type':'away_team_type', 'national_team_id':'away_team_national_team_id'}), how = 'left', left_on = 'away_team_internal_id', right_on = 'away_team_internal_id')
    
    all_events['home_venue'] = all_events[['home_team_type', 'home_team_internal_id', 'venue_national_team', 'home_venue', 'home_team_national_team_id', 'venue_internal_id']].apply(lambda x: update_home_venue_on_venue_nation(x), axis = 1)
    all_events['home_travelled'] = all_events[['home_team_type', 'home_team_internal_id', 'venue_national_team', 'home_team_national_team_id']].apply(lambda x: calculate_travel_advantage(x), axis = 1)
    all_events['away_travelled'] = all_events[['away_team_type', 'away_team_internal_id', 'venue_national_team', 'away_team_national_team_id']].apply(lambda x: calculate_travel_advantage(x), axis = 1)
    all_events['travel_advantage'] =  all_events[['home_travelled', 'away_travelled']].apply(lambda x: -1 if ( (x[0] == 1) and (x[1] == 0)) else 0 if ( (x[0] == 0) and (x[1] == 0) ) else 0 if ( (x[0] == 1) and (x[1] == 1) ) else 1 if ( (x[0] == 0) and (x[1] == 1) ) else 0, axis = 1)

    function_end_time = datetime.datetime.now()
    print('--Complete-' + str(function_end_time - function_start_time))
    print('')

    return all_events


In [22]:
def get_comp_standards(all_events, delta_column_to_calcuate):
    
    function_start_time = datetime.datetime.now()
    print('-get_comp_standards')
    
    comp_standards = all_events[ (all_events['start_time'] < '2010-01-01') & pd.notna(all_events[delta_column_to_calcuate]) ]

    all_base_home_win_margin = comp_standards[delta_column_to_calcuate].median()
    international_mens_base_home_win_margin = comp_standards[ comp_standards['home_competition_group'] == 'international_mens'][delta_column_to_calcuate].median()
    international_womens_base_home_win_margin = comp_standards[ comp_standards['home_competition_group'] == 'international_womens'][delta_column_to_calcuate].median()
    comp_standards = comp_standards[ (comp_standards['home_competition_group'] != 'international_mens')  & (comp_standards['home_competition_group'] != 'international_womens') ]
    
    competition_win_margin_means = comp_standards[['competition_internal_id', 'competition_name', delta_column_to_calcuate]].groupby(['competition_internal_id', 'competition_name']).median().reset_index()
    competition_win_margin_count = comp_standards[['competition_internal_id', 'competition_name', delta_column_to_calcuate]].groupby(['competition_internal_id', 'competition_name']).count().reset_index()

    competition_win_margin_means = competition_win_margin_means.merge(competition_win_margin_count[['competition_internal_id', delta_column_to_calcuate]].rename(columns={delta_column_to_calcuate:'count'}), how = 'left', left_on = 'competition_internal_id', right_on = 'competition_internal_id')
    competition_win_margin_means = competition_win_margin_means[competition_win_margin_means['count']>100]
    
    function_end_time = datetime.datetime.now()
    print('--Complete-' + str(function_end_time - function_start_time))
    print('')

    return all_base_home_win_margin, international_mens_base_home_win_margin, international_womens_base_home_win_margin, competition_win_margin_means

In [23]:
def check_success(all_events, validation_start, validation_end):

    function_start_time = datetime.datetime.now()
    print('-check_success')
    
    all_events['success'] = all_events[[delta_column_to_calcuate, pre_delta_diff_name, 'home_score', 'away_score']].apply(lambda x: None if (pd.isna(x[2]) & pd.isna(x[3])) else 1 if ((x[0] > 0) & (x[1] > 0)) | ((x[0] < 0) & (x[1] < 0)) else 0 , axis = 1 )

    validation_period = all_events[ (all_events['start_time'] >= validation_start) & (all_events['start_time'] <= validation_end) ]
    #validation_period = all_events
    correct = len(validation_period[(validation_period['success'] == 1)])
    incorrect = len(validation_period[(validation_period['success'] == 0)])

    success = correct/(correct+incorrect)
    
    end_time = datetime.datetime.now()
    print('--Complete-' + str(end_time - function_start_time))
    print('')

    return success

In [24]:
def generate_elo_ranks(all_events, delta_column_to_calcuate, post_delta_adjustment_name, home_pre_delta_name, home_post_delta_name, away_pre_delta_name, away_post_delta_name, pre_delta_diff_name, home_team_buffer_name, max_points_win, win_margin_buffer, level_setting, start_range, end_range, list_order_int_men, list_order_int_women_age, list_order_club, win_bonus, all_competitions, all_teams, home_team_fixture_column, away_team_fixture_column, home_error, away_error):



    max_points_win = 5
    win_margin_buffer = 0
    level_setting = 40
    win_bonus = 0

    loop_num = 0


    function_start_time = datetime.datetime.now()
    print('-generate_elo_ranks')

    all_events.reset_index(inplace = True, drop = True)
    #all_events['post_game_rank_home'] = None
    #all_events['post_game_rank_away'] = None
    #nall_events[home_team_buffer_name] = None
    all_events['pre_game_rank_historic_home_competition_group_min_home'] = None
    all_events['pre_game_rank_historic_home_competition_group_median_home'] = None
    all_events['pre_game_rank_senior_team_ranking_home'] = None
    all_events['pre_game_rank_int_comp_level_setting_home'] = None
    all_events['pre_game_rank_new_home_competition_group_home'] = None
    all_events['pre_game_rank_historic_home_competition_group_min_away'] = None
    all_events['pre_game_rank_historic_home_competition_group_median_away'] = None
    all_events['pre_game_rank_senior_team_ranking_away'] = None
    all_events['pre_game_rank_int_comp_level_setting_away'] = None
    all_events['pre_game_rank_new_home_competition_group_away'] = None
    #all_events[pre_delta_diff_name] = None
    #all_events[post_delta_adjustment_name] = None

    home_pre_elo = None
    away_pre_elo = None
    pre_elo_diff = None


    print(str(loop_num), '--updating from', start_range, end_range)
    for event in range(start_range,end_range+1):
    #for event in range(12562,12562+1):


            print('--', event, end_range)
        
        
            start_time = all_events.loc[event]['start_time']

#             if start_time < pd.to_datetime('2010-01-01', utc=True):
#                 home_pre_elo = all_events.loc[event]['home_pre_delta']
#                 away_pre_elo = all_events.loc[event]['away_pre_delta']
#             else:
            home_pre_elo = None
            away_pre_elo = None

            home_team_internal_id = all_events.loc[event]['home_team_internal_id']
            home_team_total_fixture_number = all_events.loc[event]['home_team_total_fixture_number']

            away_team_internal_id = all_events.loc[event]['away_team_internal_id']
            away_team_total_fixture_number = all_events.loc[event]['away_team_total_fixture_number']

            actual_win_margin = all_events.loc[event, delta_column_to_calcuate]
            
            home_venue = all_events.loc[event]['home_venue']

            start_time = all_events.loc[event]['start_time']
            competition_id = all_events.loc[event]['competition_internal_id']
            competition_fixture_number = all_events.loc[event]['competition_fixture_number']
            home_competition_group = all_events.loc[event]['home_competition_group']
            home_competition_fixture_number = all_events.loc[event]['home_competition_fixture_number']
            
            if competition_id in points_transfer_dict.keys():
                transfer_multiplier = points_transfer_dict[competition_id]
            else:
                transfer_multiplier = 1

            # Get team travelled buffer
            travel_buffer = all_events.loc[event, 'travel_advantage']
            #travel_buffer = 0

            # Home team is playing in away teams country
            # So set to home venue but reverse the impact
            if travel_buffer == -1:
                home_venue = True

            if home_venue == True:

                home_team_buffer = get_home_team_buffer(all_events, start_time, competition_id, competition_fixture_number, home_competition_group, home_competition_fixture_number, delta_column_to_calcuate)

                if travel_buffer == -1:
                    home_team_buffer = -home_team_buffer

                all_events.loc[event, home_team_buffer_name] = home_team_buffer

            else:
                home_team_buffer = 0
                all_events.loc[event, home_team_buffer_name] = 0


            #print(home_pre_elo)
            if pd.isna(home_pre_elo):
                #print('1')

                if home_team_total_fixture_number == 1:
                    #print('2')

                    if delta_column_to_calcuate in ['half_time_win_margin', 'second_half_win_margin']:

                        #home_pre_elo = all_events.loc[event, 'home_pre_delta'] / 2
                        home_pre_elo = 0

                    else:

                        if away_team_total_fixture_number >= 5:

                            win_margin_for_delta = all_events.loc[event, 'win_margin']
                            if pd.isna(win_margin_for_delta):
                                win_margin_for_delta = 0

                            away_pre_elo_temp = get_last_pre_elo(None, all_events, away_team_internal_id, away_team_total_fixture_number, home_post_delta_name, away_post_delta_name, home_team_fixture_column, away_team_fixture_column)
                            home_pre_elo = away_pre_elo_temp + win_margin_for_delta - home_team_buffer

                        else:


                            # Get appropriate ELO to work with
                            home_pre_elo, rank_set_type, pre_game_rank_historic_home_competition_group_min, pre_game_rank_historic_home_competition_group_median, pre_game_rank_senior_team_ranking, pre_game_rank_int_comp_level_setting, pre_game_rank_new_home_competition_group, pre_game_rank_historic_competition_min, pre_game_rank_historic_competition_median = generate_predicted_elo(all_events, home_team_internal_id, home_team_total_fixture_number, competition_id, competition_fixture_number, home_competition_group, start_time, level_setting, event, list_order_int_men, list_order_int_women_age, list_order_club, home_post_delta_name, away_post_delta_name, home_pre_delta_name, away_pre_delta_name, all_competitions, all_teams, internationl_rankings_df)
                            all_events.loc[event, home_pre_delta_name] = home_pre_elo
                            all_events.loc[event, 'rank_set_type_home'] = rank_set_type
                            all_events.loc[event, 'pre_game_rank_historic_home_competition_group_min_home'] = pre_game_rank_historic_home_competition_group_min
                            all_events.loc[event, 'pre_game_rank_historic_home_competition_group_median_home'] = pre_game_rank_historic_home_competition_group_median
                            all_events.loc[event, 'pre_game_rank_senior_team_ranking_home'] = pre_game_rank_senior_team_ranking
                            all_events.loc[event, 'pre_game_rank_int_comp_level_setting_home'] = pre_game_rank_int_comp_level_setting
                            all_events.loc[event, 'pre_game_rank_new_home_competition_group_home'] = pre_game_rank_new_home_competition_group
                            all_events.loc[event, 'pre_game_rank_historic_competition_min_home'] = pre_game_rank_historic_competition_min
                            all_events.loc[event, 'pre_game_rank_historic_competition_median_home'] = pre_game_rank_historic_competition_median

                else:
                    #print('3')

                    home_pre_elo = get_last_pre_elo(None, all_events, home_team_internal_id, home_team_total_fixture_number, home_post_delta_name, away_post_delta_name, home_team_fixture_column, away_team_fixture_column)

                all_events.loc[event, home_pre_delta_name] = home_pre_elo

            if pd.isna(home_pre_elo):
                
                print('The home delta is blank for event ', event, all_events.loc[event, 'name'])
                

                
            if pd.isna(away_pre_elo):

                #print('1')
                if away_team_total_fixture_number == 1:

                    if delta_column_to_calcuate in ['half_time_win_margin', 'second_half_win_margin']:

                        #away_pre_elo = all_events.loc[event, 'away_pre_delta'] / 2
                        away_pre_elo = 0

                    else:

                        if home_team_total_fixture_number >= 5:

                            win_margin_for_delta = all_events.loc[event, 'win_margin']
                            if pd.isna(win_margin_for_delta):
                                win_margin_for_delta = 0

                            home_pre_elo_temp = get_last_pre_elo(None, all_events, home_team_internal_id, home_team_total_fixture_number, home_post_delta_name, away_post_delta_name, home_team_fixture_column, away_team_fixture_column)
                            away_pre_elo = home_pre_elo_temp - win_margin_for_delta + home_team_buffer

                        else:

                            # Get appropriate ELO to work with
                            away_pre_elo, rank_set_type, pre_game_rank_historic_home_competition_group_min, pre_game_rank_historic_home_competition_group_median, pre_game_rank_senior_team_ranking, pre_game_rank_int_comp_level_setting, pre_game_rank_new_home_competition_group, pre_game_rank_historic_competition_min, pre_game_rank_historic_competition_median = generate_predicted_elo(all_events, away_team_internal_id, away_team_total_fixture_number, competition_id, competition_fixture_number, home_competition_group, start_time, level_setting, event, list_order_int_men, list_order_int_women_age, list_order_club, home_post_delta_name, away_post_delta_name, home_pre_delta_name, away_pre_delta_name, all_competitions, all_teams, internationl_rankings_df)                
                            all_events.loc[event, away_pre_delta_name] = away_pre_elo
                            all_events.loc[event, 'rank_set_type_away'] = rank_set_type
                            all_events.loc[event, 'pre_game_rank_historic_home_competition_group_min_away'] = pre_game_rank_historic_home_competition_group_min
                            all_events.loc[event, 'pre_game_rank_historic_home_competition_group_median_away'] = pre_game_rank_historic_home_competition_group_median
                            all_events.loc[event, 'pre_game_rank_senior_team_ranking_away'] = pre_game_rank_senior_team_ranking
                            all_events.loc[event, 'pre_game_rank_int_comp_level_setting_away'] = pre_game_rank_int_comp_level_setting
                            all_events.loc[event, 'pre_game_rank_new_home_competition_group_away'] = pre_game_rank_new_home_competition_group
                            all_events.loc[event, 'pre_game_rank_historic_competition_min_away'] = pre_game_rank_historic_competition_min
                            all_events.loc[event, 'pre_game_rank_historic_competition_median_away'] = pre_game_rank_historic_competition_median


                else:
                    away_pre_elo = get_last_pre_elo(None, all_events, away_team_internal_id, away_team_total_fixture_number, home_post_delta_name, away_post_delta_name, home_team_fixture_column, away_team_fixture_column)

                all_events.loc[event, away_pre_delta_name] = away_pre_elo

            if pd.isna(away_pre_elo):
                print('The away delta is blank for event ', event, all_events.loc[event, 'name'])


            if delta_column_to_calcuate in ['half_time_win_margin', 'second_half_win_margin']:
                
                # half of the expected result then adjust for half
                pre_elo_diff = ((all_events.loc[event, 'pre_delta_diff'] - all_events.loc[event, 'pre_delta_adjustment'])/2) + home_pre_elo - away_pre_elo

            else:

                pre_elo_diff = (home_pre_elo + home_team_buffer) - away_pre_elo

            if delta_column_to_calcuate == 'half_time_win_margin':
                pre_delta_adjustment = 0
                post_delta_adjustment = min(max(-5,(pre_elo_diff-4)/4),5)

            elif delta_column_to_calcuate == 'second_half_win_margin':                
                pre_delta_adjustment = 0
                post_delta_adjustment = min(max(-5,(pre_elo_diff-4)/3),5)
                
                
            else:
                
                pre_delta_adjustment = min(max(-7,(pre_elo_diff-4)/3.5),7)
                post_delta_adjustment = min(max(-5,(pre_elo_diff-5)/4),2)

                
                # New pre_elo_diff adjustment
                #pre_delta_adjustment = min(max(-7,(pre_elo_diff-2.5)/5),2)-0.22
                #post_delta_adjustment = min(max(-5,(pre_elo_diff-5)/4),2)

#                 pre_delta_adjustment = 0
#                 post_delta_adjustment = 0

                
            pre_elo_diff = pre_elo_diff - pre_delta_adjustment
            all_events.loc[event, pre_delta_diff_name] = pre_elo_diff
            all_events.loc[event, post_delta_adjustment_name] = post_delta_adjustment

            pre_delta_diff_adjusted_name = str(pre_delta_diff_name) + '_adjusted'
            all_events.loc[event, pre_delta_diff_adjusted_name] = pre_elo_diff - post_delta_adjustment

            
            if delta_column_to_calcuate in ['half_time_win_margin', 'second_half_win_margin']:
                delta_type_weight = 0.5
            else:
                delta_type_weight = 1


            home_team_home_error = get_team_homeaway_error('home', home_team_internal_id, home_team_total_fixture_number)
            away_team_home_error = get_team_homeaway_error('away', away_team_internal_id, away_team_total_fixture_number)
                
            all_events.loc[event, home_error] = home_team_home_error
            all_events.loc[event, away_error] = away_team_home_error
            

            if pd.isna(actual_win_margin):

                all_events.loc[event, home_post_delta_name] = home_pre_elo
                all_events.loc[event, away_post_delta_name] = away_pre_elo

            else:

                if (actual_win_margin < (pre_elo_diff - win_margin_buffer)):


                    all_events.loc[event, home_post_delta_name] = home_pre_elo - ((min(max_points_win, np.log(0.000001+abs(pre_elo_diff - actual_win_margin))*transfer_multiplier) - win_bonus) * delta_type_weight)
                    all_events.loc[event, away_post_delta_name] = away_pre_elo + ((min(max_points_win, np.log(0.000001+abs(pre_elo_diff - actual_win_margin))*transfer_multiplier) + win_bonus) * delta_type_weight)


                elif (actual_win_margin > (pre_elo_diff + win_margin_buffer)):


                    all_events.loc[event, home_post_delta_name] = home_pre_elo + ((min(max_points_win, np.log(0.000001+abs(pre_elo_diff - actual_win_margin))*transfer_multiplier) + win_bonus) * delta_type_weight)
                    all_events.loc[event, away_post_delta_name] = away_pre_elo - ((min(max_points_win, np.log(0.000001+abs(pre_elo_diff - actual_win_margin))*transfer_multiplier) - win_bonus) * delta_type_weight)

                else:

                    all_events.loc[event, home_post_delta_name] = home_pre_elo
                    all_events.loc[event, away_post_delta_name] = away_pre_elo

    function_end_time = datetime.datetime.now()
    print('--Complete-' + str(function_end_time - function_start_time))
    print('')




    return all_events


In [25]:
def generate_predicted_elo(all_events, team_internal_id, team_total_fixture_number, competition_id, competition_fixture_number, home_competition_group, start_time, level_setting, event_id, list_order_int_men, list_order_int_women_age, list_order_club, home_post_delta_name, away_post_delta_name, home_pre_delta_name, away_pre_delta_name, all_competitions, all_teams, internationl_rankings_df):

    
    comp_base_rate_fix_range = 50
    min_comp_base_rate_fix_range = 1
    pre_game_rank = None
    rank_set_type = None
    opp_pre_elo = None
    
    pre_game_rank_historic_home_competition_group_min = None
    pre_game_rank_historic_home_competition_group_median = None
    pre_game_rank_senior_team_ranking = None
    pre_game_rank_int_comp_level_setting = None
    pre_game_rank_new_home_competition_group = None
    pre_game_rank_historic_competition_median = None
    pre_game_rank_historic_competition_min = None

    dict_of_values = dict()
    
    if team_internal_id in other_initial_rankings.keys():
        pre_game_rank = other_initial_rankings[team_internal_id]
        rank_set_type = 'manual_entry'
        
    else:

        ###################### Get competition median

        games_to_aggregate = all_events[ (all_events['competition_internal_id'] == competition_id) & (all_events['competition_fixture_number'] < competition_fixture_number) & (all_events['competition_fixture_number'] >= (competition_fixture_number - 500)) ]
        if len(games_to_aggregate) > 1:

                home_games = games_to_aggregate.rename(columns = {home_pre_delta_name:'pre_game_rank'})
                away_games = games_to_aggregate.rename(columns = {away_pre_delta_name:'pre_game_rank'})
                all_games = pd.concat([home_games['pre_game_rank'], away_games['pre_game_rank']])
                pre_game_rank_historic_competition_min = all_games.min()
                pre_game_rank_historic_competition_median = all_games.median() 

        ############################################################################################################        
        # Get home competition group min value

        # Find home competition group if this game is currently not from a home competition
        if (pd.isna(home_competition_group) | (home_competition_group == 'na')):

            home_competition_group = all_events[ ((all_events['home_team_internal_id'] == team_internal_id) | (all_events['away_team_internal_id'] == team_internal_id)) & (all_events['start_time'] < start_time) & pd.notna(all_events['home_competition_group'])  & (all_events['home_competition_group'] != 'na') ]
            home_competition_group = home_competition_group.drop_duplicates('home_competition_group', keep = 'last')
            if len(home_competition_group)>0:
                home_competition_group = home_competition_group['home_competition_group'].iloc[0]
            else:
                home_competition_group = None




        if pd.notna(home_competition_group) & (home_competition_group != 'na'):

            # Get last game of their home competition group (it may not be this current game)
            home_competition_group_fixture_number = all_events[ (all_events['home_competition_group'] == home_competition_group) & (all_events['start_time'] < start_time)]['competition_fixture_number'].max()

            games_to_aggregate = all_events[ (all_events['home_competition_group'] == home_competition_group) & (all_events['home_competition_fixture_number'] < home_competition_group_fixture_number) & (all_events['home_competition_fixture_number'] >= (home_competition_group_fixture_number - comp_base_rate_fix_range)) ]

            ### If they are joining a new group
            if len(games_to_aggregate) > min_comp_base_rate_fix_range:

                home_games = games_to_aggregate.rename(columns = {home_pre_delta_name:'pre_game_rank'})
                away_games = games_to_aggregate.rename(columns = {away_pre_delta_name:'pre_game_rank'})
                all_games = pd.concat([home_games['pre_game_rank'], away_games['pre_game_rank']])
                pre_game_rank_historic_home_competition_group_min = all_games.min()
                pre_game_rank_historic_home_competition_group_median = all_games.median()


            ### If this is a new competition altogether then use the home_competition base set by international team rank and comp level
            comp_level = all_competitions[ all_competitions['home_competition_group'] == home_competition_group].iloc[0]['level']
            nat_team_id = all_teams[ all_teams['id'] == team_internal_id].iloc[0]['national_team_id']
            if pd.notna(nat_team_id):
                int_team_events = all_events[ ((all_events['home_team_internal_id'] == nat_team_id) | (all_events['away_team_internal_id'] == nat_team_id))  & (all_events['start_time'] < start_time)  & pd.notna(all_events[home_post_delta_name])   & pd.notna(all_events[away_post_delta_name]) ]

                if len(int_team_events)>0:

                    int_team_events = int_team_events.iloc[-1]

                    if int_team_events['home_team_internal_id'] == nat_team_id:
                        if comp_level == 1:
                            pre_game_rank_new_home_competition_group = int_team_events[home_post_delta_name]
                        else:
                            pre_game_rank_new_home_competition_group = int_team_events[home_post_delta_name] - ( (comp_level - 1) * level_setting )

                    elif int_team_events['away_team_internal_id'] == nat_team_id:
                        if comp_level == 1:
                            pre_game_rank_new_home_competition_group = int_team_events[away_post_delta_name]
                        else:
                            pre_game_rank_new_home_competition_group = int_team_events[away_post_delta_name] - ( (comp_level - 1) * level_setting )
                else:
                    # No previous games for their international team so just look up the dictionary to try and get it?
                    int_rank = internationl_rankings_df[ internationl_rankings_df['team_id'] == nat_team_id]

                    if len(int_rank)>0:
                        pre_game_rank_new_home_competition_group = int_rank.iloc[0]['pre_game_rank']

                        if comp_level > 1:
                            pre_game_rank_new_home_competition_group = pre_game_rank_new_home_competition_group - ( (comp_level - 1) * level_setting )


        # If there is no home_competition_group then just use the current competition they are playing in
        comp_level = all_competitions[ all_competitions['id'] == competition_id].iloc[0]['level']
        nat_team_id = all_teams[ all_teams['id'] == team_internal_id].iloc[0]['national_team_id']
        if pd.notna(nat_team_id):
            int_team_events = all_events[ ((all_events['home_team_internal_id'] == nat_team_id) | (all_events['away_team_internal_id'] == nat_team_id))  & (all_events['start_time'] < start_time)  & pd.notna(all_events[home_post_delta_name])   & pd.notna(all_events[away_post_delta_name]) ]

            if len(int_team_events)>0:

                int_team_events = int_team_events.iloc[-1]

                if int_team_events['home_team_internal_id'] == nat_team_id:
                    if comp_level == 1:
                        pre_game_rank_int_comp_level_setting = int_team_events[home_post_delta_name]
                    else:
                        pre_game_rank_int_comp_level_setting = int_team_events[home_post_delta_name] - ( (comp_level - 1) * level_setting )

                elif int_team_events['away_team_internal_id'] == nat_team_id:
                    if comp_level == 1:
                        pre_game_rank_int_comp_level_setting = int_team_events[away_post_delta_name]
                    else:
                        pre_game_rank_int_comp_level_setting = int_team_events[away_post_delta_name] - ( (comp_level - 1) * level_setting )

            else:
                # No previous games for their international team so just look up the dictionary to try and get it?
                int_rank = internationl_rankings_df[ internationl_rankings_df['team_id'] == nat_team_id]

                if len(int_rank)>0:
                    pre_game_rank_new_home_competition_group = int_rank.iloc[0]['pre_game_rank']

                    if comp_level > 1:
                        pre_game_rank_new_home_competition_group = pre_game_rank_new_home_competition_group - ( (comp_level - 1) * level_setting )



        ############################################################################################################        
        ############################################################################################################        




        ############################################################################################################        
        # Check to see if they are an A team
        # Using A teams etc add in from all teams
        team_details = all_teams[ all_teams['id'] == team_internal_id]
        senior_team = team_details['senior_team'].iloc[0]
        team_level = team_details['level'].iloc[0]
        team_type = team_details['type'].iloc[0]

        if not pd.isna(senior_team):

            senior_team_games = all_events[ ((all_events['home_team_internal_id'] == senior_team) | (all_events['away_team_internal_id'] == senior_team)) & (all_events['start_time'] < start_time)]
            if len(senior_team_games)>0:
                senior_team_games = senior_team_games.iloc[-1]

                if senior_team_games['home_team_internal_id'] == senior_team:
                    pre_game_rank_senior_team_ranking = senior_team_games[home_post_delta_name]
                if senior_team_games['away_team_internal_id'] == senior_team:
                    pre_game_rank_senior_team_ranking = senior_team_games[away_post_delta_name]

                if team_type != 'international_womens':
                        pre_game_rank_senior_team_ranking = pre_game_rank_senior_team_ranking - ((team_level - 1) * level_setting)
        ############################################################################################################        
        ############################################################################################################        



        ############################################################################################################
        ### Get their oppositions rank if they have one?
        if team_internal_id == all_events.loc[event_id]['home_team_internal_id']:
            opposition_id = all_events.loc[event_id]['away_team_internal_id']
            opposition_total_fixture_number = all_events.loc[event_id]['away_team_total_fixture_number']
        else:
            opposition_id = all_events.loc[event_id]['home_team_internal_id']
            opposition_total_fixture_number = all_events.loc[event_id]['home_team_total_fixture_number']

        home_events = all_events[ (all_events['home_team_internal_id'] == opposition_id) & (all_events['home_team_total_fixture_number'] == (opposition_total_fixture_number - 1)) ]
        if len(home_events)>0:
            opp_pre_elo = home_events.iloc[0][home_post_delta_name]

        else:
            away_events = all_events[ (all_events['away_team_internal_id'] == opposition_id) & (all_events['away_team_total_fixture_number'] == (opposition_total_fixture_number - 1)) ]
            if len(away_events) > 0:
                opp_pre_elo = away_events.iloc[0][away_post_delta_name]


        # Get median of all events
        home_events = all_events[ pd.notna(all_events[home_post_delta_name]) & (all_events['start_time'] < start_time) ][home_post_delta_name]
        away_events = all_events[ pd.notna(all_events[away_post_delta_name]) & (all_events['start_time'] < start_time) ][away_post_delta_name]
        all_events_median = pd.concat([home_events, away_events]).median()

        ############################################################################################################
        ############################################################################################################



        dict_of_values['pre_game_rank_historic_home_competition_group_min'] = pre_game_rank_historic_home_competition_group_min
        dict_of_values['pre_game_rank_historic_home_competition_group_median'] = pre_game_rank_historic_home_competition_group_median
        dict_of_values['pre_game_rank_new_home_competition_group'] = pre_game_rank_new_home_competition_group
        dict_of_values['pre_game_rank_senior_team_ranking'] = pre_game_rank_senior_team_ranking
        dict_of_values['pre_game_rank_int_comp_level_setting'] = pre_game_rank_int_comp_level_setting
        dict_of_values['opp_pre_elo'] = opp_pre_elo
        dict_of_values['all_events_median'] = all_events_median
        dict_of_values['pre_game_rank_historic_competition_min'] = pre_game_rank_historic_competition_min
        dict_of_values['pre_game_rank_historic_competition_median'] = pre_game_rank_historic_competition_median



        if (home_competition_group == 'international_mens'):
            list_order = list_order_int_men
        elif (home_competition_group == 'international_womens') | (home_competition_group == 'international_u21s') | (home_competition_group == 'international_u20s') | (home_competition_group == 'international_u19s'):
            list_order = list_order_int_women_age
        else:
            list_order = list_order_club

        pre_game_rank = None
        loopnum = 0
        while pd.isna(pre_game_rank):
            rank_set_type = list_order[loopnum]
            pre_game_rank = dict_of_values[list_order[loopnum]]
            loopnum = loopnum + 1

    return pre_game_rank, rank_set_type, pre_game_rank_historic_home_competition_group_min, pre_game_rank_historic_home_competition_group_median, pre_game_rank_senior_team_ranking, pre_game_rank_int_comp_level_setting, pre_game_rank_new_home_competition_group, pre_game_rank_historic_competition_min, pre_game_rank_historic_competition_median

In [26]:
def get_last_pre_elo(delta_type, all_events, team_internal_id, team_total_fixture_number, home_post_delta_name, away_post_delta_name, home_team_fixture_column, away_team_fixture_column):
    
    if delta_type == 'home':
        home_events = all_events[ (all_events['home_team_internal_id'] == team_internal_id) & (all_events[home_team_fixture_column] == (team_total_fixture_number-1)) ]
        post_game_rank = home_events.iloc[0][home_post_delta_name]
    if delta_type == 'away':
        away_events = all_events[ (all_events['away_team_internal_id'] == team_internal_id) & (all_events[away_team_fixture_column] == (team_total_fixture_number-1)) ]
        post_game_rank = away_events.iloc[0][home_post_delta_name]

        
    else:
        
        home_events = all_events[ (all_events['home_team_internal_id'] == team_internal_id) & (all_events[home_team_fixture_column] == (team_total_fixture_number-1)) ]

        if len(home_events)>0:
            post_game_rank = home_events.iloc[0][home_post_delta_name]

        else:

            away_events = all_events[ (all_events['away_team_internal_id'] == team_internal_id) & (all_events[away_team_fixture_column] == (team_total_fixture_number-1)) ]

            if len(away_events) > 0:
                post_game_rank = away_events.iloc[0][away_post_delta_name]


    return post_game_rank

In [27]:
def get_home_team_buffer(all_events, start_time, competition_id, competition_fixture_number, home_competition_group, home_competition_fixture_number, delta_column_to_calcuate):
    
    home_team_buffer = None
    all_events = all_events[ pd.notna(all_events[delta_column_to_calcuate])]
    
    if (home_competition_group != 'international_mens') & (home_competition_group != 'international_womens') & (home_competition_group != 'international_u21s') & (home_competition_group != 'international_u20s')  & (home_competition_group != 'international_u19s') & (home_competition_group != 'international_u18s'):

        temp_df = all_events[ (all_events['competition_internal_id'] == competition_id) & (all_events['competition_fixture_number'] < competition_fixture_number) & (all_events['competition_fixture_number'] >= (competition_fixture_number - 500)) ]
        
        # If there are more than 100 games in a competition then use this as the buffer
        if len(temp_df)>100:

            home_team_buffer = temp_df[delta_column_to_calcuate].median()

    else:

        if (home_competition_group != 'na') & (home_competition_group != '') & pd.notna(home_competition_group) & (home_competition_group != 'nan'):
        
            temp_df = all_events[ (all_events['home_competition_group'] == home_competition_group) & (all_events['home_competition_fixture_number'] < home_competition_fixture_number) & (all_events['home_competition_fixture_number'] >= (home_competition_fixture_number - 500)) ]


            if len(temp_df)>100:

                home_team_buffer = temp_df[delta_column_to_calcuate].median()


    if pd.isna(home_team_buffer):

            # If it is before 2010 then we can use our predefined buffers
            start_time_temp = pd.to_datetime(start_time).replace(tzinfo=None)
            if start_time_temp < pd.to_datetime('2010-01-01'):

                if home_competition_group == 'international_mens':
                    home_team_buffer = international_mens_base_home_win_margin
                elif home_competition_group == 'international_womens':
                    home_team_buffer = international_womens_base_home_win_margin
                else:
                    temp_df = competition_win_margin_means[ competition_win_margin_means['competition_internal_id'] == competition_id]

                    if len(temp_df)>0:
                        home_team_buffer = temp_df.iloc[0][delta_column_to_calcuate]
                    else:
                        home_team_buffer = all_base_home_win_margin


            else:

                # If there aren't more than 100 games in the home competition group then just use the global win_margin
                temp_df = all_events[ (all_events['start_time'] < start_time) & (all_events['start_time'] >= (start_time - datetime.timedelta(days = (365 * 5))) )]
                home_team_buffer = temp_df[delta_column_to_calcuate].median()

                
    return home_team_buffer

In [28]:
def team_dict_to_dataframe(team_rankings):

    function_start_time = datetime.datetime.now()
    print('-team_dict_to_dataframe')
    
    team_df = pd.DataFrame()
    team_df = pd.DataFrame(team_rankings.values(), index = team_rankings.keys()).reset_index().rename(columns = {'index':'team_id', 0:'pre_game_rank'})
        
        
    function_end_time = datetime.datetime.now()
    print('--Complete-' + str(function_end_time - function_start_time))
    print('')

    return team_df


In [29]:
def check_for_nonexistant_events(all_previous_deltas, all_events):

    function_start_time = datetime.datetime.now()
    print('-check_for_nonexistant_events')
    
    # Check to see if there are any events in the deltas table that are no longer events for whatever reason

    events_that_no_longer_exists = list(all_previous_deltas[ ~all_previous_deltas['event_id'].isin(all_events['event_id']) ]['event_id'])

    if len(events_that_no_longer_exists)>0:

        event_list_string = ''

        for event in events_that_no_longer_exists:

            event_list_string = event_list_string + "'" + str(event) + "',"


        sql_statement = 'delete from event_deltas where event_id in (' + event_list_string[:-1] + ');'
        postgres_Retreive_Insert(sql_statement, POSTGRESQL_PARAMS, False)

        print('--', str(len(events_that_no_longer_exists)), ' - events removed')
    
    function_end_time = datetime.datetime.now()
    print('--Complete-' + str(function_end_time - function_start_time))
    print('')

    return

In [30]:
def update_existing_events(all_events, home_pre_delta_name, home_post_delta_name, away_pre_delta_name, away_post_delta_name, pre_delta_diff_name, home_team_buffer_name, post_delta_adjustment_name):


    function_start_time = datetime.datetime.now()
    print('-update_existing_events')
    
    sql_statement_full = None
    existing_events = all_events[start_range:]
    existing_events = existing_events[ existing_events['event_id'].isin(all_previous_deltas['event_id'])]

    if len(existing_events)>0:

        sql_statement_full = ''
        loop_num = 1

        for event in existing_events.index:

            #print(loop_num, len(existing_events))

            sql_statement = 'update event_deltas set '

            event_id = existing_events.loc[event, 'event_id']
            home_team_internal_id = existing_events.loc[event, 'home_team_internal_id']
            away_team_internal_id = existing_events.loc[event, 'away_team_internal_id']
            competition_internal_id = existing_events.loc[event, 'competition_internal_id']
            venue_internal_id = existing_events.loc[event, 'venue_internal_id']
            
            #venue_internal_id = existing_events.loc[event, 'venue_internal_id']
            #if pd.isna(venue_internal_id) | (venue_internal_id == 'None') | (venue_internal_id == 'NaN'):
            #    venue_internal_id = "null"

            start_time = existing_events.loc[event, 'start_time']
            home_score = existing_events.loc[event, 'home_score']
            away_score = existing_events.loc[event, 'away_score']
            
            home_team_buffer = existing_events.loc[event, home_team_buffer_name]
            home_pre_delta = existing_events.loc[event, home_pre_delta_name]
            away_pre_delta = existing_events.loc[event, away_pre_delta_name]
            pre_delta_diff = existing_events.loc[event, pre_delta_diff_name]
            home_post_delta = existing_events.loc[event, home_post_delta_name]
            away_post_delta = existing_events.loc[event, away_post_delta_name]
            
            pre_delta_adjustment = existing_events.loc[event, post_delta_adjustment_name]

            if pd.isna(home_pre_delta):
                home_pre_delta = 'NULL'
            if pd.isna(away_pre_delta):
                away_pre_delta = 'NULL'
            if pd.isna(home_post_delta):
                home_post_delta = 'NULL'
            if pd.isna(away_post_delta):
                away_post_delta = 'NULL'
            if pd.isna(home_team_buffer):
                home_team_buffer = 'NULL'
            if pd.isna(home_score):
                home_score = 'NULL'
            if pd.isna(away_score):
                away_score = 'NULL'
            if pd.isna(pre_delta_adjustment):
                pre_delta_adjustment = 'NULL'    

            sql_statement = sql_statement + str(home_team_buffer_name) + " = "  + str(home_team_buffer) + ","
            sql_statement = sql_statement + str(home_pre_delta_name) + " = "  + str(home_pre_delta) + ","
            sql_statement = sql_statement + str(away_pre_delta_name) + " = "  + str(away_pre_delta) + ","
            sql_statement = sql_statement + str(pre_delta_diff_name) + " = "  + str(pre_delta_diff) + ","
            sql_statement = sql_statement + str(home_post_delta_name) + " = "  + str(home_post_delta) + ","
            sql_statement = sql_statement + str(away_post_delta_name) + " = "  + str(away_post_delta) + ","
            sql_statement = sql_statement + str(post_delta_adjustment_name) + " = "  + str(pre_delta_adjustment) + ","

            sql_statement = sql_statement + str('home_team_internal_id') + " = '"  + str(home_team_internal_id) + "',"
            sql_statement = sql_statement + str('away_team_internal_id') + " = '"  + str(away_team_internal_id) + "',"
            if pd.notna(venue_internal_id) & (venue_internal_id != 'None') & (venue_internal_id != 'Null'):
                sql_statement = sql_statement + str('venue_internal_id') + " = '"  + str(venue_internal_id) + "',"
            sql_statement = sql_statement + str('competition_internal_id') + " = '"  + str(competition_internal_id) + "',"
            sql_statement = sql_statement + str('start_time') + " = '"  + str(start_time) + "',"
            sql_statement = sql_statement + str('home_score') + " = "  + str(home_score) + ","
            sql_statement = sql_statement + str('away_score') + " = "  + str(away_score)

            sql_statement = sql_statement + " where event_id = '" + str(event_id) + "';"


            sql_statement_full = sql_statement_full + sql_statement
            loop_num += 1
            
            
    if pd.notna(sql_statement_full):
        postgres_Retreive_Insert(sql_statement_full, POSTGRESQL_PARAMS, False)
        
    function_end_time = datetime.datetime.now()
    print('--Complete-' + str(function_end_time - function_start_time))
    print('')


In [31]:
def add_new_events(all_events, pre_delta_diff_name, all_previous_deltas):


    function_start_time = datetime.datetime.now()
    print('-add_new_events')
    
    new_events = all_events[ ~all_events['event_id'].isin(all_previous_deltas['event_id'])]
    #new_events = new_events[ (pd.notna(new_events['venue_internal_id'])) & (new_events['venue_internal_id'] != 'None')]

    sql_statement_full_venues = ''
    sql_statement_full_no_venues = ''
    if len(new_events)>0:

        sql_statement_start_venues = 'insert into event_deltas (event_id, home_team_internal_id, away_team_internal_id, competition_internal_id, venue_internal_id, start_time, home_score, away_score, home_team_buffer, pre_delta_adjustment, home_pre_delta, away_pre_delta, pre_delta_diff, home_post_delta, away_post_delta) values '
        sql_statement_start_no_venues = 'insert into event_deltas (event_id, home_team_internal_id, away_team_internal_id, competition_internal_id, start_time, home_score, away_score, home_team_buffer, pre_delta_adjustment, home_pre_delta, away_pre_delta, pre_delta_diff, home_post_delta, away_post_delta) values '

        for event in new_events.index:

            sql_statement = ''

            event_id = new_events.loc[event, 'event_id']
            home_team_internal_id = new_events.loc[event, 'home_team_internal_id']
            away_team_internal_id = new_events.loc[event, 'away_team_internal_id']
            competition_internal_id = new_events.loc[event, 'competition_internal_id']
            
            venue_internal_id = new_events.loc[event, 'venue_internal_id']
            venue_exists = False
            if pd.notna(venue_internal_id) & (venue_internal_id != 'None') & (venue_internal_id != 'NaN'):
                venue_exists = True

            start_time = new_events.loc[event, 'start_time']
            home_score = new_events.loc[event, 'home_score']
            away_score = new_events.loc[event, 'away_score']
            home_pre_delta = new_events.loc[event, 'home_pre_delta']
            away_pre_delta = new_events.loc[event, 'away_pre_delta']
            pre_delta_diff = new_events.loc[event, pre_delta_diff_name]
            home_post_delta = new_events.loc[event, 'home_post_delta']
            away_post_delta = new_events.loc[event, 'away_post_delta']
            home_team_buffer = new_events.loc[event, 'home_team_buffer']
            pre_delta_adjustment = new_events.loc[event, 'pre_delta_adjustment']
            #updated_at = datetime.datetime.now()
            #created_at = datetime.datetime.now()


            if pd.isna(home_score):
                home_score = 'NULL'
            if pd.isna(away_score):
                away_score = 'NULL'
            if pd.isna(home_pre_delta):
                home_pre_delta = 'NULL'
            if pd.isna(away_pre_delta):
                away_pre_delta = 'NULL'
            if pd.isna(home_post_delta):
                home_post_delta = 'NULL'
            if pd.isna(away_post_delta):
                away_post_delta = 'NULL'
            if pd.isna(pre_delta_adjustment):
                pre_delta_adjustment = 'NULL'


            sql_statement = sql_statement + "('"  + str(event_id) + "', "
            sql_statement = sql_statement + "'"  + str(home_team_internal_id) + "', "
            sql_statement = sql_statement + "'"  + str(away_team_internal_id) + "', "
            sql_statement = sql_statement + "'"  + str(competition_internal_id) + "', "
            
            if venue_exists:
                sql_statement = sql_statement + "'"  + str(venue_internal_id) + "', "
                
            sql_statement = sql_statement + "'"  + str(start_time) + "', "
            sql_statement = sql_statement + str(home_score) + ", "
            sql_statement = sql_statement + str(away_score) + ", "
            sql_statement = sql_statement + str(home_team_buffer) + ", "
            sql_statement = sql_statement + str(pre_delta_adjustment) + ", "
            sql_statement = sql_statement + str(home_pre_delta) + ", "
            sql_statement = sql_statement + str(away_pre_delta) + ", "
            sql_statement = sql_statement + str(pre_delta_diff) + ", "
            sql_statement = sql_statement + str(home_post_delta) + ", "
            sql_statement = sql_statement + str(away_post_delta) + "), "
            #sql_statement = sql_statement + "'"  + str(updated_at) + "', "
            #sql_statement = sql_statement + "'"  + str(created_at) + "'), "

            if venue_exists:
                sql_statement_full_venues = sql_statement_full_venues + sql_statement
            else:
                sql_statement_full_no_venues = sql_statement_full_no_venues + sql_statement

    if len(sql_statement_full_venues) > 0:
        sql_statement_full_venues = sql_statement_full_venues[:-2]
        sql_statement_full_venues = sql_statement_start_venues + sql_statement_full_venues + ";"
        postgres_Retreive_Insert(sql_statement_full_venues, POSTGRESQL_PARAMS, False)

    if len(sql_statement_full_no_venues) > 0:
        sql_statement_full_no_venues = sql_statement_full_no_venues[:-2]
        sql_statement_full_no_venues = sql_statement_start_no_venues + sql_statement_full_no_venues + ";"
        postgres_Retreive_Insert(sql_statement_full_no_venues, POSTGRESQL_PARAMS, False)

    
    function_end_time = datetime.datetime.now()
    print('--Complete-' + str(function_end_time - function_start_time))
    print('')


In [32]:
def add_venue_info(all_events):
    
    sql_statement = 'Select * from venue'
    venues, error = postgres_Retreive_Insert(sql_statement, POSTGRESQL_PARAMS, True)
    all_events = all_events.merge(venues[['id', 'national_team']].rename(columns = {'id':'venue_internal_id','national_team':'venue_national_team'}), how = 'left', left_on = 'venue_internal_id', right_on = 'venue_internal_id')

    return all_events, venues

In [33]:
def get_all_deltas():

    float_columns = ['home_pre_delta', 'home_post_delta', 'away_pre_delta', 'away_post_delta', 'pre_delta_diff', 'home_team_buffer', 'home_pre_delta_halftime', 'home_post_delta_halftime', 'away_pre_delta_halftime', 'away_post_delta_halftime', 'pre_delta_diff_halftime', 'home_team_buffer_halftime', 'home_pre_delta_secondhalf', 'home_post_delta_secondhalf', 'away_pre_delta_secondhalf', 'away_post_delta_secondhalf', 'pre_delta_diff_secondhalf', 'home_team_buffer_secondhalf']
    all_previous_deltas = get_all_previous_deltas(float_columns)

    home_games = all_previous_deltas[['start_time','home_team_internal_id', 'home_pre_delta']].rename(columns = {'home_team_internal_id':'team_id', 'home_pre_delta':'delta'})
    away_games = all_previous_deltas[['start_time','away_team_internal_id', 'away_pre_delta']].rename(columns = {'away_team_internal_id':'team_id', 'away_pre_delta':'delta'})
    all_deltas = pd.concat([home_games,away_games])

    return all_deltas

In [34]:
def get_initial_delta(all_deltas, team_id, start_time, attack_defence):

    all_deltas = all_deltas[ (all_deltas['team_id'] == team_id) & (all_deltas['start_time'] <= start_time)]
    all_deltas.sort_values('start_time', ascending = False, inplace = True)

    if attack_defence == 'attack':

        delta = 1.0543565 * all_deltas.iloc[0]['delta']

    elif attack_defence == 'defence':

        delta = -0.92410307 * all_deltas.iloc[0]['delta']


    return delta

In [35]:
def update_event_deltas_total_points(all_events, start_range, end_range, home_pre_delta_name_1, home_pre_delta_name_2, home_pre_delta_name_3, home_pre_delta_name_4, away_pre_delta_name_1, away_pre_delta_name_2, away_pre_delta_name_3, away_pre_delta_name_4, home_post_delta_name_1, home_post_delta_name_2, home_post_delta_name_3, home_post_delta_name_4, away_post_delta_name_1, away_post_delta_name_2, away_post_delta_name_3, away_post_delta_name_4):


    function_start_time = datetime.datetime.now()
    print('-update_existing_events')
    
    sql_statement_full = None
    existing_events = all_events[start_range:]

    if len(existing_events)>0:

        sql_statement_full = ''
        loop_num = 1

        for event in existing_events.index:

            #print(loop_num, len(existing_events))

            sql_statement = 'update event_deltas set '

            event_id = existing_events.loc[event, 'event_id']
            
    
            home_pre_delta_1 = existing_events.loc[event, home_pre_delta_name_1]
            home_pre_delta_2 = existing_events.loc[event, home_pre_delta_name_2]
            home_pre_delta_3 = existing_events.loc[event, home_pre_delta_name_3]
            home_pre_delta_4 = existing_events.loc[event, home_pre_delta_name_4]

            home_post_delta_1 = existing_events.loc[event, home_post_delta_name_1]
            home_post_delta_2 = existing_events.loc[event, home_post_delta_name_2]
            home_post_delta_3 = existing_events.loc[event, home_post_delta_name_3]
            home_post_delta_4 = existing_events.loc[event, home_post_delta_name_4]

            away_pre_delta_1 = existing_events.loc[event, away_pre_delta_name_1]
            away_pre_delta_2 = existing_events.loc[event, away_pre_delta_name_2]
            away_pre_delta_3 = existing_events.loc[event, away_pre_delta_name_3]
            away_pre_delta_4 = existing_events.loc[event, away_pre_delta_name_4]

            away_post_delta_1 = existing_events.loc[event, away_post_delta_name_1]
            away_post_delta_2 = existing_events.loc[event, away_post_delta_name_2]
            away_post_delta_3 = existing_events.loc[event, away_post_delta_name_3]
            away_post_delta_4 = existing_events.loc[event, away_post_delta_name_4]


            if pd.isna(home_pre_delta_1):
                home_pre_delta_1 = 'NULL'
            if pd.isna(home_pre_delta_2):
                home_pre_delta_2 = 'NULL'
            if pd.isna(home_pre_delta_3):
                home_pre_delta_3 = 'NULL'
            if pd.isna(home_pre_delta_4):
                home_pre_delta_4 = 'NULL'

            if pd.isna(home_post_delta_1):
                home_post_delta_1 = 'NULL'
            if pd.isna(home_post_delta_2):
                home_post_delta_2 = 'NULL'
            if pd.isna(home_post_delta_3):
                home_post_delta_3 = 'NULL'
            if pd.isna(home_post_delta_4):
                home_post_delta_4 = 'NULL'

            if pd.isna(away_pre_delta_1):
                away_pre_delta_1 = 'NULL'
            if pd.isna(away_pre_delta_2):
                away_pre_delta_2 = 'NULL'
            if pd.isna(away_pre_delta_3):
                away_pre_delta_3 = 'NULL'
            if pd.isna(away_pre_delta_4):
                away_pre_delta_4 = 'NULL'

            if pd.isna(away_post_delta_1):
                away_post_delta_1 = 'NULL'
            if pd.isna(away_post_delta_2):
                away_post_delta_2 = 'NULL'
            if pd.isna(away_post_delta_3):
                away_post_delta_3 = 'NULL'
            if pd.isna(away_post_delta_4):
                away_post_delta_4 = 'NULL'


            sql_statement = sql_statement + str(home_pre_delta_name_1) + " = "  + str(home_pre_delta_1) + ","
            sql_statement = sql_statement + str(home_pre_delta_name_2) + " = "  + str(home_pre_delta_2) + ","
            sql_statement = sql_statement + str(home_pre_delta_name_3) + " = "  + str(home_pre_delta_3) + ","
            sql_statement = sql_statement + str(home_pre_delta_name_4) + " = "  + str(home_pre_delta_4) + ","
            sql_statement = sql_statement + str(home_post_delta_name_1) + " = "  + str(home_post_delta_1) + ","
            sql_statement = sql_statement + str(home_post_delta_name_2) + " = "  + str(home_post_delta_2) + ","
            sql_statement = sql_statement + str(home_post_delta_name_3) + " = "  + str(home_post_delta_3) + ","
            sql_statement = sql_statement + str(home_post_delta_name_4) + " = "  + str(home_post_delta_4) + ","
            sql_statement = sql_statement + str(away_pre_delta_name_1) + " = "  + str(away_pre_delta_1) + ","
            sql_statement = sql_statement + str(away_pre_delta_name_2) + " = "  + str(away_pre_delta_2) + ","
            sql_statement = sql_statement + str(away_pre_delta_name_3) + " = "  + str(away_pre_delta_3) + ","
            sql_statement = sql_statement + str(away_pre_delta_name_4) + " = "  + str(away_pre_delta_4) + ","
            sql_statement = sql_statement + str(away_post_delta_name_1) + " = "  + str(away_post_delta_1) + ","
            sql_statement = sql_statement + str(away_post_delta_name_2) + " = "  + str(away_post_delta_2) + ","
            sql_statement = sql_statement + str(away_post_delta_name_3) + " = "  + str(away_post_delta_3) + ","
            sql_statement = sql_statement + str(away_post_delta_name_4) + " = "  + str(away_post_delta_4)

            sql_statement = sql_statement + " where event_id = '" + str(event_id) + "';"


            sql_statement_full = sql_statement_full + sql_statement
            loop_num += 1
            
            
    if pd.notna(sql_statement_full):
        postgres_Retreive_Insert(sql_statement_full, POSTGRESQL_PARAMS, False)
        
    function_end_time = datetime.datetime.now()
    print('--Complete-' + str(function_end_time - function_start_time))
    print('')

    

In [36]:
def get_team_homeaway_error(home_away, team_internal_id, team_total_fixture_number):
    
    if home_away == 'home':
        events = all_events[ (all_events['home_team_internal_id'] == team_internal_id) & (all_events['home_team_total_fixture_number'] < team_total_fixture_number) & (all_events['home_team_total_fixture_number'] >= (team_total_fixture_number - 10)) ].copy()
    elif home_away == 'away':
        events = all_events[ (all_events['away_team_internal_id'] == team_internal_id) & (all_events['away_team_total_fixture_number'] < team_total_fixture_number) & (all_events['away_team_total_fixture_number'] >= (team_total_fixture_number - 10)) ].copy()
    
    #events['error'] = events[pre_delta_diff_adjusted] - events[delta_column_to_calcuate]
    
    events.loc[:, 'error'] = events.loc[:, pre_delta_diff_adjusted] - events.loc[:, delta_column_to_calcuate]

    
    event_error = events['error'].median()
    
    return event_error
    

In [37]:
def calculate_total_points_deltas():


    all_deltas = get_all_deltas()    
    
    function_start_time = datetime.datetime.now()
    
    all_events.reset_index(inplace = True, drop = True)
    #all_events['post_game_rank_home'] = None
    #all_events['post_game_rank_away'] = None
    #all_events[home_team_buffer_name] = None
    all_events['pre_game_rank_historic_home_competition_group_min_home'] = None
    all_events['pre_game_rank_historic_home_competition_group_median_home'] = None
    all_events['pre_game_rank_senior_team_ranking_home'] = None
    all_events['pre_game_rank_int_comp_level_setting_home'] = None
    all_events['pre_game_rank_new_home_competition_group_home'] = None
    all_events['pre_game_rank_historic_home_competition_group_min_away'] = None
    all_events['pre_game_rank_historic_home_competition_group_median_away'] = None
    all_events['pre_game_rank_senior_team_ranking_away'] = None
    all_events['pre_game_rank_int_comp_level_setting_away'] = None
    all_events['pre_game_rank_new_home_competition_group_away'] = None
    all_events['base_value'] = None

    #all_events['pred_home_score_all'] = None
    #all_events['pred_home_score_ha'] = None
    #all_events['pred_away_score_all'] = None
    #all_events['pred_away_score_ha'] = None
    
    #all_events['home_adjustment'] = None
    #all_events['away_adjustment'] = None
    pre_elo_diff = None

    
    print('--updating from', start_range, end_range)
    for event in range(start_range,end_range+1):
        
            print('--', event, end_range)

            home_team_internal_id = all_events.loc[event, 'home_team_internal_id']
            home_team_home_fixture_number = all_events.loc[event, 'home_team_home_fixture_number']
            home_team_total_fixture_number = all_events.loc[event, 'home_team_total_fixture_number']

            away_team_internal_id = all_events.loc[event, 'away_team_internal_id']
            away_team_away_fixture_number = all_events.loc[event, 'away_team_away_fixture_number']
            away_team_total_fixture_number = all_events.loc[event, 'away_team_total_fixture_number']

            actual_target_value_1 = all_events.loc[event, delta_column_to_calcuate_1]
            actual_target_value_2 = all_events.loc[event, delta_column_to_calcuate_2]

            home_venue = all_events.loc[event, 'home_venue']

            start_time = all_events.loc[event, 'start_time']
            competition_id = all_events.loc[event, 'competition_internal_id']
            competition_fixture_number = all_events.loc[event, 'competition_fixture_number']
            home_competition_group = all_events.loc[event, 'home_competition_group']
            home_competition_fixture_number = all_events.loc[event, 'home_competition_fixture_number']


            home_team_buffer = 0
            if get_home_advantage == True:
                
                # Get team travelled buffer
                travel_buffer = all_events.loc[event, 'travel_advantage']
                if travel_buffer == -1:
                    home_venue = True

                        
            
            if home_team_total_fixture_number == 1:
                
#                 if away_team_total_fixture_number > 1:
                    
#                     current_score = all_events.loc[event, 'home_score']
#                     away_defence = get_last_pre_elo(None, all_events, away_team_internal_id, away_team_total_fixture_number, home_post_delta_name_2, away_post_delta_name_2, 'home_team_total_fixture_number', 'away_team_total_fixture_number')
#                     home_attack = current_total_points - away_defence
                    
#                     away_attack = get_last_pre_elo(None, all_events, away_team_internal_id, away_team_total_fixture_number, home_post_delta_name_1, away_post_delta_name_1, 'home_team_total_fixture_number', 'away_team_total_fixture_number')
#                     home_defence = current_score - away_attack

                    
#                 else:

                    home_attack = get_initial_delta(all_deltas, home_team_internal_id, start_time, 'attack')
                    home_defence = get_initial_delta(all_deltas, home_team_internal_id, start_time, 'defence')
                
                # Need to fix this bit so it pulls back home_homes when needed etc
                #if pd.isna(home_attack):
                #    home_attack, rank_set_type, pre_game_rank_historic_home_competition_group_min, pre_game_rank_historic_home_competition_group_median, pre_game_rank_senior_team_ranking, pre_game_rank_int_comp_level_setting, pre_game_rank_new_home_competition_group, pre_game_rank_historic_competition_min, pre_game_rank_historic_competition_median = generate_predicted_elo(all_events, home_team_internal_id, home_team_total_fixture_number, competition_id, competition_fixture_number, home_competition_group, start_time, level_setting, event, list_order_int_men, list_order_int_women_age, list_order_club, home_post_delta_name_1, away_post_delta_name_2, home_pre_delta_name_1, away_pre_delta_name_2, all_competitions, all_teams, internationl_rankings_df)
                #if pd.isna(home_defence):
                #    home_defence, rank_set_type, pre_game_rank_historic_home_competition_group_min, pre_game_rank_historic_home_competition_group_median, pre_game_rank_senior_team_ranking, pre_game_rank_int_comp_level_setting, pre_game_rank_new_home_competition_group, pre_game_rank_historic_competition_min, pre_game_rank_historic_competition_median = generate_predicted_elo(all_events, home_team_internal_id, home_team_total_fixture_number, competition_id, competition_fixture_number, home_competition_group, start_time, level_setting, event, list_order_int_men, list_order_int_women_age, list_order_club, home_post_delta_name_2, away_post_delta_name_1, home_pre_delta_name_2, away_pre_delta_name_1, all_competitions, all_teams, internationl_rankings_df)

            else:
                
                if home_venue & (travel_buffer != -1):
                    home_attack = get_last_pre_elo(None, all_events, home_team_internal_id, home_team_total_fixture_number, home_post_delta_name_1, away_post_delta_name_1, 'home_team_total_fixture_number', 'away_team_total_fixture_number')
                    home_defence = get_last_pre_elo(None, all_events, home_team_internal_id, home_team_total_fixture_number, home_post_delta_name_2, away_post_delta_name_2, 'home_team_total_fixture_number', 'away_team_total_fixture_number')
                else:
                    home_attack = get_last_pre_elo(None, all_events, home_team_internal_id, home_team_total_fixture_number, home_post_delta_name_1, away_post_delta_name_1, 'home_team_total_fixture_number', 'away_team_total_fixture_number')
                    home_defence = get_last_pre_elo(None, all_events, home_team_internal_id, home_team_total_fixture_number, home_post_delta_name_2, away_post_delta_name_2, 'home_team_total_fixture_number', 'away_team_total_fixture_number')

            
            if home_team_home_fixture_number == 1:
                
#                 if away_team_total_fixture_number > 1:
                    
#                     current_score = all_events.loc[event, 'home_score']
#                     away_defence = get_last_pre_elo(None, all_events, away_team_internal_id, away_team_total_fixture_number, home_post_delta_name_2, away_post_delta_name_2, 'home_team_total_fixture_number', 'away_team_total_fixture_number')
#                     home_home_attack = current_total_points - away_defence
                    
#                     away_attack = get_last_pre_elo(None, all_events, away_team_internal_id, away_team_total_fixture_number, home_post_delta_name_1, away_post_delta_name_1, 'home_team_total_fixture_number', 'away_team_total_fixture_number')
#                     home_home_defence = current_score - away_attack

                    
#                 else:

                    home_home_attack = get_initial_delta(all_deltas, home_team_internal_id, start_time, 'attack')
                    home_home_defence = get_initial_delta(all_deltas, home_team_internal_id, start_time, 'defence')
                
            else:
                
                if home_venue & (travel_buffer != -1):
                    home_home_attack = get_last_pre_elo('home', all_events, home_team_internal_id, home_team_home_fixture_number, home_post_delta_name_3, away_post_delta_name_3, 'home_team_home_fixture_number', 'away_team_away_fixture_number')
                    home_home_defence = get_last_pre_elo('home', all_events, home_team_internal_id, home_team_home_fixture_number, home_post_delta_name_4, away_post_delta_name_4, 'home_team_home_fixture_number', 'away_team_away_fixture_number')
                else:
                    home_home_attack = get_last_pre_elo(None, all_events, home_team_internal_id, home_team_total_fixture_number, home_post_delta_name_1, away_post_delta_name_1, 'home_team_total_fixture_number', 'away_team_total_fixture_number')
                    home_home_defence = get_last_pre_elo(None, all_events, home_team_internal_id, home_team_total_fixture_number, home_post_delta_name_2, away_post_delta_name_2, 'home_team_total_fixture_number', 'away_team_total_fixture_number')


            
            all_events.loc[event, home_pre_delta_name_1] = home_attack
            all_events.loc[event, home_pre_delta_name_2] = home_defence
            all_events.loc[event, home_pre_delta_name_3] = home_home_attack
            all_events.loc[event, home_pre_delta_name_4] = home_home_defence

            
            if away_team_total_fixture_number == 1:
                
#                 if home_team_total_fixture_number > 1:
                    
#                     current_score = all_events.loc[event, 'away_score']
#                     home_defence = get_last_pre_elo(None, all_events, home_team_internal_id, home_team_total_fixture_number, home_post_delta_name_2, away_post_delta_name_2, 'home_team_total_fixture_number', 'away_team_total_fixture_number')
#                     away_attack = current_score - home_defence
                    
#                     home_attack = get_last_pre_elo(None, all_events, home_team_internal_id, home_team_total_fixture_number, home_post_delta_name_1, away_post_delta_name_1, 'home_team_total_fixture_number', 'away_team_total_fixture_number')
#                     away_defence = current_total_points - home_attack
                    

#                 else:

                    away_attack = get_initial_delta(all_deltas, away_team_internal_id, start_time, 'attack')
                    away_defence = get_initial_delta(all_deltas, away_team_internal_id, start_time, 'defence')               
            
            else:
                
                if home_venue & (travel_buffer != -1):
                    away_attack = get_last_pre_elo(None, all_events, away_team_internal_id, away_team_total_fixture_number, home_post_delta_name_1, away_post_delta_name_1, 'home_team_total_fixture_number', 'away_team_total_fixture_number')
                    away_defence = get_last_pre_elo(None, all_events, away_team_internal_id, away_team_total_fixture_number, home_post_delta_name_2, away_post_delta_name_2, 'home_team_total_fixture_number', 'away_team_total_fixture_number')
                else:
                    away_attack = get_last_pre_elo(None, all_events, away_team_internal_id, away_team_total_fixture_number, home_post_delta_name_1, away_post_delta_name_1, 'home_team_total_fixture_number', 'away_team_total_fixture_number')
                    away_defence = get_last_pre_elo(None, all_events, away_team_internal_id, away_team_total_fixture_number, home_post_delta_name_2, away_post_delta_name_2, 'home_team_total_fixture_number', 'away_team_total_fixture_number')

                    
            if away_team_away_fixture_number == 1:
                
#                 if home_team_total_fixture_number > 1:
                    
#                     current_score = all_events.loc[event, 'away_score']
#                     home_defence = get_last_pre_elo(None, all_events, home_team_internal_id, home_team_total_fixture_number, home_post_delta_name_2, away_post_delta_name_2, 'home_team_total_fixture_number', 'away_team_total_fixture_number')
#                     away_away_attack = current_score - home_defence
                    
#                     home_attack = get_last_pre_elo(None, all_events, home_team_internal_id, home_team_total_fixture_number, home_post_delta_name_1, away_post_delta_name_1, 'home_team_total_fixture_number', 'away_team_total_fixture_number')
#                     away_away_defence = current_total_points - home_attack

#                 else:
                    
                    away_away_attack = get_initial_delta(all_deltas, away_team_internal_id, start_time, 'attack')
                    away_away_defence = get_initial_delta(all_deltas, away_team_internal_id, start_time, 'defence')
                
            else:
                
                if home_venue & (travel_buffer != -1):
                    away_away_attack = get_last_pre_elo('away', all_events, away_team_internal_id, away_team_away_fixture_number, home_post_delta_name_3, away_post_delta_name_3, 'home_team_home_fixture_number', 'away_team_away_fixture_number')
                    away_away_defence = get_last_pre_elo('away', all_events, away_team_internal_id, away_team_away_fixture_number, home_post_delta_name_4, away_post_delta_name_4, 'home_team_home_fixture_number', 'away_team_away_fixture_number')
                else:
                    away_away_attack = get_last_pre_elo(None, all_events, away_team_internal_id, away_team_away_fixture_number, home_post_delta_name_1, away_post_delta_name_1, 'home_team_total_fixture_number', 'away_team_total_fixture_number')
                    away_away_defence = get_last_pre_elo(None, all_events, away_team_internal_id, away_team_away_fixture_number, home_post_delta_name_2, away_post_delta_name_2, 'home_team_total_fixture_number', 'away_team_total_fixture_number')

                
                
                
            all_events.loc[event, away_pre_delta_name_1] = away_attack
            all_events.loc[event, away_pre_delta_name_2] = away_defence
            all_events.loc[event, away_pre_delta_name_3] = away_away_attack
            all_events.loc[event, away_pre_delta_name_4] = away_away_defence

            
            
                
            base_value = 0
            #if get_base == True:
            #    base_value = get_base_value(buffer_type, all_events, start_time, competition_id, competition_fixture_number, home_competition_group, home_competition_fixture_number, delta_column_to_calcuate)
            
            
           


            pred_home_score_all = base_value + (home_attack + away_defence)
            pred_home_score_ha = base_value + (home_home_attack + away_away_defence)
            pred_away_score_all = base_value + (home_defence + away_attack)
            pred_away_score_ha = base_value + (home_home_defence + away_away_attack)
            

            # New home_attack adjustement
             
            if(pred_home_score_all<=20):
                home_adjustment = (pred_home_score_all/1.35)-15
            else:
                home_adjustment = (pred_home_score_all/4.5)-5
            
            pred_home_score_all = pred_home_score_all - home_adjustment
            pred_home_score_ha = pred_home_score_ha - home_adjustment
            
            if pred_away_score_all<=16:
                away_adjustement = (pred_away_score_all/1.35)-12
            else:
                away_adjustement = ((pred_away_score_all/2)-8)
                
            pred_away_score_all = pred_away_score_all - away_adjustement
            pred_away_score_ha = pred_away_score_ha - away_adjustement
                    
            
            all_events.loc[event, 'pred_home_score_all'] = pred_home_score_all
            all_events.loc[event, 'pred_home_score_ha'] = pred_home_score_ha
            all_events.loc[event, 'pred_away_score_all'] = pred_away_score_all
            all_events.loc[event, 'pred_away_score_ha'] = pred_away_score_ha
            

            all_events.loc[event, 'home_adjustment'] = home_adjustment
            all_events.loc[event, 'away_adjustement'] = away_adjustement

            
            all_events.loc[event, 'base_value'] = base_value
            all_events.loc[event, home_team_buffer_name] = home_team_buffer
            all_events.loc[event, pre_delta_diff_name] = pre_elo_diff


            if pd.isna(actual_target_value_1):

                all_events.loc[event, home_post_delta_name_1] = home_attack
                all_events.loc[event, home_post_delta_name_2] = home_defence
                all_events.loc[event, home_post_delta_name_3] = home_home_attack
                all_events.loc[event, home_post_delta_name_4] = home_home_defence
                all_events.loc[event, away_post_delta_name_1] = away_attack
                all_events.loc[event, away_post_delta_name_2] = away_defence
                all_events.loc[event, away_post_delta_name_3] = away_away_attack
                all_events.loc[event, away_post_delta_name_4] = away_away_defence

            else:

                # Comparing against the home_score (for all games and for home_home games)
                if (actual_target_value_1 < (pred_home_score_all - win_margin_buffer)):
                    all_events.loc[event, home_post_delta_name_1] = home_attack - (min(max_points_win, np.log(0.000001+abs(pred_home_score_all - actual_target_value_1))) - win_bonus)/2
                    all_events.loc[event, away_post_delta_name_2] = away_defence - (min(max_points_win, np.log(0.000001+abs(pred_home_score_all - actual_target_value_1))) + win_bonus)/2
                elif (actual_target_value_1 > (pred_home_score_all + win_margin_buffer)):
                    all_events.loc[event, home_post_delta_name_1] = home_attack + (min(max_points_win, np.log(0.000001+abs(pred_home_score_all - actual_target_value_1))) + win_bonus)/2
                    all_events.loc[event, away_post_delta_name_2] = away_defence + (min(max_points_win, np.log(0.000001+abs(pred_home_score_all - actual_target_value_1))) - win_bonus)/2
                else:
                    all_events.loc[event, home_post_delta_name_1] = home_attack
                    all_events.loc[event, away_post_delta_name_2] = away_defence
                    
                    
                if (actual_target_value_1 < (pred_home_score_ha - win_margin_buffer)):
                    all_events.loc[event, home_post_delta_name_3] = home_home_attack - (min(max_points_win, np.log(0.000001+abs(pred_home_score_ha - actual_target_value_1))) - win_bonus)/2
                    all_events.loc[event, away_post_delta_name_4] = away_away_defence - (min(max_points_win, np.log(0.000001+abs(pred_home_score_ha - actual_target_value_1))) + win_bonus)/2
                elif (actual_target_value_1 > (pred_home_score_ha + win_margin_buffer)):
                    all_events.loc[event, home_post_delta_name_3] = home_home_attack + (min(max_points_win, np.log(0.000001+abs(pred_home_score_ha - actual_target_value_1))) + win_bonus)/2
                    all_events.loc[event, away_post_delta_name_4] = away_away_defence + (min(max_points_win, np.log(0.000001+abs(pred_home_score_ha - actual_target_value_1))) - win_bonus)/2
                else:
                    all_events.loc[event, home_post_delta_name_3] = home_home_attack
                    all_events.loc[event, away_post_delta_name_4] = away_away_defence
                    
                # Comparing against the away_score (for all games and for away_away games)
                if (actual_target_value_2 < (pred_away_score_all - win_margin_buffer)):
                    all_events.loc[event, home_post_delta_name_2] = home_defence - (min(max_points_win, np.log(0.000001+abs(pred_away_score_all - actual_target_value_2))) - win_bonus)/2
                    all_events.loc[event, away_post_delta_name_1] = away_attack - (min(max_points_win, np.log(0.000001+abs(pred_away_score_all - actual_target_value_2))) + win_bonus)/2
                elif (actual_target_value_2 > (pred_away_score_all + win_margin_buffer)):
                    all_events.loc[event, home_post_delta_name_2] = home_defence + (min(max_points_win, np.log(0.000001+abs(pred_away_score_all - actual_target_value_2))) + win_bonus)/2
                    all_events.loc[event, away_post_delta_name_1] = away_attack + (min(max_points_win, np.log(0.000001+abs(pred_away_score_all - actual_target_value_2))) - win_bonus)/2
                else:
                    all_events.loc[event, home_post_delta_name_2] = home_defence
                    all_events.loc[event, away_post_delta_name_1] = away_attack
                    
                    
                if (actual_target_value_2 < (pred_away_score_ha - win_margin_buffer)):
                    all_events.loc[event, home_post_delta_name_4] = home_home_defence - (min(max_points_win, np.log(0.000001+abs(pred_away_score_ha - actual_target_value_2))) - win_bonus)/2
                    all_events.loc[event, away_post_delta_name_3] = away_away_attack - (min(max_points_win, np.log(0.000001+abs(pred_away_score_ha - actual_target_value_2))) + win_bonus)/2
                elif (actual_target_value_2 > (pred_away_score_ha + win_margin_buffer)):
                    all_events.loc[event, home_post_delta_name_4] = home_home_defence + (min(max_points_win, np.log(0.000001+abs(pred_away_score_ha - actual_target_value_2))) + win_bonus)/2
                    all_events.loc[event, away_post_delta_name_3] = away_away_attack + (min(max_points_win, np.log(0.000001+abs(pred_away_score_ha - actual_target_value_2))) - win_bonus)/2
                else:
                    all_events.loc[event, home_post_delta_name_4] = home_home_defence
                    all_events.loc[event, away_post_delta_name_3] = away_away_attack
                    
                    
                          
    function_end_time = datetime.datetime.now()
    print('--Complete-' + str(function_end_time - function_start_time))
    print('')
    

    
    return all_events


# Add Trends

In [38]:
def add_event_deltas(all_events, event_deltas, float_columns):
    
    function_start_time = datetime.datetime.now()
    print('-add_event_deltas')
    
    for col_name in float_columns:
        if col_name in all_events.columns:
            all_events.drop(col_name, axis = 1, inplace = True)
            
    temp_columns = float_columns.copy()
    temp_columns.append('event_id')
    event_deltas = event_deltas[temp_columns]

    all_events = all_events.merge(event_deltas, how = 'left', left_on = 'event_id', right_on = 'event_id') 

    for col in float_columns:
        all_events[col] = all_events[col].apply(lambda x: float(x))

    #all_events['home_pre_delta'] = all_events['home_pre_delta'].apply(lambda x: float(x))
    #all_events['home_post_delta'] = all_events['home_post_delta'].apply(lambda x: float(x))
    #all_events['away_pre_delta'] = all_events['away_pre_delta'].apply(lambda x: float(x))
    #all_events['away_post_delta'] = all_events['away_post_delta'].apply(lambda x: float(x))
    #all_events['home_team_buffer'] = all_events['home_team_buffer'].apply(lambda x: float(x) if pd.notna(x) else None)
    #all_events['pre_delta_diff'] = all_events['pre_delta_diff'].apply(lambda x: float(x) if pd.notna(x) else None)

    function_end_time = datetime.datetime.now()
    print('--Complete-' + str(function_end_time - function_start_time))
    print('')
    
    return all_events

In [39]:
def add_trends(all_events, trend_name, trend_type, num_games, start_range):


    function_start_time = datetime.datetime.now()
    print('-add_trends')
    print('--', trend_name, num_games)
    
    home_trend_name = 'home_' + trend_name
    away_trend_name = 'away_' + trend_name

    trend_name_final = trend_name + '_trend' + '_' + str(num_games)
    home_trend_name_final = 'home_' + trend_name_final
    away_trend_name_final = 'away_' + trend_name_final

    if home_trend_name_final in all_events.columns:
        all_events.drop(home_trend_name_final, axis = 1, inplace = True)
    if away_trend_name_final in all_events.columns:
        all_events.drop(away_trend_name_final, axis = 1, inplace = True)

    home_fixtures = all_events[['event_id', 'home_team_internal_id', 'home_team_total_fixture_number', 'home_' + trend_type]].rename(columns = {'home_team_internal_id':'team_id', 'home_team_total_fixture_number':'team_total_fixture_number', 'home_' + trend_type:trend_name})
    away_fixtures = all_events[['event_id', 'away_team_internal_id', 'away_team_total_fixture_number', 'away_' + trend_type]].rename(columns = {'away_team_internal_id':'team_id', 'away_team_total_fixture_number':'team_total_fixture_number', 'away_' + trend_type:trend_name})

    all_team_fixtures = home_fixtures.append(away_fixtures, ignore_index = True)
    all_team_fixtures = all_team_fixtures.drop_duplicates().reset_index()
    all_team_fixtures.sort_values('team_total_fixture_number', inplace = True)

    all_team_fixtures.loc[ : , trend_name_final] = None
    
    
    for team in all_team_fixtures['team_id'].drop_duplicates():

        team_fixtures = all_team_fixtures[ all_team_fixtures['team_id'] == team]
        
        team_fixtures_to_calculate = team_fixtures[start_range :]

        for row in team_fixtures_to_calculate.index:

            current_fixture_number = team_fixtures.loc[row, 'team_total_fixture_number']
            team_trend = team_fixtures[ (team_fixtures['team_total_fixture_number'] < current_fixture_number) & (team_fixtures['team_total_fixture_number'] >= (current_fixture_number - num_games)) ][trend_name].mean()        
            team_fixtures.loc[row, trend_name_final] = team_trend


        all_team_fixtures[trend_name_final].update(team_fixtures[trend_name_final])


    
    all_events = all_events.merge(all_team_fixtures[['event_id', 'team_id', trend_name_final]].rename(columns = {'team_id':'home_team_internal_id', trend_name_final: home_trend_name_final}), how = 'left', left_on = ['event_id', 'home_team_internal_id'], right_on = ['event_id', 'home_team_internal_id'])
    all_events = all_events.merge(all_team_fixtures[['event_id', 'team_id', trend_name_final]].rename(columns = {'team_id':'away_team_internal_id', trend_name_final: away_trend_name_final}), how = 'left', left_on = ['event_id', 'away_team_internal_id'], right_on = ['event_id', 'away_team_internal_id'])
    
    function_end_time = datetime.datetime.now()
    print('--Complete-' + str(function_end_time - function_start_time))
    print('')
    

    return all_events

In [40]:
def update_event_deltas_trends(all_events, delta_change_name, error_name):

    function_start_time = datetime.datetime.now()
    print('-update_event_deltas_trends')
    
    sql_statement_full = ''
    loop_num = 0

    if len(all_events)>0:

        for event in all_events.index:

            #print(loop_num, len(all_events))

            event_id = all_events.loc[event, 'event_id']

            home_delta_change_trend_5_name = 'home_' + delta_change_name + '_trend_5'
            home_delta_change_trend_10_name = 'home_' + delta_change_name + '_trend_10'
            home_delta_change_trend_20_name = 'home_' + delta_change_name + '_trend_20'
            
            away_delta_change_trend_5_name = 'away_' + delta_change_name + '_trend_5'
            away_delta_change_trend_10_name = 'away_' + delta_change_name + '_trend_10'
            away_delta_change_trend_20_name = 'away_' + delta_change_name + '_trend_20'
            
            home_error_trend_5_name = 'home_' + error_name + '_trend_5'
            home_error_trend_10_name = 'home_' + error_name + '_trend_10'
            home_error_trend_20_name = 'home_' + error_name + '_trend_20'
            
            away_error_trend_5_name = 'away_' + error_name + '_trend_5'
            away_error_trend_10_name = 'away_' + error_name + '_trend_10'
            away_error_trend_20_name = 'away_' + error_name + '_trend_20'
            
            
            home_delta_change_trend_5 = all_events.loc[event, home_delta_change_trend_5_name]
            home_delta_change_trend_10 = all_events.loc[event, home_delta_change_trend_10_name]
            home_delta_change_trend_20 = all_events.loc[event, home_delta_change_trend_20_name]

            away_delta_change_trend_5 = all_events.loc[event, away_delta_change_trend_5_name]
            away_delta_change_trend_10 = all_events.loc[event, away_delta_change_trend_10_name]
            away_delta_change_trend_20 = all_events.loc[event, away_delta_change_trend_20_name]

            home_error_trend_5 = all_events.loc[event, home_error_trend_5_name]
            home_error_trend_10 = all_events.loc[event, home_error_trend_10_name]
            home_error_trend_20 = all_events.loc[event, home_error_trend_20_name]

            away_error_trend_5 = all_events.loc[event, away_error_trend_5_name]
            away_error_trend_10 = all_events.loc[event, away_error_trend_10_name]
            away_error_trend_20 = all_events.loc[event, away_error_trend_20_name]

            if pd.isna(home_delta_change_trend_5):
                home_delta_change_trend_5 = 'NULL'
            if pd.isna(home_delta_change_trend_10):
                home_delta_change_trend_10 = 'NULL'
            if pd.isna(home_delta_change_trend_20):
                home_delta_change_trend_20 = 'NULL'
            if pd.isna(away_delta_change_trend_5):
                away_delta_change_trend_5 = 'NULL'
            if pd.isna(away_delta_change_trend_10):
                away_delta_change_trend_10 = 'NULL'
            if pd.isna(away_delta_change_trend_20):
                away_delta_change_trend_20 = 'NULL'

            if pd.isna(home_error_trend_5):
                home_error_trend_5 = 'NULL'
            if pd.isna(home_error_trend_10):
                home_error_trend_10 = 'NULL'
            if pd.isna(home_error_trend_20):
                home_error_trend_20 = 'NULL'
            if pd.isna(away_error_trend_5):
                away_error_trend_5 = 'NULL'
            if pd.isna(away_error_trend_10):
                away_error_trend_10 = 'NULL'
            if pd.isna(away_error_trend_20):
                away_error_trend_20 = 'NULL'

            sql_statement = 'update event_deltas set ' + str(home_delta_change_trend_5_name) + ' = ' + str(home_delta_change_trend_5) + ', ' + str(home_delta_change_trend_10_name) + ' = ' + str(home_delta_change_trend_10) + ', ' + str(home_delta_change_trend_20_name) + ' = ' + str(home_delta_change_trend_20) + ', ' + str(away_delta_change_trend_5_name) + ' = ' + str(away_delta_change_trend_5) + ', ' + str(away_delta_change_trend_10_name) + ' = ' + str(away_delta_change_trend_10) + ', ' + str(away_delta_change_trend_20_name) + ' = ' + str(away_delta_change_trend_20) + ', ' + str(home_error_trend_5_name) + ' = ' + str(home_error_trend_5) + ', ' + str(home_error_trend_10_name) + ' = ' + str(home_error_trend_10) + ', ' + str(home_error_trend_20_name) + ' = ' + str(home_error_trend_20) +  ', ' + str(away_error_trend_5_name) + ' = ' +str(away_error_trend_5) + ', ' + str(away_error_trend_10_name) + ' = ' + str(away_error_trend_10) + ', ' + str(away_error_trend_20_name) + ' = ' + str(away_error_trend_20) + " where event_id = '" + str(event_id) + "';"

            sql_statement_full = sql_statement_full + sql_statement
            loop_num += 1
            
    if len(sql_statement_full) > 0:
        postgres_Retreive_Insert(sql_statement_full, POSTGRESQL_PARAMS, False)
        
    function_end_time = datetime.datetime.now()
    print('--Complete-' + str(function_end_time - function_start_time))
    print('')

In [41]:
def check_for_duplicate_events(all_events):

    home_games = all_events[['event_id', 'start_time', 'home_team_internal_id']].rename( columns = {'home_team_internal_id':'team_internal_id'})
    away_games = all_events[['event_id', 'start_time', 'away_team_internal_id']].rename( columns = {'away_team_internal_id':'team_internal_id'})
    both_teams = pd.concat([home_games, away_games], ignore_index = True)

    both_teams = both_teams.groupby(['team_internal_id', 'start_time']).count().reset_index()
    events_to_check = both_teams[ both_teams['event_id'] > 1]


    duplicate_events = []

    for etc in events_to_check.index:

        start_time = events_to_check.loc[etc, 'start_time']
        team_id = events_to_check.loc[etc, 'team_internal_id']

        event_ids = all_events[ ( (all_events['home_team_internal_id'] == team_id)  | (all_events['away_team_internal_id'] == team_id) ) & (all_events['start_time'] == start_time)  ]['event_id']

        for e in event_ids:

            duplicate_events.append(e)

    duplicate_events = list(set([item for item in duplicate_events]))
    
    return duplicate_events

In [42]:
def check_all_teams_exist(all_events, all_teams):
    
    proceed = True

    home_teams = list(all_events['home_team_internal_id'])
    away_teams = list(all_events['away_team_internal_id'])

    both_teams = home_teams + away_teams
    both_teams = list(set(both_teams))

    db_teams = list(all_teams['id'])

    not_in_db = [team for team in both_teams if team not in db_teams]
    
    if len(not_in_db) > 0:
        print('There are no enties for the team: ' + str(not_in_db))
        proceed = False

    return proceed


In [43]:
def check_competitions_exist(all_events, all_teams):
    
    proceed = True

    comps = list(all_events['competition_internal_id'])
    comps = list(set(comps))

    db_comps = list(all_competitions['id'])

    not_in_db = [team for team in comps if team not in db_comps]
    
    if len(not_in_db) > 0:
        print('There are no enties for the competition: ' + str(not_in_db))
        proceed = False

    return proceed


In [44]:
international_rankings = {
    "cae79170-e3ef-4329-9903-c10186a49cec" : 270.531289753092,
"bf41a52c-94f3-426a-b707-f2f11cb01de8" : 262.6964625275002,
"216fc1fc-f969-462b-8875-b1910c85e482" : 262.500333594012,
"ee8f8374-1c04-4276-ab75-4bafc7441912" : 254.8639020080243,
"7d3aef5a-7793-46ba-89b7-cbddf6fe12eb" : 253.4310945724153,
"294eaf52-dfdd-4918-ad0a-3274f5dbd2cf" : 247.30755460216426,
"f9413c17-aeb0-4739-b9ba-b9bf52991c68" : 247.2347794000136,
"33c5dddd-470e-4bbf-888a-41de5eba7f59" : 245.00975061064813,
"1f4f7424-1bc0-4aed-adc2-6ccfab40fbae" : 243.98002632761163,
"cfe6acd2-6656-4c02-bfae-28bd6f368cfc" : 242.10353682767337,
"9a001913-8ea4-454a-8224-7514825912ca" : 238.51489518896864,
"8b06e788-3af4-4392-bb82-062f18cea1cb" : 228.77813569242878,
"0f3fc29e-5c6a-4f8f-ad76-38d420f69810" : 227.73865458436666,
"b2a25dd9-e46d-4673-bda6-76638b4dd5e0" : 225.98819795537295,
"5776fd4c-0e23-4c79-82e7-c7f4cb9ec9ea" : 222.262648344341,
"74412498-e87e-42d0-9270-7d5fec902b46" : 220.30282319455296,
"361d5ba8-4d57-4093-940c-01b36fc274ff" : 215.40287788427787,
"95c15dd0-52af-47bf-8322-753183c0477d" : 208.87866585716216,
"9fc3a386-e2c7-4556-a78a-ffe91319b405" : 205.2509102091039,
"16750364-f193-4c66-85e9-b04a2ae03fa2" : 203.20504662846588,
"e72cf7c2-876b-41c4-bdd1-d1e4a0ebbd31" : 202.67940172277048,
"1a492fae-2e3a-4203-90c4-0f6eac201ff3" : 199.47695265923403,
"a3a64375-a2ae-4342-bbd2-35665c2f31aa" : 188.11899841831925,
"f6c7df6a-bb15-4053-a4e5-b988f2eed556" : 180.77568958513632,
"22241244-4a29-42fc-80b8-a92b49fbba82" : 177.72447906761704,
"84527f85-edeb-4cf0-a3c0-89ff1e5df4dd" : 177.04385203858126,
"a57b246b-26ce-4fd4-820b-4c91caa3e7e5" : 167.38336177875624,
"46f2867c-4b46-417a-a118-8c195f90ce02" : 167.23294005805872,
"6caa9439-17ef-4d7d-9c60-de9d59c2f7e7" : 163.8260216716606,
"83a024eb-6f8b-4dda-ad67-c49e4898fdb5" : 162.27822488075222,
"3fe21187-7c48-4ccf-bf9d-34cd7e21ea7d" : 161.7496001003144,
"ff12aad4-85c2-45a8-a366-f45eb8506809" : 161.52414271456007,
"6bae6049-8544-4f4e-b72b-05521b2c37b6" : 161.09709915930733,
"cfdd7fce-43a1-4af6-a4ca-0c2e23d075f9" : 160.71915319427933,
"e7377542-f91c-45be-a38e-c8dfe43678a4" : 159.06091054561256,
"db719e32-2b29-40d3-8b62-b05da1bdd93b" : 158.73960573515143,
"b9148cde-89c8-4e81-8f65-09fe38b15a0f" : 157.786054875649,
"fcf8cd85-21ea-45e9-b659-cfc44f41cea9" : 156.42468129767366,
"840e016f-2571-479e-8ce1-4996d6d41987" : 155.42063357150843,
"a6918a34-b0e2-43c9-990f-cb1d814c600b" : 155.22144950068278,
"4016a1db-70ea-4a31-9d59-434dc82893cc" : 154.31354953756653,
"a08e1e60-f1b9-408f-942a-1cad150609e9" : 152.0832477055657,
"62e8345c-c91c-4bbc-8d26-f151e1424171" : 151.4871549278716,
"cecc2bb5-19a0-4c6e-80c7-ba79ef3acf35" : 150.68656880447938,
"4f9e48fc-a34e-474b-a26e-ad1b57cd8880" : 150.40153530533885,
"9aaa03bb-560d-4911-9167-171b29098c03" : 150.02845828606462,
"8af0eb19-e0f8-491c-9f70-ce9503b561a7" : 146.99729102200715,
"c101883b-5108-4a55-985e-4b84d2a3c02d" : 146.8550253324795,
"bb19921f-d2d9-486a-85ac-50453154687f" : 142.99815087101373,
"6a753deb-93d1-4451-9bba-21955169e224" : 141.50799495220156,
"d9a0676e-6902-49fd-8108-fcc78afed1ad" : 141.2198395114673,
"2f4b0698-1d52-4b3f-a051-3704eb8f982f" : 139.90878738814266,
"74fb1ee4-f7ee-429c-b9ad-6dd7531b6097" : 139.44929507800913,
"5b10114c-a0f7-425e-abd3-cc821c166848" : 139.3977766764734,
"c8511e0a-875a-4544-925e-e414b4a2cbd6" : 138.85197749998602,
"0828700f-e81f-4b79-b9e4-146e47bd02a6" : 138.4791476037188,
"3b3ba5f2-bc7c-4d2c-bad9-dbc8c4bda38d" : 136.21854543780327,
"92261fdf-2983-42f0-b086-e1725377613b" : 135.71144433852797,
"6f1cdbae-e0d3-4072-a218-d9f20efec921" : 135.58771136357825,
"5ae92037-d1aa-4a3e-ab6c-54b7f8a5628e" : 134.8461789848428,
"0b88e7c1-517e-4610-ad50-c3771a5e4531" : 134.04125630299973,
"cd0d53f5-4a11-4ded-8594-641ef025842d" : 131.94380898614452,
"e82cba06-a49b-413d-ad62-8e91a5a4ac0a" : 131.63730193052288,
"62368651-78df-4e34-8041-dbb264cdeb7c" : 131.0607968684225,
"bd9c9946-aabc-499c-8309-78534629c1bd" : 129.4097348418902,
"8ba5c2aa-6b0f-481c-989d-8fc3ad0370f3" : 128.99149081436522,
"de14dbca-cc9c-4cb7-bebf-efae01dcb822" : 127.48771076846144,
"ce102af7-f595-4990-84db-8f37988bd26e" : 126.67794757475305,
"3c424a5f-586a-4625-aaa8-22d5ff95fc39" : 125.99916191491872,
"dce422e9-d1f2-4f70-8db4-f1ee69e9a5e0" : 125.82023924397132,
"27c96b31-4be6-454b-88bd-5e970473e645" : 125.80224756552967,
"265db8e9-a59d-4bb7-8858-62fa7170833d" : 124.62833767755313,
"312497cd-b9d1-4cd7-adf4-b081c29cb6d7" : 124.44348147086419,
"1ebda97c-0182-453f-8da5-7022b630dea7" : 123.73397659875182,
"600c8b51-10fb-44aa-ae77-834e3881a3ae" : 123.1830496555568,
"e6d01bb0-a67a-4195-9156-999343d08697" : 122.73580453894543,
"f4648662-ae61-45db-9fe8-5b818a11d599" : 121.91678889930125,
"190f9e23-c24a-4f3f-88e2-0f1dde8543c7" : 121.73935430918232,
"400f8ef2-9ec1-4a6a-92e5-14b6ddba35e7" : 121.69707797090167,
"b26274d1-6d81-46c4-a8ca-0038056ad58f" : 121.19286245276098,
"02ebc964-b4a1-497e-8821-283610635bd5" : 121.05516511164572,
"17a8fec6-aa93-4807-8ec6-5b6c1ec25b0d" : 120.8815730359892,
"2e4414c4-43b1-4a1d-b6ca-cece82f0390a" : 119.86430015307373,
"868a967a-3ac8-41f1-9ec0-4af59209a3a9" : 119.52448149945093,
"96e85b30-b94c-4fd9-bf82-8eabda0fb8db" : 118.08251319932555,
"a6905fe3-e6c9-4458-8a45-03a26a73b0f4" : 117.3702290679179,
"f1bbf189-7206-4192-8b9c-8edc271a8d20" : 117.36175699612654,
"7a25b6c2-b2c2-4677-ac60-1f8131d61a07" : 117.23369137911129,
"bd12b013-350a-4d33-9c84-e2ecda14f418" : 116.97099254581401,
"50914914-43ba-42f8-93d0-e52303562ba6" : 116.8680019111917,
"8a8a1c21-cdb1-4824-886d-b90ef69085e2" : 116.07467466068154,
"6db54310-4ea9-42fa-9b4f-601f06aabe83" : 112.76621252999175,
"f5c70b87-ba49-4824-b2ca-d10dfaf1edb1" : 112.43027184339287,
"ecf78241-3356-40fd-9d48-52b4056d8473" : 109.63408332718878,
"b6f38f03-3e4a-41d0-86df-be494b3967af" : 108.24079219216625,
"2d78c309-36aa-4df1-8763-3d4d56bfb55a" : 107.78785206352754,
"58966b9c-aebc-4042-a6be-129c10172615" : 106.22823766008682,
"1c415060-a1d3-401e-bdf5-086328982ee6" : 105.34209703785767,
"1f829cbd-4c9e-4ef8-a72a-f21c9625e0b5" : 105.10151298717244,
"8a67eb9b-970d-426b-83cc-67f59221eef7" : 102.14109769557506,
"a9f1012c-7d0d-4c63-a274-a31fd1abac56" : 102.03758738201476,
"68e9ddce-2dcd-48e3-8eeb-805ba86c73aa" : 102.00177536673573,
"a50cc0d4-8c28-4688-8aa1-4a3ab52d97af" : 101.26345951499675,
"47c150ae-b6f9-4864-b00b-3a72228aff16" : 100.47818770037529,
"7f714b76-9695-4906-8858-de1131923630" : 96.37686986387892,
"2dd2dbbc-c332-4652-9f55-fd4d98e798ed" : 95.64384512398674,
"30255d9a-3f51-4764-a892-c22bdb4b70da" : 93.67847306927693,
"3f67e978-77ee-45aa-ad67-1ee66c241221" : 93.54191981475445,
"35b61808-0694-4321-83ee-5b66708d0e5a" : 93.46380306906244,
"59224544-6010-4265-968a-9b8a358c59eb" : 92.60864583068141,
"92402401-c5f4-475e-a581-8ff59193c357" : 92.37389532935714,
"7f12ac33-db8c-4a9d-8807-1a96af3f9443" : 91.23892766861444,
"43fb73bd-5b2f-4bea-9f3f-b70783497484" : 89.0382079620317,
"5d01fec8-034e-4639-8674-2cebe4258589" : 87.05936985980472,
"7220605e-2f27-40e1-a52d-6493ea7304b8" : 84.18651014454024,
"73352379-f726-4987-a355-463a0ec75dc1" : 82.10887201753671,
"586460e0-afbb-45d0-84d7-26d06b106520" : 80.97169689996696,
"bb577965-c80c-46ca-9ed8-bffad88aa293" : 78.05674825859674,
"cce33035-2831-4fe7-8198-0961c980028d" : 77.31068792701126,
"5f2d4c7c-119d-4b29-8f40-be1eecd9320d" : 74.7427715529407,
"be3cfe43-a4af-4bda-815a-a97cab6a4bfd" : 70.19954414162407,
"b202d6dd-f7e2-4cf7-80eb-14071c4c1af1" : 61.09982388767824,
"0b92ee08-36f5-4e6a-a3e0-d437b836d3ac" : 60.1324816547296,
"46c2e844-b771-48f4-b405-d6886a93486b" : 53.52622426075546,
"1e1ddf89-0c52-448b-9bd1-3208730ae559" : 47.55712894607879,
"71388112-0253-4552-a48f-5080b8dab061" : 40.772886599343444,
"716039d6-fcfc-4949-bd31-46d789c84ed3" : 38.61582079150535,
"f4a63316-66f3-49d6-ad31-754144e1058b" : 25.416164676227453}

In [45]:
international_rankings_women = {"19fb07f1-955c-43f7-9140-047ca0a27f0f" : 242.39338072280987,
"9f99e503-a583-4709-acc1-eff04e148ce5" : 237.27127523060165,
"df5575dc-63f8-4549-980b-8a6e96104030" : 229.12303160300922,
"3d83768f-0207-4387-93eb-d7727bcda166" : 216.36359111435615,
"ae932176-8290-4fc8-a92a-75f8de57e87b" : 215.61376478796262,
"95c0a975-15e0-4da8-b02d-04ff39eb021e" : 213.9352094738502,
"8b48650e-c406-48f4-bdfa-ea7fb05b88e0" : 212.93391294863684,
"df013f15-33c3-426e-a8a9-9ffe2b16b021" : 211.39583456032352,
"d046337f-0a39-4096-9d1e-06a94927f1dd" : 195.21608806674567,
"030888f9-d261-490a-b52a-c2b39b0fc3ae" : 193.83897040260544,
"acf18456-0cef-4f19-b2ef-393c61e2d187" : 187.98838671817265,
"6e3994e8-cda9-4946-a613-c15e987546fb" : 186.76146899893314,
"7bf4d486-8075-42a6-9543-6fc46cb51354" : 185.13838376536359,
"590711c9-fd7d-4354-aaf6-6d03595cf7f2" : 184.3565408879047,
"dcc03189-a3ba-43d8-8cfe-d28af5c937fd" : 184.35165548270942,
"7a37de83-e765-499e-baec-c9841c7a60a1" : 178.73656598617242,
"e53871dc-305b-4fc8-a3f2-6d1ab35df098" : 175.4780230741793,
"68d37772-e670-4a85-bf2d-69befacd2776" : 172.8556574380159,
"2de25f00-9bc7-4b60-b0c9-bbfcbdd2a250" : 172.22850512818252,
"e08811e5-bdb1-45f5-a5ef-3fb74403b62e" : 171.53225956477257,
"02e9dc14-cc03-4f0c-a124-70f58037d30a" : 169.99943889440192,
"718d3f41-906a-4bff-9628-f00d304590e5" : 166.8862484270072,
"aef43d70-ed03-4dc3-9e72-2121728ba75c" : 166.82657077792612,
"3039976c-05a8-4953-a9ec-38e471d47df8" : 149.97735692741625,
"b8d97768-4519-4be8-b6dc-dbb1695f10f3" : 149.84319845779012,
"09b61b86-abdc-4da0-a500-378da8525a87" : 149.7815910606296,
"8c36af7f-f512-4d1b-b1d5-cac31db9f769" : 147.8757185263841,
"3ae0c2af-1b7a-42ac-91b1-c24dafe46816" : 145.47524426958591,
"b29dd972-aa17-4c18-89de-9dcedb144d25" : 137.29395509759658,
"8fc4c6d9-7f2d-4711-9abd-300ac453d134" : 136.59819863146888,
"6bf11bed-5fc1-4e1d-b745-acc2505f6699" : 135.0017667433363,
"d4464a9e-56b6-45c2-9f7b-322cd08e44fa" : 132.46417283393012,
"67c23e5a-b9eb-4d71-9496-57a28dce9208" : 130.32623750373827,
"4c3819ab-78d5-4b8b-9dbb-4d2e930421ea" : 128.92260315693065,
"63b96c9f-a0b3-490d-a2da-34acf973d1c6" : 124.68642968432688,
"2c32d712-afe4-4934-9a8e-af2b5e50017d" : 124.24173492862501,
"c911b5fd-207d-4004-8ada-147ff2f5771a" : 122.56700437034058,
"f6101db3-5db3-4b85-974f-c02aeeb1c72a" : 122.52429103585038,
"33b7cfd8-93ce-4bde-87c2-e9c6316ffc7d" : 111.73440410670977,
"e303a7b1-bd20-4eee-86c2-2ee46ea634e2" : 103.71724817189305,
"0d8b972f-a34f-4ab4-8290-820f4ece141b" : 99.17196812703068,
"7fb72287-7b41-4355-86e3-ea8e546f415d" : 91.76179541794515,
"dc3744d9-0c01-4ab6-817b-ccdc59292f53" : 91.2123729096142,
"5db95245-16a1-4525-9b4a-200cbb98b472" : 85.83448842762587,
"93458328-6923-4135-b712-003b97052461" : 82.88457315540145,
"4a594de1-30b9-4c36-a401-92c078938608" : 74.56303044523703,
"9be399e8-3147-40f7-9faa-3fdd0af8a3b1" : 71.6501471635801,
"07631c46-5ac9-4c64-a35d-f774763a0b85" : 65.17696743364813}

In [46]:
new_comp_levels = {'4f0ae53d-8cd4-4d93-9fbe-0ce2df4bb56b':2,
                  'f58adc7b-8883-4ad7-a3ea-7e4b68cd3bc3':3,
                  '1885fb8a-c0f2-4b19-aaaa-111e4a40567e':3.4,
                  'd4c72439-882a-4ac5-9f95-d306ed321ccc':2,
                  '5a15b335-c65a-4878-96b6-41f90e785dad':2.5,
                  'f081b233-551c-4948-8840-29251e1952d3':2,
                  '4d148966-2365-4a01-9448-c1190b7d08c8':2,
                  '35f5e778-1144-44fa-a70f-03743c031bee':3,
                  '7df6de6a-e1a0-4b11-bffd-262c84b74789':4,
                  '0524a4e0-efeb-4780-93b9-be98b2f3fbe2':3,
                  '95ab3fbf-4670-4405-8025-6c65efce65d7':3,
                  '69397a9a-19ef-4d82-bc86-370baf9c53c3':3,
                  '0494931c-b163-11ea-8832-001a7dda7115':4,
                  '283d2a95-fa56-4316-aa1e-15a087349166':3.5,
                  'cd7e1345-a727-4fed-a3a9-617203745ba9':2,
                  '518f3a42-b7d3-4640-a5c8-e0225fd88093':3,
                  '7527e849-f9fd-4b2c-8919-c3d76ee0087e':3,
                  '6fcfb116-d342-4080-8bd8-fd080b462ef4':3,
                  '61df7e56-1300-464e-84f1-54663d44f004':4,
                  'ae3d6158-8a63-4065-8b21-0a1fbc283e18':3,
                  'f4b46636-b162-11ea-af86-001a7dda7115':3.5,
                  '7ad4f3cc-83e4-45f4-8391-94360198d64f':2,
                  '5276f62a-428a-4f02-9f29-7913ef00a5d1':2.6,
                   '1fe03390-c49e-48da-b101-cbece0039968':3,
                   'e9913e66-2b89-489b-9978-f72e53e167b4':3}

In [47]:
other_initial_rankings = {'4ec911de-2240-4bf1-a239-b75bbf036977':270.531289753092,
                          '3f081086-0a7a-4fb1-afd8-4ba48f720565':190,
                        'ee3a03c3-2153-43f5-bf34-dd6e518328bf': 148.63, 
                        'b27166c4-17ee-4d9b-9160-6c7ed5875cb2':162.85,
                         'a2de5edd-f074-4c95-9057-a1a2f058a230':216,
                         '22fe9a0d-42f1-4987-af55-e15556232d0e':170,
                         '2ee0306c-103d-4771-8a18-bb3bc234d5a5':240,
                         '9ab9d258-1e66-45a3-874c-a262db8a1b76':200,
                         '904538c8-4f8b-4a99-ae4d-ad31a12cfda5':180,
                         '6a018e47-b86f-4b69-9b11-5cd61b3ff007':162,
                         '5c856e7b-2e24-4c7a-b37f-2a7e3098b1ab':129,
                         '0953dff9-8850-4041-a921-5f0590374ee8':180,
                         'ed6caa7b-f83b-469b-8fab-0b752e72421b':50,
                         'e6a692fe-80d9-43b2-8bd7-c67f2d075b88':64,
                         '5bbbced0-ae87-11ea-a648-001a7dda7115':64,
                         '46c2e844-b771-48f4-b405-d6886a93486b':64,
                         '3af67b25-52cd-409d-9fb8-16979081d9d8':64,
                         '633e6046-5857-4e41-9243-8bb1c286f817':200,
                         'da82a9ec-4def-4629-895d-08d5e4b2bd2a':120,
                         '3af67b25-52cd-409d-9fb8-16979081d9d8':142,
                         '7220605e-2f27-40e1-a52d-6493ea7304b8':140,
                         '7f12ac33-db8c-4a9d-8807-1a96af3f9443':140,
                         '6caa9439-17ef-4d7d-9c60-de9d59c2f7e7':194,
                         'a4b072df-153b-49f9-8bbd-6b03c7592edc':65,
                         '35f352a1-018b-4181-92a8-776c1007ff94':165}

In [48]:
list_order_int_men = ['pre_game_rank_senior_team_ranking', 
                      'opp_pre_elo',
                      'pre_game_rank_historic_competition_median', 
                      'pre_game_rank_historic_home_competition_group_median', 
                      'pre_game_rank_new_home_competition_group', 
                      'pre_game_rank_int_comp_level_setting',
                      'all_events_median']

list_order_int_women_age = ['pre_game_rank_senior_team_ranking', 
                      'opp_pre_elo',
                      'pre_game_rank_historic_competition_median', 
                      'pre_game_rank_historic_home_competition_group_median', 
                      'pre_game_rank_new_home_competition_group', 
                      'pre_game_rank_int_comp_level_setting',
                      'all_events_median']

list_order_club = ['pre_game_rank_senior_team_ranking',
                   'pre_game_rank_historic_competition_median',
                   'pre_game_rank_historic_home_competition_group_median',
                   'opp_pre_elo',
                   'pre_game_rank_new_home_competition_group',
                   'pre_game_rank_int_comp_level_setting',
                   'all_events_median']

In [49]:
# latest one without pre_delta_diff_adjusted
points_transfer_dict = {'d4c72439-882a-4ac5-9f95-d306ed321ccc':0.5, '4f0ae53d-8cd4-4d93-9fbe-0ce2df4bb56b':0.5, 'f330a8ae-38bf-4ec2-9735-6269f6b14e77':0.6, '34832c47-d30e-40ca-b5c6-4065b8b01715':1.2, '92c3cd86-8fb6-4b0b-8044-da30f89b4c8d':0.6, '7e901bf2-005f-461b-8105-acabc014ff2c':0.6, '11552296-5f73-427f-9308-6e0013a3ff8c':1.3, 'cead6c33-bc0d-4358-86cc-71db61af7d9c':1.3, '7ad4f3cc-83e4-45f4-8391-94360198d64f':0.5, 'ddefeea0-f8bc-4295-93ee-e1113b694bee':1.3, 'f081b233-551c-4948-8840-29251e1952d3':0.4, 'ab1959c0-41be-4e55-8dd7-87f40892520a':1.3, '0524a4e0-efeb-4780-93b9-be98b2f3fbe2':1.1, 'f58adc7b-8883-4ad7-a3ea-7e4b68cd3bc3':0.6, '396d8107-2dac-436e-a9f8-5883991655c0':1.3, 'd4f47c1f-2257-4dbc-8ff6-68666628f456':0.7, '35f5e778-1144-44fa-a70f-03743c031bee':0.7, 'e9913e66-2b89-489b-9978-f72e53e167b4':0.4, '5a15b335-c65a-4878-96b6-41f90e785dad':0.5, '6fcfb116-d342-4080-8bd8-fd080b462ef4':0.9, 'd1ab7c83-e333-423f-8470-237a8467e01d':0.8, '7df6de6a-e1a0-4b11-bffd-262c84b74789':0.9, '1885fb8a-c0f2-4b19-aaaa-111e4a40567e':0.8, '1fe03390-c49e-48da-b101-cbece0039968':0.6, 'c5ec9df5-f62f-48b9-8298-35f95fe64538':0.6, 'e7d6e23d-bd03-4592-ab9e-4ae0695acd79':1.3, '2ac22fec-3cbd-4e5e-8ca7-520c785c3a36':1.3, '183ec923-6902-4c1c-9035-506588249902':1, 'c2f15e2a-41fa-40b4-8779-8156a2cc74e2':0.8, 'dce1cf60-fdce-45f9-83fa-b9708818a623':1.1, '5dab0ba3-08df-49c0-a8b3-34219efb3a6d':0.9, '4e28d4f0-5f4d-4783-b7fd-c47e07374471':1.3, '9597f979-d95b-48ca-8fd3-401357e98e1c': 0.6, 'e5132fce-9605-4ab5-9083-c21a10e8afff': 0.9, '7cbf859b-970e-4a6e-9960-78d1995aced9':0.8}

In [50]:
# First ttransfer but trained on already adjussted pre_delta_diff_aadjusted
points_transfer_dict = {'4f0ae53d-8cd4-4d93-9fbe-0ce2df4bb56b':0.5, 'd4c72439-882a-4ac5-9f95-d306ed321ccc':0.4, 'f330a8ae-38bf-4ec2-9735-6269f6b14e77':0.9, '34832c47-d30e-40ca-b5c6-4065b8b01715':1.2, '92c3cd86-8fb6-4b0b-8044-da30f89b4c8d':0.7, '7e901bf2-005f-461b-8105-acabc014ff2c':0.5, '11552296-5f73-427f-9308-6e0013a3ff8c':1.2, 'cead6c33-bc0d-4358-86cc-71db61af7d9c':1.2, '7ad4f3cc-83e4-45f4-8391-94360198d64f':0.8, 'ddefeea0-f8bc-4295-93ee-e1113b694bee':1.1, 'f081b233-551c-4948-8840-29251e1952d3':0.5, 'ab1959c0-41be-4e55-8dd7-87f40892520a': 1.1, '0524a4e0-efeb-4780-93b9-be98b2f3fbe2':1, 'f58adc7b-8883-4ad7-a3ea-7e4b68cd3bc3':0.7, '396d8107-2dac-436e-a9f8-5883991655c0':1.2, 'd4f47c1f-2257-4dbc-8ff6-68666628f456':0.9, '35f5e778-1144-44fa-a70f-03743c031bee':0.7, 'e9913e66-2b89-489b-9978-f72e53e167b4':0.6, '5a15b335-c65a-4878-96b6-41f90e785dad':0.4, '6fcfb116-d342-4080-8bd8-fd080b462ef4':1, 'd1ab7c83-e333-423f-8470-237a8467e01d':1.1, '7df6de6a-e1a0-4b11-bffd-262c84b74789':1.2, '1885fb8a-c0f2-4b19-aaaa-111e4a40567e':1.1, '1fe03390-c49e-48da-b101-cbece0039968':0.6, 'c5ec9df5-f62f-48b9-8298-35f95fe64538': 0.9, 'e7d6e23d-bd03-4592-ab9e-4ae0695acd79':1.2, '2ac22fec-3cbd-4e5e-8ca7-520c785c3a36': 1.3, '183ec923-6902-4c1c-9035-506588249902':0.8, 'c2f15e2a-41fa-40b4-8779-8156a2cc74e2': 1.3, 'dce1cf60-fdce-45f9-83fa-b9708818a623': 1, '5dab0ba3-08df-49c0-a8b3-34219efb3a6d': 1, '4e28d4f0-5f4d-4783-b7fd-c47e07374471':1.3, '9597f979-d95b-48ca-8fd3-401357e98e1c':0.7, 'e5132fce-9605-4ab5-9083-c21a10e8afff':1.3, '7cbf859b-970e-4a6e-9960-78d1995aced9': 1.1}

# Functions Above

# 

In [51]:
#points_transfer_dict = {}

In [53]:
proceed = True

while proceed:

    all_teams = pd.read_csv('all_teams.csv')
    all_competitions = pd.read_csv('all_competitions.csv')

    ### Get all the fixtures, teams and competitions
    all_events = get_all_events()
    
    # Check that all the competitions and teams exist
    proceed = check_all_teams_exist(all_events, all_teams)
    proceed = check_competitions_exist(all_events, all_competitions)
    
    float_columns = ['home_pre_delta', 'home_post_delta', 'away_pre_delta', 'away_post_delta', 'pre_delta_diff', 'home_team_buffer', 'home_pre_delta_halftime', 'home_post_delta_halftime', 'away_pre_delta_halftime', 'away_post_delta_halftime', 'pre_delta_diff_halftime', 'home_team_buffer_halftime', 'home_pre_delta_secondhalf', 'home_post_delta_secondhalf', 'away_pre_delta_secondhalf', 'away_post_delta_secondhalf', 'pre_delta_diff_secondhalf', 'home_team_buffer_secondhalf', 'pre_delta_diff_adjusted', 'pre_delta_adjustment', 'pre_delta_adjustment_halftime', 'pre_delta_diff_halftime_adjusted', 'pre_delta_adjustment_secondhalf', 'pre_delta_diff_secondhalf_adjusted']
    all_previous_deltas = get_all_previous_deltas(float_columns)


    # Remove any faulty events
    all_events = remove_faulty_fixtures(all_events)
    all_events['total_points'] = all_events[['home_score', 'away_score']].apply(lambda x: x[0] + x[1] if ( pd.notna(x[0]) & pd.notna(x[1]) ) * ((x[0] > 0) | (x[1] > 0) ) else None, axis = 1)

    # Remove any events that no longer exist for whatever reason
    check_for_nonexistant_events(all_previous_deltas, all_events)

    # Check for duplicate events
    duplicate_events = check_for_duplicate_events(all_events)
    
    if len(duplicate_events):
        
        earliest_duplicate_event = all_events[ all_events['event_id'].isin(duplicate_events)].index.min()
        all_events = all_events.loc[ :earliest_duplicate_event]

        error_string = 'There are duplicate events that need fixed before the event deltas can be calculated - ' + str(duplicate_events)[1:-1                                                                                                                ]
        notifyTeams(error_string)
        print(error_string)
        proceed = False
    

    # Check to see if current events competition id and scores match up with what was calculated before
    previous_deltas_to_keep = check_fixtures_that_have_changed(all_previous_deltas, all_events)


    # Check to see if any new events have been added historically
    all_events = check_for_any_new_events(all_events, previous_deltas_to_keep, float_columns)


    all_competitions['level'] = all_competitions['level'].apply(lambda x: float(x) if x != 'na' else None)
    for key in new_comp_levels:
        all_competitions['level'] = all_competitions[['id', 'level']].apply(lambda x: new_comp_levels[x[0]] if x[0] == key else x[1] , axis = 1)



    ### Add in fixture numbers
    all_events = get_team_fixture_numbers(all_events)
    all_events = all_events.sort_values(['start_time', 'home_team_total_fixture_number', 'away_team_total_fixture_number'])


    ## Add venue info
    all_events, venues = add_venue_info(all_events)
    ## Add home venue info
    all_events = set_home_venues(all_events, all_teams, venues)


    ### Add competition details
    all_events = all_events.merge(all_competitions[['id', 'name', 'home_competition_group', 'level']].rename(columns = {'name':'competition_name'}), how = 'left', left_on = 'competition_internal_id', right_on = 'id')


    ### Add competition fixture numbers
    all_events = get_competition_fixture_numbers()


    ### Make sure the sort order is still chronological
    all_events = all_events.sort_values(['start_time', 'home_team_total_fixture_number', 'away_team_total_fixture_number'])

    # Set any ambigious halftime scores to None
    halftime_zero_indexes = all_events[ (all_events['home_halftime_score'] == 0) & (all_events['away_halftime_score'] == 0) ].index
    all_events.loc[halftime_zero_indexes, 'home_halftime_score'] = None
    all_events.loc[halftime_zero_indexes, 'away_halftime_score'] = None
    del halftime_zero_indexes

    # Make sure future events are set to None and not 0 for their scores
    future_events = all_events[ all_events['start_time'] >= str(datetime.datetime.now())].index
    all_events.loc[future_events, 'home_halftime_score'] = None
    all_events.loc[future_events, 'away_halftime_score'] = None



    ### Convert team dicts to dataframes
    international_rankings.update(international_rankings_women)
    internationl_rankings_df = team_dict_to_dataframe(international_rankings)

    ### Set start range of calculations
    start_range = all_events[ pd.isna(all_events['home_pre_delta']) | pd.isna(all_events['home_post_delta']) | pd.isna(all_events['away_pre_delta']) | pd.isna(all_events['away_post_delta'])].index[0]
    print(start_range)
    start_range = max(all_events[ all_events['start_time'] >= '2010-01-01'].index.min(), start_range)
    #start_range = min(all_events[ all_events['start_time'] >= '2010-01-01'].index.min(), start_range)
    
    end_range = all_events.index.max()
    print('Calculation Deltas from', all_events.loc[start_range, 'start_time'], all_events.loc[start_range, 'name'], all_events.loc[start_range, 'event_id'])

    #start_range = all_events[ all_events['start_time'] >= '2010-01-01'].index.min()
    
    max_points_win = 5
    win_margin_buffer = 0
    level_setting = 40
    win_bonus = 0

    home_team_fixture_column = 'home_team_total_fixture_number'
    away_team_fixture_column = 'away_team_total_fixture_number'


        

    ################################################################################
    ################################## Win Margin ##################################
    ################################################################################

            
    delta_column_to_calcuate = 'win_margin'
    all_events[delta_column_to_calcuate] = (all_events['home_score'] - all_events['away_score'])
    
    ### Get standard home win margins to use in the algorithm
    all_base_home_win_margin, international_mens_base_home_win_margin, international_womens_base_home_win_margin, competition_win_margin_means = get_comp_standards(all_events, delta_column_to_calcuate)

    ### Set column names
    home_pre_delta_name = 'home_pre_delta'
    home_post_delta_name = 'home_post_delta'
    away_pre_delta_name = 'away_pre_delta'
    away_post_delta_name = 'away_post_delta'
    pre_delta_diff_name = 'pre_delta_diff'
    home_team_buffer_name = 'home_team_buffer'
    post_delta_adjustment_name = 'pre_delta_adjustment'
    pre_delta_diff_adjusted = 'pre_delta_diff_adjusted'
    home_error = pre_delta_diff_name + '_home_team_home_error'
    away_error = pre_delta_diff_name + '_away_team_away_error'
    
    if home_error not in all_events.columns:
        all_events[home_error] = None
    if away_error not in all_events.columns:
        all_events[away_error] = None

    all_events = generate_elo_ranks(all_events, delta_column_to_calcuate, post_delta_adjustment_name, home_pre_delta_name, home_post_delta_name, away_pre_delta_name, away_post_delta_name, pre_delta_diff_name, home_team_buffer_name, max_points_win, win_margin_buffer, level_setting, start_range, end_range, list_order_int_men, list_order_int_women_age, list_order_club, win_bonus, all_competitions, all_teams, home_team_fixture_column, away_team_fixture_column, home_error, away_error)
    
        

    # Remove events that haven't been calculated - Potentially combined events affecting fixture numbers being calculated properly?
    events_to_remove = list(all_events[ pd.isna(all_events['pre_delta_diff']) ]['event_id'])
    if len(events_to_remove) > 0:
        all_events = all_events[ ~all_events['event_id'].isin(events_to_remove)]
        message = 'There are events where the pre_delta_diff could not be calculated, please check.  Select * from materialised_view_event where event_id in (' + str(events_to_remove) + ');'
        notifyTeams(message)
        ### Add in fixture numbers
        all_events = get_team_fixture_numbers(all_events)
        all_events = all_events.sort_values(['start_time', 'home_team_total_fixture_number', 'away_team_total_fixture_number'])
        all_events.reset_index(drop = True, inplace = True)
        
        ### Set start range of calculations
        #start_range = all_events[ pd.isna(all_events['home_pre_delta']) | pd.isna(all_events['home_post_delta']) | pd.isna(all_events['away_pre_delta']) | pd.isna(all_events['away_post_delta'])].index[0]
        start_range = max(all_events[ all_events['start_time'] >= '2010-01-01'].index.min(), start_range)
        end_range = all_events.index.max()
        print('Calculation Deltas from', all_events.loc[start_range, 'start_time'])
        
        
    sql_statement = "select event_id from event_deltas;"
    current_event_ids, error = postgres_Retreive_Insert(sql_statement, POSTGRESQL_PARAMS, retrieve_data = True)

    
    current_events = all_events.loc[start_range:]
    current_events = current_events[ (current_events['event_id'].isin(list(current_event_ids['event_id']))) ]
    new_events = all_events.loc[start_range:]
    new_events = new_events[ ~new_events['event_id'].isin(list(current_event_ids['event_id'])) ]
    
    new_events_venue = new_events[ pd.notna(new_events['venue_internal_id'])]
    if len(new_events_venue)>0:
        formatted_data = new_events_venue[[
    'event_id',
    'home_team_internal_id',
    'away_team_internal_id',
    'competition_internal_id',
    'venue_internal_id',
    'start_time',
    'home_score',
    'away_score',
    'home_pre_delta',
    'away_pre_delta',
    'pre_delta_diff',
    'pre_delta_diff_adjusted',
    'home_post_delta',
    'away_post_delta',
    'home_team_buffer',
    'pre_delta_adjustment', home_error, away_error]]
        powerbi_table_info, error = get_table_info('event_deltas')
        formatted_data = format_data_for_postgres(formatted_data, powerbi_table_info)
        insert_sql('event_deltas', formatted_data)
        
        
    new_events_novenue = new_events[ pd.isna(new_events['venue_internal_id'])]
    if len(new_events_novenue)>0:
        formatted_data = new_events_novenue[[
    'event_id',
    'home_team_internal_id',
    'away_team_internal_id',
    'competition_internal_id',
    'venue_internal_id',
    'start_time',
    'home_score',
    'away_score',
    'home_pre_delta',
    'away_pre_delta',
    'pre_delta_diff',
    'pre_delta_diff_adjusted',
    'home_post_delta',
    'away_post_delta',
    'home_team_buffer',
    'pre_delta_adjustment', home_error, away_error]]
        powerbi_table_info, error = get_table_info('event_deltas')
        formatted_data = format_data_for_postgres(formatted_data, powerbi_table_info)
        insert_sql('event_deltas', formatted_data)
    
    
    current_events_venue = current_events[ pd.notna(current_events['venue_internal_id']) & (current_events['venue_internal_id'] != 'None')]
    if len(current_events_venue)>0:
        formatted_data = current_events_venue[[
    'event_id',
    'home_team_internal_id',
    'away_team_internal_id',
    'competition_internal_id',
    'venue_internal_id',
    'start_time',
    'home_score',
    'away_score',
    'home_pre_delta',
    'away_pre_delta',
    'pre_delta_diff',
    'pre_delta_diff_adjusted',
    'home_post_delta',
    'away_post_delta',
    'home_team_buffer',
    'pre_delta_adjustment', home_error, away_error]]
        powerbi_table_info, error = get_table_info('event_deltas')
        formatted_data = format_data_for_postgres(formatted_data, powerbi_table_info)
        update_sql('event_deltas', formatted_data, 'event_id')
        
    
    current_events_novenue = current_events[ pd.isna(current_events['venue_internal_id'])]
    if len(current_events_novenue)>0:
        formatted_data = current_events_novenue[[
    'event_id',
    'home_team_internal_id',
    'away_team_internal_id',
    'competition_internal_id',
    'start_time',
    'home_score',
    'away_score',
    'home_pre_delta',
    'away_pre_delta',
    'pre_delta_diff',
    'pre_delta_diff_adjusted',
    'home_post_delta',
    'away_post_delta',
    'home_team_buffer',
    'pre_delta_adjustment', home_error, away_error]]
        powerbi_table_info, error = get_table_info('event_deltas')
        formatted_data = format_data_for_postgres(formatted_data, powerbi_table_info)
        update_sql('event_deltas', formatted_data, 'event_id')
        
        
    ########################################################################
    ########################################################################
    ########################################################################

    
    proceed = False


-get_all_events
--Complete-0:00:16.504473

-get_all_previous_deltas
--Complete-0:00:29.051047

-remove_faulty_fixtures
--Complete-0:00:24.352351

-check_for_nonexistant_events
--Complete-0:00:00.124999

There are duplicate events that need fixed before the event deltas can be calculated - '7de6e2b2-1b96-11ee-9367-0242ac120002', '7dba7d3a-1b96-11ee-9367-0242ac120002', '7d7770f8-1b96-11ee-9367-0242ac120002', '7dd05222-1b96-11ee-9367-0242ac120002', '7d8e6204-1b96-11ee-9367-0242ac120002'
Notifying teams
Request sent
There are duplicate events that need fixed before the event deltas can be calculated - '7de6e2b2-1b96-11ee-9367-0242ac120002', '7dba7d3a-1b96-11ee-9367-0242ac120002', '7d7770f8-1b96-11ee-9367-0242ac120002', '7dd05222-1b96-11ee-9367-0242ac120002', '7d8e6204-1b96-11ee-9367-0242ac120002'
-check_fixtures_that_have_changed
--Complete-0:00:09.077915

-check_for_any_new_events
--Complete-0:00:00.437495

-get_team_fixture_numbers
-set_home_venues
--Complete-0:03:27.575338

-get_competi

-- 13907 57153
The home delta is blank for event  13907 I Cavalieri Prato vs Roma
The away delta is blank for event  13907 I Cavalieri Prato vs Roma
-- 13908 57153
The home delta is blank for event  13908 Venezia vs Viadana
The away delta is blank for event  13908 Venezia vs Viadana
-- 13909 57153
The home delta is blank for event  13909 L'Aquila Rugby Club vs Gran Parma
The away delta is blank for event  13909 L'Aquila Rugby Club vs Gran Parma
-- 13910 57153
The home delta is blank for event  13910 Crociati vs Rovigo
The away delta is blank for event  13910 Crociati vs Rovigo
-- 13911 57153
The home delta is blank for event  13911 Lannemezan vs Grenoble
The away delta is blank for event  13911 Lannemezan vs Grenoble
-- 13912 57153
The home delta is blank for event  13912 Bordeaux Begles vs Lyon
The away delta is blank for event  13912 Bordeaux Begles vs Lyon
-- 13913 57153
The home delta is blank for event  13913 Toulouse vs Clermont Auvergne
The away delta is blank for event  13913 T

-- 13964 57153
The home delta is blank for event  13964 Montauban vs Albi
The away delta is blank for event  13964 Montauban vs Albi
-- 13965 57153
The home delta is blank for event  13965 Munster vs Northampton
The away delta is blank for event  13965 Munster vs Northampton
-- 13966 57153
The home delta is blank for event  13966 Montpellier vs Worcester
The away delta is blank for event  13966 Montpellier vs Worcester
-- 13967 57153
The home delta is blank for event  13967 Perpignan vs Benetton
The away delta is blank for event  13967 Perpignan vs Benetton
-- 13968 57153
The home delta is blank for event  13968 Newcastle Falcons vs Petrarca Padova
The away delta is blank for event  13968 Newcastle Falcons vs Petrarca Padova
-- 13969 57153
The home delta is blank for event  13969 West of Scotland vs Selkirk
The away delta is blank for event  13969 West of Scotland vs Selkirk
-- 13970 57153
The home delta is blank for event  13970 Blaydon vs Redruth
The away delta is blank for event  13

The home delta is blank for event  14020 Toulon vs Montpellier
The away delta is blank for event  14020 Toulon vs Montpellier
-- 14021 57153
The home delta is blank for event  14021 Brive vs Montauban
The away delta is blank for event  14021 Brive vs Montauban
-- 14022 57153
The home delta is blank for event  14022 Castres vs Stade Francais
The away delta is blank for event  14022 Castres vs Stade Francais
-- 14023 57153
The home delta is blank for event  14023 Albi vs Toulouse
The away delta is blank for event  14023 Albi vs Toulouse
-- 14024 57153
The home delta is blank for event  14024 Perpignan vs Bourgoin
The away delta is blank for event  14024 Perpignan vs Bourgoin
-- 14025 57153
The home delta is blank for event  14025 Bayonne vs Biarritz
The away delta is blank for event  14025 Bayonne vs Biarritz
-- 14026 57153
The home delta is blank for event  14026 Dragons vs Ospreys
The away delta is blank for event  14026 Dragons vs Ospreys
-- 14027 57153
The home delta is blank for eve

The away delta is blank for event  14076 Cambridge vs Cinderford
-- 14077 57153
The home delta is blank for event  14077 Swansea vs Llandovery
The away delta is blank for event  14077 Swansea vs Llandovery
-- 14078 57153
The home delta is blank for event  14078 Blackheath vs London Scottish
The away delta is blank for event  14078 Blackheath vs London Scottish
-- 14079 57153
The home delta is blank for event  14079 Neath vs Glamorgan Wanderers
The away delta is blank for event  14079 Neath vs Glamorgan Wanderers
-- 14080 57153
The home delta is blank for event  14080 Esher vs Wharfedale
The away delta is blank for event  14080 Esher vs Wharfedale
-- 14081 57153
The home delta is blank for event  14081 Moseley vs Bedford Blues
The away delta is blank for event  14081 Moseley vs Bedford Blues
-- 14082 57153
The home delta is blank for event  14082 Sedgley Park vs Redruth
The away delta is blank for event  14082 Sedgley Park vs Redruth
-- 14083 57153
The home delta is blank for event  140

...

The home delta is blank for event  14651 Clermont Auvergne vs Castres
The away delta is blank for event  14651 Clermont Auvergne vs Castres
-- 14652 57153
The home delta is blank for event  14652 Singapore Women vs Malaysia Women
The away delta is blank for event  14652 Singapore Women vs Malaysia Women
-- 14653 57153
The home delta is blank for event  14653 London Scottish vs Blackheath
The away delta is blank for event  14653 London Scottish vs Blackheath
-- 14654 57153
The home delta is blank for event  14654 Eagles vs Border Bulldogs
The away delta is blank for event  14654 Eagles vs Border Bulldogs
-- 14655 57153
The home delta is blank for event  14655 West of Scotland vs Currie
The away delta is blank for event  14655 West of Scotland vs Currie
-- 14656 57153
The home delta is blank for event  14656 Cinderford vs Cambridge
The away delta is blank for event  14656 Cinderford vs Cambridge
-- 14657 57153
The home delta is blank for event  14657 Edinburgh Academical vs Heriots
The a

The home delta is blank for event  14709 Nottingham Rugby vs Doncaster Knights
The away delta is blank for event  14709 Nottingham Rugby vs Doncaster Knights
-- 14710 57153
The home delta is blank for event  14710 Bristol vs Bedford Blues
The away delta is blank for event  14710 Bristol vs Bedford Blues
-- 14711 57153
The home delta is blank for event  14711 London Irish vs Yorkshire Carnegie
The away delta is blank for event  14711 London Irish vs Yorkshire Carnegie
-- 14712 57153
The home delta is blank for event  14712 Agen vs Narbonne
The away delta is blank for event  14712 Agen vs Narbonne
-- 14713 57153
The home delta is blank for event  14713 Newcastle Falcons vs Leicester
The away delta is blank for event  14713 Newcastle Falcons vs Leicester
-- 14714 57153
The home delta is blank for event  14714 Connacht vs Munster
The away delta is blank for event  14714 Connacht vs Munster
-- 14715 57153
The home delta is blank for event  14715 Dragons vs Edinburgh
The away delta is blank 

The home delta is blank for event  14768 Perpignan vs Albi
The away delta is blank for event  14768 Perpignan vs Albi
-- 14769 57153
The home delta is blank for event  14769 Biarritz vs Clermont Auvergne
The away delta is blank for event  14769 Biarritz vs Clermont Auvergne
-- 14770 57153
The home delta is blank for event  14770 Toulouse vs Castres
The away delta is blank for event  14770 Toulouse vs Castres
-- 14771 57153
The home delta is blank for event  14771 Kazakhstan vs Arabian Gulf
The away delta is blank for event  14771 Kazakhstan vs Arabian Gulf
-- 14772 57153
The home delta is blank for event  14772 Hong Kong vs South Korea
The away delta is blank for event  14772 Hong Kong vs South Korea
-- 14773 57153
The home delta is blank for event  14773 Belgium vs Poland
The away delta is blank for event  14773 Belgium vs Poland
-- 14774 57153
The home delta is blank for event  14774 Bulgaria vs Greece
The away delta is blank for event  14774 Bulgaria vs Greece
-- 14775 57153
The hom

-- 14826 57153
The home delta is blank for event  14826 Upolu Samoa vs Savai'i Samoa
The away delta is blank for event  14826 Upolu Samoa vs Savai'i Samoa
-- 14827 57153
The home delta is blank for event  14827 Heriots vs Ayr
The away delta is blank for event  14827 Heriots vs Ayr
-- 14828 57153
The home delta is blank for event  14828 Tautahi Gold vs Tau'uta Reds
The away delta is blank for event  14828 Tautahi Gold vs Tau'uta Reds
-- 14829 57153
The home delta is blank for event  14829 Fiji A vs Fiji Barbarians
The away delta is blank for event  14829 Fiji A vs Fiji Barbarians
-- 14830 57153
The home delta is blank for event  14830 Hurricanes vs Reds
The away delta is blank for event  14830 Hurricanes vs Reds
-- 14831 57153
The home delta is blank for event  14831 Bulls vs Crusaders
The away delta is blank for event  14831 Bulls vs Crusaders
-- 14832 57153
The home delta is blank for event  14832 Ospreys vs Dragons
The away delta is blank for event  14832 Ospreys vs Dragons
-- 14833 

KeyboardInterrupt: 

In [64]:
    all_teams = pd.read_csv('all_teams.csv')
    all_competitions = pd.read_csv('all_competitions.csv')

    ### Get all the fixtures, teams and competitions
    all_events = get_all_events()
    
    # Check that all the competitions and teams exist
    proceed = check_all_teams_exist(all_events, all_teams)
    proceed = check_competitions_exist(all_events, all_competitions)
    
    float_columns = ['home_pre_delta', 'home_post_delta', 'away_pre_delta', 'away_post_delta', 'pre_delta_diff', 'home_team_buffer', 'home_pre_delta_halftime', 'home_post_delta_halftime', 'away_pre_delta_halftime', 'away_post_delta_halftime', 'pre_delta_diff_halftime', 'home_team_buffer_halftime', 'home_pre_delta_secondhalf', 'home_post_delta_secondhalf', 'away_pre_delta_secondhalf', 'away_post_delta_secondhalf', 'pre_delta_diff_secondhalf', 'home_team_buffer_secondhalf', 'pre_delta_diff_adjusted', 'pre_delta_adjustment', 'pre_delta_adjustment_halftime', 'pre_delta_diff_halftime_adjusted', 'pre_delta_adjustment_secondhalf', 'pre_delta_diff_secondhalf_adjusted']
    all_previous_deltas = get_all_previous_deltas(float_columns)


    # Remove any faulty events
    all_events = remove_faulty_fixtures(all_events)
    all_events['total_points'] = all_events[['home_score', 'away_score']].apply(lambda x: x[0] + x[1] if ( pd.notna(x[0]) & pd.notna(x[1]) ) * ((x[0] > 0) | (x[1] > 0) ) else None, axis = 1)

    # Remove any events that no longer exist for whatever reason
    check_for_nonexistant_events(all_previous_deltas, all_events)

    # Check for duplicate events
    duplicate_events = check_for_duplicate_events(all_events)
    
    if len(duplicate_events):
        
        earliest_duplicate_event = all_events[ all_events['event_id'].isin(duplicate_events)].index.min()
        all_events = all_events.loc[ :earliest_duplicate_event]

        error_string = 'There are duplicate events that need fixed before the event deltas can be calculated - ' + str(duplicate_events)[1:-1                                                                                                                ]
        notifyTeams(error_string)
        print(error_string)
        proceed = False
    

    # Check to see if current events competition id and scores match up with what was calculated before
    previous_deltas_to_keep = check_fixtures_that_have_changed(all_previous_deltas, all_events)



-get_all_events
--Complete-0:00:15.758111

-get_all_previous_deltas
--Complete-0:00:23.391917

-remove_faulty_fixtures
--Complete-0:00:07.017460

-check_for_nonexistant_events
--Complete-0:00:00.046881

There are duplicate events that need fixed before the event deltas can be calculated - '7d7770f8-1b96-11ee-9367-0242ac120002', '7dd05222-1b96-11ee-9367-0242ac120002'
Notifying teams
Request sent
There are duplicate events that need fixed before the event deltas can be calculated - '7d7770f8-1b96-11ee-9367-0242ac120002', '7dd05222-1b96-11ee-9367-0242ac120002'
-check_fixtures_that_have_changed
--Complete-0:00:04.544249



In [65]:
    function_start_time = datetime.datetime.now()
    print('-check_for_any_new_events')
    
    temp_columns = float_columns.copy()
    temp_columns.append('event_id')
    all_previous_deltas = all_previous_deltas[temp_columns]
    
    all_events = all_events.merge(all_previous_deltas.rename(columns = {'event_id':'event_id_PD'}), how = 'left', left_on = 'event_id', right_on = 'event_id_PD').reset_index()    
    earliest_fixture_change = all_events[ pd.isna(all_events['home_pre_delta']) |  pd.isna(all_events['home_post_delta']) |  pd.isna(all_events['away_pre_delta']) |  pd.isna(all_events['away_post_delta']) ]['start_time'].min()


-check_for_any_new_events


In [67]:
earliest_fixture_change

Timestamp('2023-05-13 00:00:00+0000', tz='UTC')

In [68]:

    for col in float_columns:
        all_events.loc[ all_events['start_time'] >= earliest_fixture_change, col] = None
    #all_events.loc[ all_events['start_time'] >= earliest_fixture_change, 'away_pre_delta'] = None
    #all_events.loc[ all_events['start_time'] >= earliest_fixture_change, 'home_post_delta'] = None
    #all_events.loc[ all_events['start_time'] >= earliest_fixture_change, 'away_post_delta'] = None

    function_end_time = datetime.datetime.now()
    print('--Complete-' + str(function_end_time - function_start_time))
    print('')


--Complete-0:19:41.783458



In [70]:
    # Check to see if any new events have been added historically
#     all_events = check_for_any_new_events(all_events, previous_deltas_to_keep, float_columns)


    all_competitions['level'] = all_competitions['level'].apply(lambda x: float(x) if x != 'na' else None)
    for key in new_comp_levels:
        all_competitions['level'] = all_competitions[['id', 'level']].apply(lambda x: new_comp_levels[x[0]] if x[0] == key else x[1] , axis = 1)



In [72]:
    ### Add in fixture numbers
    all_events = get_team_fixture_numbers(all_events)
    all_events = all_events.sort_values(['start_time', 'home_team_total_fixture_number', 'away_team_total_fixture_number'])


    ## Add venue info
    all_events, venues = add_venue_info(all_events)
    ## Add home venue info
    all_events = set_home_venues(all_events, all_teams, venues)


-get_team_fixture_numbers
-set_home_venues
--Complete-0:02:24.376775



In [73]:
all_events['home_pre_delta']

0        174.436265
1        151.567911
2        149.813130
3        166.263307
4        161.971236
            ...    
57155           NaN
57156           NaN
57157           NaN
57158           NaN
57159           NaN
Name: home_pre_delta, Length: 57160, dtype: float64

In [74]:



    ### Add competition details
    all_events = all_events.merge(all_competitions[['id', 'name', 'home_competition_group', 'level']].rename(columns = {'name':'competition_name'}), how = 'left', left_on = 'competition_internal_id', right_on = 'id')


    ### Add competition fixture numbers
    all_events = get_competition_fixture_numbers()


    ### Make sure the sort order is still chronological
    all_events = all_events.sort_values(['start_time', 'home_team_total_fixture_number', 'away_team_total_fixture_number'])

    # Set any ambigious halftime scores to None
    halftime_zero_indexes = all_events[ (all_events['home_halftime_score'] == 0) & (all_events['away_halftime_score'] == 0) ].index
    all_events.loc[halftime_zero_indexes, 'home_halftime_score'] = None
    all_events.loc[halftime_zero_indexes, 'away_halftime_score'] = None
    del halftime_zero_indexes

    # Make sure future events are set to None and not 0 for their scores
    future_events = all_events[ all_events['start_time'] >= str(datetime.datetime.now())].index
    all_events.loc[future_events, 'home_halftime_score'] = None
    all_events.loc[future_events, 'away_halftime_score'] = None



    ### Convert team dicts to dataframes
    international_rankings.update(international_rankings_women)
    internationl_rankings_df = team_dict_to_dataframe(international_rankings)

    ### Set start range of calculations
    start_range = all_events[ pd.isna(all_events['home_pre_delta']) | pd.isna(all_events['home_post_delta']) | pd.isna(all_events['away_pre_delta']) | pd.isna(all_events['away_post_delta'])].index[0]
    print(start_range)
    start_range = max(all_events[ all_events['start_time'] >= '2010-01-01'].index.min(), start_range)
    #start_range = min(all_events[ all_events['start_time'] >= '2010-01-01'].index.min(), start_range)
    
    end_range = all_events.index.max()
    print('Calculation Deltas from', all_events.loc[start_range, 'start_time'], all_events.loc[start_range, 'name'], all_events.loc[start_range, 'event_id'])

    #start_range = all_events[ all_events['start_time'] >= '2010-01-01'].index.min()
    
    max_points_win = 5
    win_margin_buffer = 0
    level_setting = 40
    win_bonus = 0

    home_team_fixture_column = 'home_team_total_fixture_number'
    away_team_fixture_column = 'away_team_total_fixture_number'



-get_competition_fixture_numbers
--Complete-0:00:00.062515

-team_dict_to_dataframe
--Complete-0:00:00

56042
Calculation Deltas from 2023-05-13 00:00:00+00:00 Krasny Yar vs Dynamo Moscow 65dd5a16-a95e-11ed-88a1-0242ac120002


In [ ]:
all_events['home_pre_delta']

In [75]:
    

    ################################################################################
    ################################## Win Margin ##################################
    ################################################################################

            
    delta_column_to_calcuate = 'win_margin'
    all_events[delta_column_to_calcuate] = (all_events['home_score'] - all_events['away_score'])
    
    ### Get standard home win margins to use in the algorithm
    all_base_home_win_margin, international_mens_base_home_win_margin, international_womens_base_home_win_margin, competition_win_margin_means = get_comp_standards(all_events, delta_column_to_calcuate)

    ### Set column names
    home_pre_delta_name = 'home_pre_delta'
    home_post_delta_name = 'home_post_delta'
    away_pre_delta_name = 'away_pre_delta'
    away_post_delta_name = 'away_post_delta'
    pre_delta_diff_name = 'pre_delta_diff'
    home_team_buffer_name = 'home_team_buffer'
    post_delta_adjustment_name = 'pre_delta_adjustment'
    pre_delta_diff_adjusted = 'pre_delta_diff_adjusted'
    home_error = pre_delta_diff_name + '_home_team_home_error'
    away_error = pre_delta_diff_name + '_away_team_away_error'
    
    if home_error not in all_events.columns:
        all_events[home_error] = None
    if away_error not in all_events.columns:
        all_events[away_error] = None

    all_events = generate_elo_ranks(all_events, delta_column_to_calcuate, post_delta_adjustment_name, home_pre_delta_name, home_post_delta_name, away_pre_delta_name, away_post_delta_name, pre_delta_diff_name, home_team_buffer_name, max_points_win, win_margin_buffer, level_setting, start_range, end_range, list_order_int_men, list_order_int_women_age, list_order_club, win_bonus, all_competitions, all_teams, home_team_fixture_column, away_team_fixture_column, home_error, away_error)
    
        

    # Remove events that haven't been calculated - Potentially combined events affecting fixture numbers being calculated properly?
    events_to_remove = list(all_events[ pd.isna(all_events['pre_delta_diff']) ]['event_id'])
    if len(events_to_remove) > 0:
        all_events = all_events[ ~all_events['event_id'].isin(events_to_remove)]
        message = 'There are events where the pre_delta_diff could not be calculated, please check.  Select * from materialised_view_event where event_id in (' + str(events_to_remove) + ');'
        notifyTeams(message)
        ### Add in fixture numbers
        all_events = get_team_fixture_numbers(all_events)
        all_events = all_events.sort_values(['start_time', 'home_team_total_fixture_number', 'away_team_total_fixture_number'])
        all_events.reset_index(drop = True, inplace = True)
        
        ### Set start range of calculations
        #start_range = all_events[ pd.isna(all_events['home_pre_delta']) | pd.isna(all_events['home_post_delta']) | pd.isna(all_events['away_pre_delta']) | pd.isna(all_events['away_post_delta'])].index[0]
        start_range = max(all_events[ all_events['start_time'] >= '2010-01-01'].index.min(), start_range)
        end_range = all_events.index.max()
        print('Calculation Deltas from', all_events.loc[start_range, 'start_time'])
        
        
    sql_statement = "select event_id from event_deltas;"
    current_event_ids, error = postgres_Retreive_Insert(sql_statement, POSTGRESQL_PARAMS, retrieve_data = True)

    
    current_events = all_events.loc[start_range:]
    current_events = current_events[ (current_events['event_id'].isin(list(current_event_ids['event_id']))) ]
    new_events = all_events.loc[start_range:]
    new_events = new_events[ ~new_events['event_id'].isin(list(current_event_ids['event_id'])) ]
    
    new_events_venue = new_events[ pd.notna(new_events['venue_internal_id'])]
    if len(new_events_venue)>0:
        formatted_data = new_events_venue[[
    'event_id',
    'home_team_internal_id',
    'away_team_internal_id',
    'competition_internal_id',
    'venue_internal_id',
    'start_time',
    'home_score',
    'away_score',
    'home_pre_delta',
    'away_pre_delta',
    'pre_delta_diff',
    'pre_delta_diff_adjusted',
    'home_post_delta',
    'away_post_delta',
    'home_team_buffer',
    'pre_delta_adjustment', home_error, away_error]]
        powerbi_table_info, error = get_table_info('event_deltas')
        formatted_data = format_data_for_postgres(formatted_data, powerbi_table_info)
        insert_sql('event_deltas', formatted_data)
        
        
    new_events_novenue = new_events[ pd.isna(new_events['venue_internal_id'])]
    if len(new_events_novenue)>0:
        formatted_data = new_events_novenue[[
    'event_id',
    'home_team_internal_id',
    'away_team_internal_id',
    'competition_internal_id',
    'venue_internal_id',
    'start_time',
    'home_score',
    'away_score',
    'home_pre_delta',
    'away_pre_delta',
    'pre_delta_diff',
    'pre_delta_diff_adjusted',
    'home_post_delta',
    'away_post_delta',
    'home_team_buffer',
    'pre_delta_adjustment', home_error, away_error]]
        powerbi_table_info, error = get_table_info('event_deltas')
        formatted_data = format_data_for_postgres(formatted_data, powerbi_table_info)
        insert_sql('event_deltas', formatted_data)
    
    
    current_events_venue = current_events[ pd.notna(current_events['venue_internal_id']) & (current_events['venue_internal_id'] != 'None')]
    if len(current_events_venue)>0:
        formatted_data = current_events_venue[[
    'event_id',
    'home_team_internal_id',
    'away_team_internal_id',
    'competition_internal_id',
    'venue_internal_id',
    'start_time',
    'home_score',
    'away_score',
    'home_pre_delta',
    'away_pre_delta',
    'pre_delta_diff',
    'pre_delta_diff_adjusted',
    'home_post_delta',
    'away_post_delta',
    'home_team_buffer',
    'pre_delta_adjustment', home_error, away_error]]
        powerbi_table_info, error = get_table_info('event_deltas')
        formatted_data = format_data_for_postgres(formatted_data, powerbi_table_info)
        update_sql('event_deltas', formatted_data, 'event_id')
        
    
    current_events_novenue = current_events[ pd.isna(current_events['venue_internal_id'])]
    if len(current_events_novenue)>0:
        formatted_data = current_events_novenue[[
    'event_id',
    'home_team_internal_id',
    'away_team_internal_id',
    'competition_internal_id',
    'start_time',
    'home_score',
    'away_score',
    'home_pre_delta',
    'away_pre_delta',
    'pre_delta_diff',
    'pre_delta_diff_adjusted',
    'home_post_delta',
    'away_post_delta',
    'home_team_buffer',
    'pre_delta_adjustment', home_error, away_error]]
        powerbi_table_info, error = get_table_info('event_deltas')
        formatted_data = format_data_for_postgres(formatted_data, powerbi_table_info)
        update_sql('event_deltas', formatted_data, 'event_id')
        
        
    ########################################################################
    ########################################################################
    ########################################################################



-get_comp_standards
--Complete-0:00:00.106998

-generate_elo_ranks
0 --updating from 56042 57159
-- 56042 57159
-- 56043 57159
-- 56044 57159
-- 56045 57159
-- 56046 57159
-- 56047 57159
-- 56048 57159
-- 56049 57159
-- 56050 57159
-- 56051 57159
-- 56052 57159
-- 56053 57159
-- 56054 57159
-- 56055 57159
-- 56056 57159
-- 56057 57159
-- 56058 57159
-- 56059 57159
-- 56060 57159
-- 56061 57159
-- 56062 57159
-- 56063 57159
-- 56064 57159
-- 56065 57159
-- 56066 57159
-- 56067 57159
-- 56068 57159
-- 56069 57159
-- 56070 57159
-- 56071 57159
-- 56072 57159
-- 56073 57159
-- 56074 57159
-- 56075 57159
-- 56076 57159
-- 56077 57159
-- 56078 57159
-- 56079 57159
-- 56080 57159
-- 56081 57159
-- 56082 57159
-- 56083 57159
-- 56084 57159
-- 56085 57159
-- 56086 57159
-- 56087 57159
-- 56088 57159
-- 56089 57159
-- 56090 57159
-- 56091 57159
-- 56092 57159
-- 56093 57159
-- 56094 57159
-- 56095 57159
-- 56096 57159
-- 56097 57159
-- 56098 57159
-- 56099 57159
-- 56100 57159
-- 56101 57159
-- 

-- 56583 57159
-- 56584 57159
-- 56585 57159
-- 56586 57159
-- 56587 57159
-- 56588 57159
-- 56589 57159
-- 56590 57159
-- 56591 57159
-- 56592 57159
-- 56593 57159
-- 56594 57159
-- 56595 57159
-- 56596 57159
-- 56597 57159
-- 56598 57159
-- 56599 57159
-- 56600 57159
-- 56601 57159
-- 56602 57159
-- 56603 57159
-- 56604 57159
-- 56605 57159
-- 56606 57159
-- 56607 57159
-- 56608 57159
-- 56609 57159
-- 56610 57159
-- 56611 57159
-- 56612 57159
-- 56613 57159
-- 56614 57159
-- 56615 57159
-- 56616 57159
-- 56617 57159
-- 56618 57159
-- 56619 57159
-- 56620 57159
-- 56621 57159
-- 56622 57159
-- 56623 57159
-- 56624 57159
-- 56625 57159
-- 56626 57159
-- 56627 57159
-- 56628 57159
-- 56629 57159
-- 56630 57159
-- 56631 57159
-- 56632 57159
-- 56633 57159
-- 56634 57159
-- 56635 57159
-- 56636 57159
-- 56637 57159
-- 56638 57159
-- 56639 57159
-- 56640 57159
-- 56641 57159
-- 56642 57159
-- 56643 57159
-- 56644 57159
-- 56645 57159
-- 56646 57159
-- 56647 57159
-- 56648 57159
-- 56649 5

-- 57130 57159
-- 57131 57159
-- 57132 57159
-- 57133 57159
-- 57134 57159
-- 57135 57159
-- 57136 57159
-- 57137 57159
-- 57138 57159
-- 57139 57159
-- 57140 57159
-- 57141 57159
-- 57142 57159
-- 57143 57159
-- 57144 57159
-- 57145 57159
-- 57146 57159
-- 57147 57159
-- 57148 57159
-- 57149 57159
-- 57150 57159
-- 57151 57159
-- 57152 57159
-- 57153 57159
-- 57154 57159
-- 57155 57159
-- 57156 57159
-- 57157 57159
-- 57158 57159
-- 57159 57159
--Complete-0:01:57.879327

insert_sql - Inserts Complete - 337
insert_sql - Inserts Complete - 23
update_sql - Update Complete - 665
update_sql - Update Complete - 93


In [ ]:




    ### Add in fixture numbers
    all_events = get_team_fixture_numbers(all_events)
    all_events = all_events.sort_values(['start_time', 'home_team_total_fixture_number', 'away_team_total_fixture_number'])


    ## Add venue info
    all_events, venues = add_venue_info(all_events)
    ## Add home venue info
    all_events = set_home_venues(all_events, all_teams, venues)

In [ ]:
# Take 0.22 off URC teams

median_change_values = pd.DataFrame( data = ['ee8f8374-1c04-4276-ab75-4bafc7441912', '1f4f7424-1bc0-4aed-adc2-6ccfab40fbae', 'cfe6acd2-6656-4c02-bfae-28bd6f368cfc', '33c5dddd-470e-4bbf-888a-41de5eba7f59'], columns = ['national_team_id'])
median_change_values['change'] = -0.22

In [ ]:
temp = all_teams.merge(median_change_values, how = 'inner', left_on = 'national_team_id', right_on = 'national_team_id')

In [ ]:
median_change_values = temp[['id', 'change']].rename(columns = {'id':'team_internal_id'})

In [ ]:
median_change_values = pd.read_csv('posneg_change_values.csv')

In [ ]:
median_change_values

In [ ]:
median_change_values

In [ ]:
all_events = all_events.merge(median_change_values[['team_internal_id', 'change']], how = 'left', left_on = 'home_team_internal_id', right_on = 'team_internal_id')

In [ ]:
all_events['home_post_delta'] = all_events[['home_post_delta', 'change']].apply(lambda x: x[0] if pd.isna(x[1]) else x[0] + x[1], axis = 1)
all_events['home_pre_delta'] = all_events[['home_pre_delta', 'change']].apply(lambda x: x[0] if pd.isna(x[1]) else x[0] + x[1], axis = 1)

In [ ]:
all_events.drop(['team_internal_id', 'change'], axis = 1, inplace = True)

In [ ]:
all_events = all_events.merge(median_change_values[['team_internal_id', 'change']], how = 'left', left_on = 'away_team_internal_id', right_on = 'team_internal_id')

In [ ]:
all_events['away_post_delta'] = all_events[['away_post_delta', 'change']].apply(lambda x: x[0] if pd.isna(x[1]) else x[0] + x[1], axis = 1)
all_events['away_pre_delta'] = all_events[['away_pre_delta', 'change']].apply(lambda x: x[0] if pd.isna(x[1]) else x[0] + x[1], axis = 1)

In [ ]:
all_events.drop(['team_internal_id', 'change'], axis = 1, inplace = True)

In [ ]:
start_range = all_events[ all_events['start_time']>= '2010-01-01'].index.min()

In [ ]:
start_range

In [ ]:
all_events = generate_elo_ranks(all_events, delta_column_to_calcuate, post_delta_adjustment_name, home_pre_delta_name, home_post_delta_name, away_pre_delta_name, away_post_delta_name, pre_delta_diff_name, home_team_buffer_name, max_points_win, win_margin_buffer, level_setting, start_range, end_range, list_order_int_men, list_order_int_women_age, list_order_club, win_bonus, all_competitions, all_teams, home_team_fixture_column, away_team_fixture_column, home_error, away_error)


In [ ]:
events_to_remove = list(all_events[ pd.isna(all_events['pre_delta_diff']) ]['event_id'])
events_to_remove

In [ ]:
all_events.to_csv('all_events_median_adjustement_4.csv', index = False)

In [ ]:
all_events[['name', 'home_post_delta']]

In [ ]:
all_events[['name', 'home_post_delta']]

In [ ]:
all_events = all_events[ pd.notna(all_events['home_pre_delta']) & pd.notna(all_events['home_post_delta']) & pd.notna(all_events['away_pre_delta']) & pd.notna(all_events['away_post_delta'])]

In [ ]:
all_events = pd.read_csv('all_events_median_adjustement.csv')

In [ ]:
    start_range = all_events[ pd.isna(all_events['home_pre_delta']) | pd.isna(all_events['home_post_delta']) | pd.isna(all_events['away_pre_delta']) | pd.isna(all_events['away_post_delta'])].index[0]
    print(start_range)
    start_range = max(all_events[ all_events['start_time'] >= '2010-01-01'].index.min(), start_range)
    #start_range = min(all_events[ all_events['start_time'] >= '2010-01-01'].index.min(), start_range)


In [ ]:
    start_range = all_events[ all_events['start_time'] >= '2010-01-01'].index.min()


In [ ]:
all_events =pd.read_csv('event_deltas_202305252057.csv')

In [76]:
    ########################### First Half ###########################
    delta_column_to_calcuate = 'half_time_win_margin'
    all_events[delta_column_to_calcuate] = all_events[['home_halftime_score', 'away_halftime_score']].apply(lambda x: x[0] - x[1] if (pd.notna(x[0]) and pd.notna(x[1])) and ((x[0]>0) | (x[1]>0)) else None )
    
    
    ### Get standard home win margins to use in the algorithm
    all_base_home_win_margin, international_mens_base_home_win_margin, international_womens_base_home_win_margin, competition_win_margin_means = get_comp_standards(all_events, delta_column_to_calcuate)

    home_pre_delta_name = 'home_pre_delta_halftime'
    home_post_delta_name = 'home_post_delta_halftime'
    away_pre_delta_name = 'away_pre_delta_halftime'
    away_post_delta_name = 'away_post_delta_halftime'
    pre_delta_diff_name = 'pre_delta_diff_halftime'
    home_team_buffer_name = 'home_team_buffer_halftime'
    post_delta_adjustment_name = 'pre_delta_adjustment_halftime'
    pre_delta_diff_adjusted = 'pre_delta_diff_halftime_adjusted'
    home_error = pre_delta_diff_name + '_home_team_home_error'
    away_error = pre_delta_diff_name + '_away_team_away_error'
    
    if home_error not in all_events.columns:
        all_events[home_error] = None
    if away_error not in all_events.columns:
        all_events[away_error] = None
        

    all_events = generate_elo_ranks(all_events, delta_column_to_calcuate, post_delta_adjustment_name, home_pre_delta_name, home_post_delta_name, away_pre_delta_name, away_post_delta_name, pre_delta_diff_name, home_team_buffer_name, max_points_win, win_margin_buffer, level_setting, start_range, end_range, list_order_int_men, list_order_int_women_age, list_order_club, win_bonus, all_competitions, all_teams, home_team_fixture_column, away_team_fixture_column, home_error, away_error)

    events_to_remove = list(all_events[ pd.isna(all_events['pre_delta_diff_halftime']) ]['event_id'])
    if len(events_to_remove) > 0:
        temp_events = all_events[ ~all_events['event_id'].isin(events_to_remove)]
        message = 'There are events where the pre_delta_diff could not be calculated, please check.  Select * from materialised_view_event where event_id in (' + str(events_to_remove) + ');'
        notifyTeams(message)
    else:
        temp_events = all_events
        
    formatted_data = all_events.loc[start_range:]
    formatted_data = formatted_data[[
'event_id',
'home_pre_delta_halftime',
'home_post_delta_halftime',
'away_pre_delta_halftime',
'away_post_delta_halftime',
'pre_delta_diff_halftime',
'pre_delta_diff_halftime_adjusted',
'home_team_buffer_halftime',
'pre_delta_adjustment_halftime', home_error, away_error]]
    powerbi_table_info, error = get_table_info('event_deltas')
    formatted_data = format_data_for_postgres(formatted_data, powerbi_table_info)
    update_sql('event_deltas', formatted_data, 'event_id')
    ########################################################################
    ########################################################################
    ########################################################################


-get_comp_standards
--Complete-0:00:00.059000

-generate_elo_ranks
0 --updating from 56042 57159
-- 56042 57159
-- 56043 57159
-- 56044 57159
-- 56045 57159


C:\ProgramData\Anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1117: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\ProgramData\Anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1117: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\ProgramData\Anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1117: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\ProgramData\Anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1117: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\ProgramData\Anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1117: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\ProgramData\Anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1117: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=ke

-- 56046 57159
-- 56047 57159
-- 56048 57159
-- 56049 57159
-- 56050 57159


C:\ProgramData\Anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1117: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\ProgramData\Anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1117: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\ProgramData\Anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1117: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\ProgramData\Anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1117: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\ProgramData\Anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1117: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\ProgramData\Anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1117: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=ke

-- 56051 57159
-- 56052 57159
-- 56053 57159
-- 56054 57159


...

-- 57149 57159
-- 57150 57159
-- 57151 57159
-- 57152 57159
-- 57153 57159
-- 57154 57159


C:\ProgramData\Anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1117: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\ProgramData\Anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1117: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\ProgramData\Anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1117: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\ProgramData\Anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1117: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\ProgramData\Anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1117: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\ProgramData\Anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1117: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=ke

-- 57155 57159
-- 57156 57159
-- 57157 57159
-- 57158 57159
-- 57159 57159


C:\ProgramData\Anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1117: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\ProgramData\Anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1117: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


--Complete-0:00:48.862668

update_sql - Update Complete - 1118


In [77]:
    ########################### Second Half ###########################
    delta_column_to_calcuate = 'second_half_win_margin'
    
    all_events[delta_column_to_calcuate] = (all_events['home_score'] - all_events['away_score']) - (all_events['home_halftime_score'] - all_events['away_halftime_score'])

    all_base_home_win_margin, international_mens_base_home_win_margin, international_womens_base_home_win_margin, competition_win_margin_means = get_comp_standards(all_events, delta_column_to_calcuate)

    home_pre_delta_name = 'home_pre_delta_secondhalf'
    home_post_delta_name = 'home_post_delta_secondhalf'
    away_pre_delta_name = 'away_pre_delta_secondhalf'
    away_post_delta_name = 'away_post_delta_secondhalf'
    pre_delta_diff_name = 'pre_delta_diff_secondhalf'
    home_team_buffer_name = 'home_team_buffer_secondhalf'
    post_delta_adjustment_name = 'pre_delta_adjustment_secondhalf'
    pre_delta_diff_adjusted = 'pre_delta_diff_secondhalf_adjusted'
    home_error = pre_delta_diff_name + '_home_team_home_error'
    away_error = pre_delta_diff_name + '_away_team_away_error'

    if home_error not in all_events.columns:
        all_events[home_error] = None
    if away_error not in all_events.columns:
        all_events[away_error] = None
        
    all_events = generate_elo_ranks(all_events, delta_column_to_calcuate, post_delta_adjustment_name, home_pre_delta_name, home_post_delta_name, away_pre_delta_name, away_post_delta_name, pre_delta_diff_name, home_team_buffer_name, max_points_win, win_margin_buffer, level_setting, start_range, end_range, list_order_int_men, list_order_int_women_age, list_order_club, win_bonus, all_competitions, all_teams, home_team_fixture_column, away_team_fixture_column, home_error, away_error)

    events_to_remove = list(all_events[ pd.isna(all_events['pre_delta_diff_secondhalf']) ]['event_id'])
    if len(events_to_remove) > 0:
        temp_events = all_events[ ~all_events['event_id'].isin(events_to_remove)]
        message = 'There are events where the pre_delta_diff could not be calculated, please check.  Select * from materialised_view_event where event_id in (' + str(events_to_remove) + ');'
        notifyTeams(message)
    else:
        temp_events = all_events
        
    formatted_data = all_events.loc[start_range:]
    formatted_data = formatted_data[[
    'event_id',
    'home_pre_delta_secondhalf',
    'home_post_delta_secondhalf',
    'away_pre_delta_secondhalf',
    'away_post_delta_secondhalf',
    'pre_delta_diff_secondhalf',
    'pre_delta_diff_secondhalf_adjusted',
    'home_team_buffer_secondhalf',
    'pre_delta_adjustment_secondhalf', home_error, away_error]]
    powerbi_table_info, error = get_table_info('event_deltas')
    formatted_data = format_data_for_postgres(formatted_data, powerbi_table_info)
    update_sql('event_deltas', formatted_data, 'event_id')
    ########################################################################
    ########################################################################
    ########################################################################
    


-get_comp_standards
--Complete-0:00:00.033004

-generate_elo_ranks
0 --updating from 56042 57159
-- 56042 57159
-- 56043 57159
-- 56044 57159
-- 56045 57159
-- 56046 57159
-- 56047 57159
-- 56048 57159
-- 56049 57159
-- 56050 57159
-- 56051 57159
-- 56052 57159
-- 56053 57159
-- 56054 57159
-- 56055 57159
-- 56056 57159
-- 56057 57159
-- 56058 57159
-- 56059 57159
-- 56060 57159
-- 56061 57159
-- 56062 57159
-- 56063 57159
-- 56064 57159
-- 56065 57159
-- 56066 57159
-- 56067 57159
-- 56068 57159
-- 56069 57159
-- 56070 57159
-- 56071 57159
-- 56072 57159
-- 56073 57159
-- 56074 57159
-- 56075 57159
-- 56076 57159
-- 56077 57159
-- 56078 57159
-- 56079 57159
-- 56080 57159
-- 56081 57159
-- 56082 57159
-- 56083 57159
-- 56084 57159
-- 56085 57159
-- 56086 57159
-- 56087 57159
-- 56088 57159
-- 56089 57159
-- 56090 57159
-- 56091 57159
-- 56092 57159
-- 56093 57159
-- 56094 57159
-- 56095 57159
-- 56096 57159
-- 56097 57159
-- 56098 57159
-- 56099 57159
-- 56100 57159
-- 56101 57159
-- 

-- 56583 57159
-- 56584 57159
-- 56585 57159
-- 56586 57159
-- 56587 57159
-- 56588 57159
-- 56589 57159
-- 56590 57159
-- 56591 57159
-- 56592 57159
-- 56593 57159
-- 56594 57159
-- 56595 57159
-- 56596 57159
-- 56597 57159
-- 56598 57159
-- 56599 57159
-- 56600 57159
-- 56601 57159
-- 56602 57159
-- 56603 57159
-- 56604 57159
-- 56605 57159
-- 56606 57159
-- 56607 57159
-- 56608 57159
-- 56609 57159
-- 56610 57159
-- 56611 57159
-- 56612 57159
-- 56613 57159
-- 56614 57159
-- 56615 57159
-- 56616 57159
-- 56617 57159
-- 56618 57159
-- 56619 57159
-- 56620 57159
-- 56621 57159
-- 56622 57159
-- 56623 57159
-- 56624 57159
-- 56625 57159
-- 56626 57159
-- 56627 57159
-- 56628 57159
-- 56629 57159
-- 56630 57159
-- 56631 57159
-- 56632 57159
-- 56633 57159
-- 56634 57159
-- 56635 57159
-- 56636 57159
-- 56637 57159
-- 56638 57159
-- 56639 57159
-- 56640 57159
-- 56641 57159
-- 56642 57159
-- 56643 57159
-- 56644 57159
-- 56645 57159
-- 56646 57159
-- 56647 57159
-- 56648 57159
-- 56649 5

-- 57132 57159
-- 57133 57159
-- 57134 57159
-- 57135 57159
-- 57136 57159
-- 57137 57159
-- 57138 57159
-- 57139 57159
-- 57140 57159
-- 57141 57159
-- 57142 57159
-- 57143 57159
-- 57144 57159
-- 57145 57159
-- 57146 57159
-- 57147 57159
-- 57148 57159
-- 57149 57159
-- 57150 57159
-- 57151 57159
-- 57152 57159
-- 57153 57159
-- 57154 57159
-- 57155 57159
-- 57156 57159
-- 57157 57159
-- 57158 57159
-- 57159 57159
--Complete-0:01:31.072557

update_sql - Update Complete - 1118


In [78]:
all_previous_deltas = get_all_previous_deltas(float_columns)

-get_all_previous_deltas
--Complete-0:00:11.004140



In [79]:
tp_columns = [
    'event_id',
    'home_attack_pre',
    'home_defence_pre',
    'home_home_attack_pre',
    'home_home_defence_pre',
    'away_attack_pre',
    'away_defence_pre',
    'away_away_attack_pre',
    'away_away_defence_pre',
    'home_attack_post',
    'home_defence_post',
    'home_home_attack_post',
    'home_home_defence_post',
    'away_attack_post',
    'away_defence_post',
    'away_away_attack_post',
    'away_away_defence_post',
    'pred_home_score_all',
    'pred_home_score_ha', 
    'pred_away_score_all', 
    'pred_away_score_ha',
    'pred_total_points_all',
    'pred_total_points_ha',
    'pred_total_points_all_adjusted',
    'pred_total_points_ha_adjusted']

In [80]:
for col in all_events.columns:
    if (col in tp_columns) & (col != 'event_id') :
        all_events.drop(col, axis = 1, inplace = True)

In [81]:
all_events = all_events.merge(all_previous_deltas[tp_columns])

In [82]:
    ##################################################################################################
    ########################################## Total Points ##########################################
    ##################################################################################################
    
    home_pre_delta_name_1 = 'home_attack_pre'
    home_pre_delta_name_2 = 'home_defence_pre'
    home_pre_delta_name_3 = 'home_home_attack_pre'
    home_pre_delta_name_4 = 'home_home_defence_pre'

    away_pre_delta_name_1 = 'away_attack_pre'
    away_pre_delta_name_2 = 'away_defence_pre'
    away_pre_delta_name_3 = 'away_away_attack_pre'
    away_pre_delta_name_4 = 'away_away_defence_pre'

    home_post_delta_name_1 = 'home_attack_post'
    home_post_delta_name_2 = 'home_defence_post'
    home_post_delta_name_3 = 'home_home_attack_post'
    home_post_delta_name_4 = 'home_home_defence_post'

    away_post_delta_name_1 = 'away_attack_post'
    away_post_delta_name_2 = 'away_defence_post'
    away_post_delta_name_3 = 'away_away_attack_post'
    away_post_delta_name_4 = 'away_away_defence_post'

    delta_column_to_calcuate_1 = 'home_score'
    delta_column_to_calcuate_2 = 'away_score'

    get_home_advantage = True
    all_events = calculate_total_points_deltas()
    #update_event_deltas_total_points(all_events, start_range, end_range, home_pre_delta_name_1, home_pre_delta_name_2, home_pre_delta_name_3, home_pre_delta_name_4, away_pre_delta_name_1, away_pre_delta_name_2, away_pre_delta_name_3, away_pre_delta_name_4, home_post_delta_name_1, home_post_delta_name_2, home_post_delta_name_3, home_post_delta_name_4, away_post_delta_name_1, away_post_delta_name_2, away_post_delta_name_3, away_post_delta_name_4)
    
    
    all_events['pred_total_points_all'] = all_events[['pred_home_score_all', 'pred_away_score_all']].apply(lambda x: max(0,x[0]) + max(0,x[1]), axis = 1 )
    all_events['pred_total_points_ha'] = all_events[['pred_home_score_ha', 'pred_away_score_ha']].apply(lambda x: max(0,x[0]) + max(0,x[1]), axis = 1 )
    all_events['pred_total_points_all_adjusted'] = all_events[['pred_total_points_all', 'pre_delta_diff', 'pre_delta_adjustment']].apply(lambda x: x[0] - ( -0.00175*((x[1] - x[2])*(x[1] - x[2]))), axis = 1 )
    all_events['pred_total_points_ha_adjusted'] = all_events[['pred_total_points_ha', 'pre_delta_diff', 'pre_delta_adjustment']].apply(lambda x: x[0] - ( -0.00175*((x[1] - x[2])*(x[1] - x[2]))), axis = 1 )

    formatted_data = all_events.loc[start_range:]
    formatted_data = formatted_data[[
    'event_id',
    'home_attack_pre',
    'home_defence_pre',
    'home_home_attack_pre',
    'home_home_defence_pre',
    'away_attack_pre',
    'away_defence_pre',
    'away_away_attack_pre',
    'away_away_defence_pre',
    'home_attack_post',
    'home_defence_post',
    'home_home_attack_post',
    'home_home_defence_post',
    'away_attack_post',
    'away_defence_post',
    'away_away_attack_post',
    'away_away_defence_post',
    'pred_home_score_all',
    'pred_home_score_ha', 
    'pred_away_score_all', 
    'pred_away_score_ha',
    'pred_total_points_all',
    'pred_total_points_ha',
    'pred_total_points_all_adjusted',
    'pred_total_points_ha_adjusted']]

    powerbi_table_info, error = get_table_info('event_deltas')
    formatted_data = format_data_for_postgres(formatted_data, powerbi_table_info)
    update_sql('event_deltas', formatted_data, 'event_id')
    ##############################################################################################################################
    ##############################################################################################################################
    ##############################################################################################################################
    
    

-get_all_previous_deltas
--Complete-0:00:07.475095

--updating from 56042 57159
-- 56042 57159
-- 56043 57159
-- 56044 57159
-- 56045 57159
-- 56046 57159
-- 56047 57159
-- 56048 57159
-- 56049 57159
-- 56050 57159
-- 56051 57159
-- 56052 57159
-- 56053 57159
-- 56054 57159
-- 56055 57159
-- 56056 57159
-- 56057 57159
-- 56058 57159
-- 56059 57159
-- 56060 57159
-- 56061 57159
-- 56062 57159
-- 56063 57159
-- 56064 57159
-- 56065 57159
-- 56066 57159
-- 56067 57159
-- 56068 57159
-- 56069 57159
-- 56070 57159
-- 56071 57159
-- 56072 57159
-- 56073 57159
-- 56074 57159
-- 56075 57159
-- 56076 57159
-- 56077 57159
-- 56078 57159
-- 56079 57159
-- 56080 57159
-- 56081 57159
-- 56082 57159
-- 56083 57159
-- 56084 57159
-- 56085 57159
-- 56086 57159
-- 56087 57159
-- 56088 57159
-- 56089 57159
-- 56090 57159
-- 56091 57159
-- 56092 57159
-- 56093 57159
-- 56094 57159
-- 56095 57159
-- 56096 57159
-- 56097 57159
-- 56098 57159
-- 56099 57159
-- 56100 57159
-- 56101 57159
-- 56102 57159
-- 56

-- 56585 57159
-- 56586 57159
-- 56587 57159
-- 56588 57159
-- 56589 57159
-- 56590 57159
-- 56591 57159
-- 56592 57159
-- 56593 57159
-- 56594 57159
-- 56595 57159
-- 56596 57159
-- 56597 57159
-- 56598 57159
-- 56599 57159
-- 56600 57159
-- 56601 57159
-- 56602 57159
-- 56603 57159
-- 56604 57159
-- 56605 57159
-- 56606 57159
-- 56607 57159
-- 56608 57159
-- 56609 57159
-- 56610 57159
-- 56611 57159
-- 56612 57159
-- 56613 57159
-- 56614 57159
-- 56615 57159
-- 56616 57159
-- 56617 57159
-- 56618 57159
-- 56619 57159
-- 56620 57159
-- 56621 57159
-- 56622 57159
-- 56623 57159
-- 56624 57159
-- 56625 57159
-- 56626 57159
-- 56627 57159
-- 56628 57159
-- 56629 57159
-- 56630 57159
-- 56631 57159
-- 56632 57159
-- 56633 57159
-- 56634 57159
-- 56635 57159
-- 56636 57159
-- 56637 57159
-- 56638 57159
-- 56639 57159
-- 56640 57159
-- 56641 57159
-- 56642 57159
-- 56643 57159
-- 56644 57159
-- 56645 57159
-- 56646 57159
-- 56647 57159
-- 56648 57159
-- 56649 57159
-- 56650 57159
-- 56651 5

-- 57132 57159
-- 57133 57159
-- 57134 57159
-- 57135 57159
-- 57136 57159
-- 57137 57159
-- 57138 57159
-- 57139 57159
-- 57140 57159
-- 57141 57159
-- 57142 57159
-- 57143 57159
-- 57144 57159
-- 57145 57159
-- 57146 57159
-- 57147 57159
-- 57148 57159
-- 57149 57159
-- 57150 57159
-- 57151 57159
-- 57152 57159
-- 57153 57159
-- 57154 57159
-- 57155 57159
-- 57156 57159
-- 57157 57159
-- 57158 57159
-- 57159 57159
--Complete-0:01:33.332310

update_sql - Update Complete - 1118


In [83]:
    ############################################################################################
    ########################################## Trends ##########################################
    ############################################################################################
    
    # Get all event deltas
    sql_statement = "select * from event_deltas;"
    event_deltas, error_occured_new = postgres_Retreive_Insert(sql_statement, POSTGRESQL_PARAMS, True)

    
    # Add all event deltas to all events
    all_events = add_event_deltas(all_events, event_deltas, float_columns)
    
    
    ########################### Win Margin ###########################
    delta_column_to_calcuate = 'win_margin'
    all_events[delta_column_to_calcuate] = (all_events['home_score'] - all_events['away_score'])
    
    ### Set column names
    home_pre_delta_name = 'home_pre_delta'
    home_post_delta_name = 'home_post_delta'
    away_pre_delta_name = 'away_pre_delta'
    away_post_delta_name = 'away_post_delta'
    pre_delta_diff_name = 'pre_delta_diff'
    home_team_buffer_name = 'home_team_buffer'
    
    delta_change_name = 'delta_change'
    error_name = 'error'

    # Delta change trend
    all_events['home_delta_change'] = all_events[home_post_delta_name] - all_events[home_pre_delta_name]
    all_events['away_delta_change'] = all_events[away_post_delta_name] - all_events[away_pre_delta_name]
    # Error trend
    all_events['home_error'] = all_events[pre_delta_diff_name] - all_events[delta_column_to_calcuate]
    all_events['away_error'] = -all_events[pre_delta_diff_name] - -all_events[delta_column_to_calcuate]

    # Uses start range calculated earlier
    for num_games in [5, 10, 20]:

        trend_name = delta_change_name
        trend_type = 'delta_change'
        all_events = add_trends(all_events, trend_name, trend_type, num_games, start_range)

        trend_name = error_name
        trend_type = 'delta_change'
        all_events = add_trends(all_events, trend_name, trend_type, num_games, start_range)

    update_event_deltas_trends(all_events, delta_change_name, error_name)
    #################################################################################
    
    
    ########################### First Half ###########################
    delta_column_to_calcuate = 'half_time_win_margin'
    all_events[delta_column_to_calcuate] = (all_events['home_halftime_score'] - all_events['away_halftime_score'])
    
    home_pre_delta_name = 'home_pre_delta_halftime'
    home_post_delta_name = 'home_post_delta_halftime'
    away_pre_delta_name = 'away_pre_delta_halftime'
    away_post_delta_name = 'away_post_delta_halftime'
    pre_delta_diff_name = 'pre_delta_diff_halftime'
    home_team_buffer_name = 'home_team_buffer_halftime'

    delta_change_name = 'delta_change_halftime'
    error_name = 'error_halftime'
    
    
    # Delta change trend
    all_events['home_delta_change'] = all_events[home_post_delta_name] - all_events[home_pre_delta_name]
    all_events['away_delta_change'] = all_events[away_post_delta_name] - all_events[away_pre_delta_name]
    # Error trend
    all_events['home_error'] = all_events[pre_delta_diff_name] - all_events[delta_column_to_calcuate]
    all_events['away_error'] = -all_events[pre_delta_diff_name] - -all_events[delta_column_to_calcuate]

    # Uses start range calculated earlier
    for num_games in [5, 10, 20]:

        trend_name = delta_change_name
        trend_type = 'delta_change'
        all_events = add_trends(all_events, trend_name, trend_type, num_games, start_range)

        trend_name = error_name
        trend_type = 'delta_change'
        all_events = add_trends(all_events, trend_name, trend_type, num_games, start_range)

    update_event_deltas_trends(all_events, delta_change_name, error_name)
    #################################################################################
    
    
    
    ########################### Second Half ###########################
    delta_column_to_calcuate = 'second_half_win_margin'
    
    all_events[delta_column_to_calcuate] = (all_events['home_score'] - all_events['away_score']) - (all_events['home_halftime_score'] - all_events['away_halftime_score'])

    home_pre_delta_name = 'home_pre_delta_secondhalf'
    home_post_delta_name = 'home_post_delta_secondhalf'
    away_pre_delta_name = 'away_pre_delta_secondhalf'
    away_post_delta_name = 'away_post_delta_secondhalf'
    pre_delta_diff_name = 'pre_delta_diff_secondhalf'
    home_team_buffer_name = 'home_team_buffer_secondhalf'

    delta_change_name = 'delta_change_secondhalf'
    error_name = 'error_secondhalf'
    
    
    # Delta change trend
    all_events['home_delta_change'] = all_events[home_post_delta_name] - all_events[home_pre_delta_name]
    all_events['away_delta_change'] = all_events[away_post_delta_name] - all_events[away_pre_delta_name]
    # Error trend
    all_events['home_error'] = all_events[pre_delta_diff_name] - all_events[delta_column_to_calcuate]
    all_events['away_error'] = -all_events[pre_delta_diff_name] - -all_events[delta_column_to_calcuate]

    # Uses start range calculated earlier
    for num_games in [5, 10, 20]:

        trend_name = delta_change_name
        trend_type = 'delta_change'
        all_events = add_trends(all_events, trend_name, trend_type, num_games, start_range)

        trend_name = error_name
        trend_type = 'delta_change'
        all_events = add_trends(all_events, trend_name, trend_type, num_games, start_range)

    update_event_deltas_trends(all_events, delta_change_name, error_name)
    #################################################################################

    

-add_event_deltas
--Complete-0:00:02.526041

-add_trends
-- delta_change 5


C:\Users\jupyter\AppData\Local\Temp\ipykernel_10736\180331540.py:23: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_team_fixtures = home_fixtures.append(away_fixtures, ignore_index = True)


--Complete-0:00:35.204446

-add_trends
-- error 5


C:\Users\jupyter\AppData\Local\Temp\ipykernel_10736\180331540.py:23: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_team_fixtures = home_fixtures.append(away_fixtures, ignore_index = True)


--Complete-0:00:35.547165

-add_trends
-- delta_change 10


...

--Complete-0:00:37.346113

-add_trends
-- delta_change_secondhalf 20


C:\Users\jupyter\AppData\Local\Temp\ipykernel_10736\180331540.py:23: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_team_fixtures = home_fixtures.append(away_fixtures, ignore_index = True)


--Complete-0:00:37.491432

-add_trends
-- error_secondhalf 20


C:\Users\jupyter\AppData\Local\Temp\ipykernel_10736\180331540.py:23: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_team_fixtures = home_fixtures.append(away_fixtures, ignore_index = True)


--Complete-0:00:37.657281

-update_event_deltas_trends
--Complete-0:01:15.552145

